# Agent and skill registries: what runs, in which version, on whose authority

### Chapter 4, notebook 2 of 3

Notebook 1 pinned every execution to the skill files it started with, and could check
that they had not changed. It could not say which version a *new* execution should
get, keep an old version runnable after a newer one appears, or stop a version
everywhere once it is withdrawn. The skill files on disk were the only copy.

This notebook adds the registries the harness resolves those questions against. The
home-purchase investigation continues: the same team of agents, the same Drive data,
the same kernel from notebook 1. One thing changes during the run: a newer
`environmental-risk` skill is released while an investigation is waiting for the
buyer.

**What gets built, in order**

| § | Builds | Chapter section |
|---|---|---|
| 1 | What notebook 1 left open | 4.3 |
| 2 | Five logically distinct registries on one SQLite file, as Pydantic models | 4.3 |
| 3 | Registration and discovery: the home-purchase team | 4.3 |
| 4 | An investigation starts and pins its resolved configuration | 4.3 |
| 5 | `environmental-risk` 1.1.0: version and digest tracking, compatibility, review, approval, a policy grant | 4.3 |
| 6 | The pinned branch runs on the version it started with | 4.3 |
| 7 | An explicit, validated, authorized migration | 4.3 |
| 8 | Deprecation and revocation | 4.3 |
| 9 | Operational metadata, derived from the run, and the registry log | 4.3, toward 4.4 |

**Paper and harness.** Registries, resolution, pinning and migration are harness
responsibilities. None of them is part of *SKILL.state*. The per-step contract the
agents run under is unchanged from notebook 1.

## 0. Setup and the harness from notebook 1

The setup cell works as in notebook 1. `RUN_MODE = "live"` calls
`openai/gpt-5.6-luna` through OpenRouter, about 50 calls, and records them to
`ch04_recordings/nb2_latest.jsonl`. `RUN_MODE = "replay"` re-runs a recording of this
notebook instead: this session's latest, or the reference recording in the Drive
folder's `ch04_replay`. It needs no API key and has no model cost. This
notebook also reads `ch04_skill_releases`, the folder where new skill versions arrive
for registration.

The next three cells are notebook 1's harness, unchanged: the skill parser, data and
tools, the state and protocols, the LangGraph kernel, the store, the agents and the
coordinator, and the helpers for events, forks, audits and invalidation. They are
generated from the same source as notebook 1, so the two cannot drift apart, and
they are collapsed. Notebook 1 explains every part of them.

This notebook replaces exactly three things in that harness, in section 2:
`resolve_agent` (which skill version an agent runs under, and which tools it gets),
`ARTIFACTS` (where skill text comes from), and the pin check (which now also asks the
registry whether anything pinned has been revoked).

In [ ]:
%pip install -q gdown openai pyyaml pandas numpy matplotlib python-dotenv langgraph langgraph-checkpoint-sqlite pydantic

import importlib, os, subprocess, sys

SKILL_STATE_REPO = "https://github.com/Nicolepcx/skill_state"
try:
    import skill_state
except ModuleNotFoundError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"git+{SKILL_STATE_REPO}"],
                       capture_output=True, text=True)
    if r.returncode != 0:                      # repository without pyproject.toml: use a clone
        if not os.path.isdir("/tmp/skill_state_src"):
            subprocess.run(["git", "clone", "-q", "--depth", "1", SKILL_STATE_REPO,
                            "/tmp/skill_state_src"], check=True)
        sys.path.insert(0, "/tmp/skill_state_src")
    importlib.invalidate_caches()
    import skill_state
print("skill_state from", skill_state.__file__)

skill_state from /usr/local/lib/python3.13/dist-packages/skill_state/__init__.py


In [ ]:
# @title Setup: run mode, Drive data, model client and recording
import getpass, hashlib, inspect, json, re, shutil, sqlite3, textwrap, time, uuid
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
from typing import Any, Callable, Dict, Iterator, List, Optional, Tuple

import gdown
import numpy as np
import openai
import pandas as pd
import yaml

from skill_state import (LLMResult, ParseError, Skill, SkillStateRuntime, build_prompt,
                         merge, parse_response, validate)

# "live": real model calls through OpenRouter, each one recorded.
# "replay": re-run a recording of a live run; no API key, no model cost.
RUN_MODE = "live"  # @param ["live", "replay"]
REPLAY_FILE = None    # for replay; default: this session's latest recording, else the one in the Drive folder
NB = "nb2"                                 # this notebook's recordings are nb1_*.jsonl
NOTEBOOK_VERSION = "ch04-nb2/2026-10-31"   # a recording replays only with the version that made it

DRIVE_FOLDER_URL = ("https://drive.google.com/drive/folders/"
                    "1_zFkQWD8Mjyv_K5cCjMT2w7QTH2ttu1I?usp=sharing")
WORK = Path("ch04_work").resolve()
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)

kwargs = {"url": DRIVE_FOLDER_URL, "output": str(WORK), "quiet": True, "use_cookies": False}
if "remaining_ok" in inspect.signature(gdown.download_folder).parameters:
    kwargs["remaining_ok"] = True
if not gdown.download_folder(**kwargs):
    raise RuntimeError("Nothing downloaded; is the Drive folder shared with 'Anyone with the link'?")

def drive_folder(name: str) -> Path:
    found = [p for p in WORK.rglob(name) if p.is_dir()]
    if not found:
        raise FileNotFoundError(f"The Drive folder has no subfolder '{name}'. Downloaded: "
                                f"{sorted(p.name for p in WORK.iterdir())}")
    return found[0]


DATA = drive_folder("ch04_data")
SKILLS = drive_folder("ch04_skills")
RELEASES = drive_folder("ch04_skill_releases")   # new skill versions arrive here
INPUT_DIRS = [DATA, SKILLS, RELEASES]          # the files a recording depends on
for folder in (DATA, SKILLS, RELEASES):
    print(folder.name)
    for p in sorted(folder.iterdir()):
        print(f"   {p.name:46}{p.stat().st_size:>10,} bytes")

# --- model access -------------------------------------------------------------------
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(), override=False)
except Exception:
    pass
if RUN_MODE == "live" and not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = "openai/gpt-5.6-luna"
BASE_URL = "https://openrouter.ai/api/v1"
USAGE: List[Dict[str, Any]] = []          # one row per model call, including calls stopped early
RECORDER = None                           # set below in live mode
_CALL_IDS = iter(range(1, 10**9))


class Generation:
    """One streamed model call. Iterate it for text pieces. `record` is this call's usage row;
    it is completed and logged once, when the stream ends or is closed early.

    Usage comes from the provider when the stream runs to the end. A stream closed early ends
    before the provider's usage chunk arrives, so its tokens are estimated from characters
    and marked as estimated; the provider may still bill for tokens generated after the close.
    `generation_id` lets section 7 ask the provider for the billed figures."""

    def __init__(self, client, model: str, label: str, prompt: str):
        self.record = {"call": next(_CALL_IDS), "label": label, "model": model, "execution_id": None,
                       "generation_id": None, "prompt_chars": len(prompt), "replayed": False}
        self._prompt, self._parts, self._usage, self._t0 = prompt, [], None, time.perf_counter()
        self._finished, self._complete = False, False
        self._resp = client.chat.completions.create(
            model=model, stream=True, stream_options={"include_usage": True},
            messages=[{"role": "user", "content": prompt}])
        self._gen = self._pieces()

    def _pieces(self) -> Iterator[str]:
        try:
            for chunk in self._resp:
                self.record["generation_id"] = self.record["generation_id"] or getattr(chunk, "id", None)
                if getattr(chunk, "usage", None):
                    self._usage = chunk.usage
                if chunk.choices and chunk.choices[0].delta and chunk.choices[0].delta.content:
                    self._parts.append(chunk.choices[0].delta.content)
                    yield self._parts[-1]
            self._complete = True
        finally:
            self._finish()

    def __iter__(self) -> Iterator[str]:
        return self._gen

    def close(self) -> None:
        self._gen.close()
        self._finish()                      # also covers a stream closed before its first piece

    def _finish(self) -> None:
        if self._finished:
            return
        self._finished = True
        self._resp.close()                  # closing the HTTP response stops the generation
        text, u = "".join(self._parts), self._usage
        self.record.update({
            "prompt_tokens": getattr(u, "prompt_tokens", None) or len(self._prompt) // 4,
            "completion_tokens": getattr(u, "completion_tokens", None) or len(text) // 4,
            "usage_source": "provider" if u else "estimated", "aborted": not self._complete,
            "completion_chars": len(text), "seconds": round(time.perf_counter() - self._t0, 2)})
        USAGE.append(self.record)
        if RECORDER:
            RECORDER.write(self.record, self._prompt, text)


class OpenRouterLLM:
    """skill_state's LLM protocol (`complete`) plus `start`, which the kernel uses to read
    the output while it is generated and to stop it early."""

    def __init__(self, model: str = MODEL, label: str = "unlabelled"):
        self.model, self.label = model, label
        self.client = openai.OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url=BASE_URL)

    def with_label(self, label: str) -> "OpenRouterLLM":
        other = OpenRouterLLM.__new__(OpenRouterLLM)
        other.model, other.label, other.client = self.model, label, self.client
        return other

    def start(self, prompt: str) -> Generation:
        return Generation(self.client, self.model, self.label, prompt)

    def complete(self, prompt: str) -> LLMResult:
        g = self.start(prompt)
        text = "".join(g)
        return LLMResult(text, g.record["prompt_tokens"], g.record["completion_tokens"])


def provider_usage(generation_id: str, attempts: int = 3) -> Optional[Dict[str, Any]]:
    """The provider's own record of a generation (OpenRouter `GET /generation?id=`)."""
    import requests
    for i in range(attempts):
        try:
            r = requests.get(f"{BASE_URL}/generation", params={"id": generation_id}, timeout=20,
                             headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"})
            if r.ok:
                return r.json().get("data")
        except requests.RequestException:
            pass
        time.sleep(2 * (i + 1))             # the record can take a moment to appear
    return None



import difflib
from collections import defaultdict, deque
from contextlib import contextmanager

RECORDINGS = Path("ch04_recordings").resolve()
RECORDINGS.mkdir(exist_ok=True)


def prompt_digest(prompt: str) -> str:
    return hashlib.sha256(prompt.encode()).hexdigest()[:16]


def input_fingerprint() -> Dict[str, Any]:
    """What a recording depends on besides the harness code: the notebook version, the model,
    and every data and skill file, by digest."""
    files = {f"{d.name}/{p.name}": hashlib.sha256(p.read_bytes()).hexdigest()[:16]
             for d in INPUT_DIRS for p in sorted(d.iterdir()) if p.is_file()}
    return {"notebook": NOTEBOOK_VERSION, "model": MODEL, "files": files}


class Recorder:
    """A header line with the input fingerprint, then one JSON line per model call: its usage
    row, the exact prompt, and the response text exactly as the API returned it."""

    def __init__(self, path: Path):
        self.path, self.count = path, 0
        self.file = path.open("w")
        self.file.write(json.dumps({"kind": "header", **input_fingerprint()}) + "\n")
        self.file.flush()

    def write(self, record: Dict, prompt: str, text: str):
        row = {**record, "prompt_digest": prompt_digest(prompt), "prompt": prompt, "text": text}
        self.file.write(json.dumps(row) + "\n")
        self.file.flush()
        self.count += 1


def read_recording(path: Path) -> Tuple[Dict, List[Dict]]:
    rows = [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]
    header = rows[0] if rows and rows[0].get("kind") == "header" else {}
    return header, [r for r in rows if r.get("kind") != "header"]


def load_recording(path: Path) -> List[Dict]:
    return read_recording(path)[1]


def incompatibilities(header: Dict) -> List[str]:
    """Why a recording cannot be replayed with this notebook and these files, if it cannot."""
    now = input_fingerprint()
    if not header:
        return ["the recording has no header"]
    out = []
    if header.get("notebook") != now["notebook"]:
        out.append(f"made by notebook version {header.get('notebook')}, this is {now['notebook']}")
    for name in sorted(set(header.get("files", {})) | set(now["files"])):
        if header.get("files", {}).get(name) != now["files"].get(name):
            out.append(f"{name} differs from the file the recording was made with")
    return out


class ReplayDivergence(Exception):
    """The harness produced a prompt that the recorded run never sent."""

    def __init__(self, label: str, prompt: str, expected: Optional[Dict], reason: Optional[str] = None):
        self.label, self.prompt, self.expected = label, prompt, expected
        if reason:
            self.diff = [reason]
        elif expected is None:
            self.diff = ["(the recording has no further call with this label)"]
        else:
            self.diff = [l.rstrip("\n") for l in difflib.unified_diff(
                expected["prompt"].splitlines(), prompt.splitlines(),
                f"recorded call {expected['call']}", "this run", n=1, lineterm="")]
        super().__init__(f"replay diverged at '{label}': " + (reason or "the prompt was never recorded"))


class ReplayGeneration:
    """Plays one recorded response back in pieces, through the same interface as Generation.
    A recorded call the harness stopped early must be stopped early again, and vice versa;
    otherwise the replay is not reproducing the run."""

    def __init__(self, rec: Dict, label: str, prompt: str, piece: int = 40):
        self.rec, self.text = rec, rec["text"]
        self.record = {"call": next(_CALL_IDS), "label": label, "model": rec["model"], "execution_id": None,
                       "generation_id": rec.get("generation_id"), "prompt_chars": len(prompt),
                       "replayed": True, "recorded_call": rec["call"]}
        self._finished, self._exhausted = False, False
        self._gen = self._pieces(piece)

    def _pieces(self, n: int) -> Iterator[str]:
        for i in range(0, len(self.text), n):
            yield self.text[i:i + n]
        self._exhausted = True

    def __iter__(self):
        return self._gen

    def close(self):
        self._gen.close()
        if self._finished:
            return
        self._finished = True
        stopped_early = not self._exhausted
        self.record.update({"prompt_tokens": self.rec["prompt_tokens"],
                            "completion_tokens": self.rec["completion_tokens"], "usage_source": "recorded",
                            "aborted": stopped_early, "completion_chars": len(self.text), "seconds": 0.0})
        USAGE.append(self.record)
        if stopped_early != self.rec["aborted"]:
            raise ReplayDivergence(self.record["label"], "", self.rec, reason=(
                f"recorded call {self.rec['call']} was {'stopped early' if self.rec['aborted'] else 'completed'}; "
                f"in this replay it was {'stopped early' if stopped_early else 'completed'}"))


class ReplayLLM:
    """Serves recorded responses to identical prompts. Each instance consumes its own copy."""

    def __init__(self, calls: List[Dict], labels: Optional[set] = None, label: str = "unlabelled", _q=None):
        self.label, self.calls = label, calls
        self.model = calls[0]["model"] if calls else MODEL
        if _q is None:
            _q = {"exact": defaultdict(deque), "by_label": defaultdict(deque), "served": [0]}
            for c in calls:
                if labels is None or c["label"] in labels:
                    _q["exact"][(c["label"], c["prompt_digest"])].append(c)
                    _q["by_label"][c["label"]].append(c)
        self._q = _q

    def with_label(self, label: str) -> "ReplayLLM":
        return ReplayLLM(self.calls, label=label, _q=self._q)

    def start(self, prompt: str) -> ReplayGeneration:
        hits = self._q["exact"].get((self.label, prompt_digest(prompt)))
        if not hits:                    # report the closest unserved recorded prompt with this label
            pending = list(self._q["by_label"].get(self.label) or [])
            closest = max(pending, key=lambda c: difflib.SequenceMatcher(None, c["prompt"], prompt).quick_ratio(),
                          default=None)
            raise ReplayDivergence(self.label, prompt, closest)
        rec = hits.popleft()
        self._q["by_label"][self.label].remove(rec)
        self._q["served"][0] += 1
        return ReplayGeneration(rec, self.label, prompt)

    def complete(self, prompt: str) -> LLMResult:
        g = self.start(prompt)
        text = "".join(g)
        g.close()
        return LLMResult(text, g.record["prompt_tokens"], g.record["completion_tokens"])

    def unserved(self) -> int:
        return sum(len(q) for q in self._q["by_label"].values())


@contextmanager
def using_llm(other):
    """Temporarily route every kernel built inside the block to another model client."""
    global llm
    previous, llm = llm, other
    try:
        yield other
    finally:
        llm = previous


if RUN_MODE == "live":
    RECORDING_PATH = RECORDINGS / f"{NB}_latest.jsonl"
    RECORDER = Recorder(RECORDING_PATH)
    llm = OpenRouterLLM()
elif RUN_MODE == "replay":
    # REPLAY_FILE if set; else this session's own latest live recording; else one from the Drive
    # folder, under either name (nb1_recording.jsonl or nb1_latest.jsonl).
    candidates = [Path(REPLAY_FILE)] if REPLAY_FILE else \
        [RECORDINGS / f"{NB}_latest.jsonl",
         *sorted(p for name in (f"{NB}_recording.jsonl", f"{NB}_latest.jsonl") for p in WORK.rglob(name))]
    RECORDING_PATH = next((p for p in candidates if p.exists()), None)
    if RECORDING_PATH is None:
        raise FileNotFoundError(
            f"Replay needs a recording made by this notebook ({NB}_recording.jsonl or {NB}_latest.jsonl), "
            "and none was found in ch04_recordings/ or in the Drive folder. Set RUN_MODE = 'live' to make one.")
    header, calls = read_recording(RECORDING_PATH)
    problems = incompatibilities(header)
    if problems:
        raise RuntimeError("This recording cannot be replayed here. A replay re-runs one recorded "
                           "investigation; it cannot stand in for a different one.\n  - " + "\n  - ".join(problems)
                           + "\nUse a recording made with this notebook and these files, or set RUN_MODE = 'live'.")
    llm = ReplayLLM(calls)
    MODEL = llm.model
else:
    raise ValueError(f"RUN_MODE must be 'live' or 'replay', not {RUN_MODE!r}")
print(f"mode: {RUN_MODE}   model: {MODEL}   recording: {RECORDING_PATH}")

ch04_data
   SOURCES.md                                         1,760 bytes
   hazard_profiles_illustrative.csv                   1,073 bytes
   insurance_scenarios_illustrative.csv                 808 bytes
   king_county_sales_2014_2015.csv                2,326,367 bytes
   zillow_forecast_seattle_metro.csv                    148 bytes
   zillow_market_heat_seattle_metro.csv               1,569 bytes
   zillow_zhvi_seattle_metro.csv                      9,434 bytes
   zipcode_city.csv                                   1,028 bytes
ch04_skills
   AGENTS.md                                          1,403 bytes
   PROJECT.md                                         1,146 bytes
   home-purchase-research-state-schema.json           2,407 bytes
   skill-environmental-risk.md                          930 bytes
   skill-home-purchase-research.md                    2,492 bytes
   skill-insurance-finance.md                         1,225 bytes
   skill-market-research.md                           

In [ ]:
# @title Harness from notebook 1: skill parser, data, tools, state and protocols
# --- Chapter 3's artifact parser and surface compiler, unchanged in behaviour --------
_FRONTMATTER = re.compile(r"\A---\n(.*?)\n---\n?", re.S)
_SECTION = re.compile(r"^<!--\s*section:\s*([\w.\-]+)\s*-->\s*$", re.M)
KIND_PRECEDENCE = ("identity", "project", "skill", "persona")


@dataclass(frozen=True)
class Artifact:
    kind: str
    name: str
    version: str
    path: Path
    meta: Dict
    sections: Dict[str, str]
    digest: str

    @property
    def always_loaded(self) -> bool:
        return bool(self.meta.get("always_loaded"))

    @property
    def owns(self) -> List[str]:
        return list(self.meta.get("owns") or [])

    @property
    def requires(self) -> Dict[str, List[str]]:
        """Capability requests (Chapter 3b). A request is never a grant."""
        return {k: list(v or []) for k, v in (self.meta.get("requires") or {}).items()}


def parse(path: Path) -> Artifact:
    raw = path.read_text(encoding="utf-8")
    m = _FRONTMATTER.match(raw)
    if not m:
        raise ValueError(f"{path.name}: no frontmatter")
    meta, body = yaml.safe_load(m.group(1)) or {}, raw[m.end():]
    marks = list(_SECTION.finditer(body))
    if not marks or body[:marks[0].start()].strip():
        raise ValueError(f"{path.name}: text outside a section")
    sections = {}
    for i, mark in enumerate(marks):
        end = marks[i + 1].start() if i + 1 < len(marks) else len(body)
        sections[mark.group(1)] = body[mark.end():end].strip()
    for missing in set(meta.get("owns") or []) - set(sections):
        raise ValueError(f"{path.name}: owns '{missing}' but has no such section")
    return Artifact(meta["artifact"], meta["name"], str(meta["version"]), path, meta, sections,
                    hashlib.sha256(raw.encode()).hexdigest()[:12])


def overlap(artifacts) -> Dict[str, List[str]]:
    claims: Dict[str, List[str]] = {}
    for a in artifacts:
        for s in a.owns:
            claims.setdefault(s, []).append(a.name)
    return {s: n for s, n in claims.items() if len(n) > 1}


@dataclass
class Surface:
    text: str
    manifest: Dict[str, Any]


def compile_surface(artifacts, facts: Optional[Dict[str, Any]] = None) -> Surface:
    """Identity, then project, then skill; repeated paragraphs kept once; a manifest of
    every section that went in, with its artifact version and digest."""
    clash = overlap(artifacts)
    if clash:
        raise ValueError(f"ambiguous surface: {clash}")
    blocks, seen, used = [], set(), []
    for a in sorted(artifacts, key=lambda a: KIND_PRECEDENCE.index(a.kind)):
        for sid, body in a.sections.items():
            kept = [p.strip() for p in re.split(r"\n\s*\n", body)
                    if p.strip() and re.sub(r"\s+", " ", p.lower()) not in seen]
            seen.update(re.sub(r"\s+", " ", p.lower()) for p in kept)
            if kept:
                blocks.append("\n\n".join(kept))
                used.append({"artifact": a.name, "version": a.version, "digest": a.digest,
                             "section": sid})
    if facts:
        blocks.append("## Facts supplied by the harness\n\n" +
                      "\n".join(f"- {k}: {v}" for k, v in facts.items()) +
                      "\n\nThese are given. Do not infer facts that are not listed.")
    text = "\n\n".join(blocks)
    return Surface(text, {"sections": used,
                          "surface_digest": hashlib.sha256(text.encode()).hexdigest()[:12]})


ARTIFACTS = {a.name: a for a in (parse(p) for p in sorted(SKILLS.glob("*.md")))}
ALWAYS = [a for a in ARTIFACTS.values() if a.always_loaded]


CITIES = ("Seattle", "Bellevue", "Kent")


def _wide_series(file: str) -> pd.Series:
    row = pd.read_csv(DATA / file).iloc[0]
    months = [c for c in row.index if c[:2] in ("19", "20")]
    s = pd.to_numeric(row[months])
    s.index = pd.to_datetime(months)
    return s.dropna()


SALES = (pd.read_csv(DATA / "king_county_sales_2014_2015.csv", dtype={"zipcode": str, "id": str},
                     parse_dates=["date"])
         .merge(pd.read_csv(DATA / "zipcode_city.csv", dtype={"zipcode": str}), on="zipcode"))
ZHVI = _wide_series("zillow_zhvi_seattle_metro.csv")
HEAT = _wide_series("zillow_market_heat_seattle_metro.csv")
_fc = pd.read_csv(DATA / "zillow_forecast_seattle_metro.csv").iloc[0]
FORECAST = {"base_date": _fc["BaseDate"],
            "growth_pct": {c: float(_fc[c]) for c in _fc.index if c[:2] == "20" and c != _fc["BaseDate"]}}
HAZARDS = pd.read_csv(DATA / "hazard_profiles_illustrative.csv")
INSURANCE = pd.read_csv(DATA / "insurance_scenarios_illustrative.csv")


def _evidence(file: str, as_of: str, illustrative: bool = False) -> Dict[str, Any]:
    return {"file": file, "as_of": as_of, "illustrative": illustrative,
            "sha256": hashlib.sha256((DATA / file).read_bytes()).hexdigest()[:16]}


# One provenance record per source. Artifacts record which of these they used, so a
# changed file can invalidate exactly the artifacts built on it.
EVIDENCE = {
    "king_county_sales": _evidence("king_county_sales_2014_2015.csv", str(SALES.date.max().date())),
    "zillow_zhvi": _evidence("zillow_zhvi_seattle_metro.csv", str(ZHVI.index.max().date())),
    "zillow_market_heat": _evidence("zillow_market_heat_seattle_metro.csv", str(HEAT.index.max().date())),
    "zillow_forecast": _evidence("zillow_forecast_seattle_metro.csv", FORECAST["base_date"]),
    "hazard_profiles": _evidence("hazard_profiles_illustrative.csv", "n/a", True),
    "insurance_scenarios": _evidence("insurance_scenarios_illustrative.csv", "n/a", True),
}


class ToolUnavailable(Exception):
    """An external data source did not respond."""


SERVICE_DOWN: set = set()          # regions whose insurance feed is not responding
RATE_LIMITED: set = set()          # tools whose service answers every request with HTTP 429


class RateLimited(Exception):
    """A service refused the request for now. Not an outage: the harness does not suspend for it."""


def _check_rate_limit(tool: str) -> None:
    if tool in RATE_LIMITED:
        raise RateLimited(f"{tool}: HTTP 429 Too Many Requests. Retry the request.")
ILLUSTRATIVE_RATE_PCT = 6.5        # mortgage rate assumption for the payment tool


def _records(df: pd.DataFrame) -> List[Dict[str, Any]]:
    return json.loads(df.to_json(orient="records"))


def _check_city(city: str) -> None:
    if city not in CITIES:
        raise ValueError(f"unknown city '{city}'; use one of {list(CITIES)}")


def _est_value_today(d: pd.DataFrame) -> np.ndarray:
    """2014-15 sale price scaled by the metro value index from the sale month to now."""
    at_sale = ZHVI.reindex(d["date"] + pd.offsets.MonthEnd(0)).to_numpy()
    return d["price"].to_numpy() * float(ZHVI.iloc[-1]) / at_sale


def query_sales(city: str, budget_usd: Optional[float] = None) -> Dict[str, Any]:
    _check_city(city)
    d = SALES[SALES.city == city].copy()
    d["est_value_today"] = _est_value_today(d)
    within = d[d.est_value_today <= budget_usd] if budget_usd else d.iloc[0:0]
    examples = within.nlargest(5, "est_value_today")[
        ["id", "bedrooms", "bathrooms", "sqft_living", "grade", "yr_built", "est_value_today"]].round(0)
    return {"city": city, "sales_2014_2015": int(len(d)),
            "median_price_2014_2015_usd": int(d.price.median()),
            "median_est_value_today_usd": int(d.est_value_today.median()),
            "est_value_method": "2014-15 price scaled by the Seattle-metro value index; metro-level estimate",
            "budget_usd": budget_usd,
            "comparables_within_budget": int(len(within)) if budget_usd else None,
            "median_est_value_within_budget_usd": int(within.est_value_today.median()) if len(within) else None,
            "examples_closest_to_budget": _records(examples)}


def metro_value_series() -> Dict[str, Any]:
    s = ZHVI
    five_years_ago = s[s.index <= s.index[-1] - pd.DateOffset(years=5)].iloc[-1]
    return {"region": "Seattle metro", "as_of": str(s.index[-1].date()),
            "latest_usd": int(s.iloc[-1]), "value_2014_05_usd": int(s.loc["2014-05-31"]),
            "peak": {"date": str(s.idxmax().date()), "value_usd": int(s.max())},
            "change_since_peak_pct": round(100 * (s.iloc[-1] / s.max() - 1), 1),
            "change_5y_pct": round(100 * (s.iloc[-1] / five_years_ago - 1), 1)}


def market_heat(months: int = 24) -> Dict[str, Any]:
    _check_rate_limit("market_heat")
    s = HEAT.iloc[-int(months):]
    return {"region": "Seattle metro", "as_of": str(s.index[-1].date()), "latest": float(s.iloc[-1]),
            "mean_last_12": round(float(HEAT.iloc[-12:].mean()), 1),
            "change_12m": float(HEAT.iloc[-1] - HEAT.iloc[-13]),
            "trend_per_month": round(float(np.polyfit(np.arange(len(s)), s.to_numpy(), 1)[0]), 2),
            "window_min": float(s.min()), "window_max": float(s.max())}


def value_forecast() -> Dict[str, Any]:
    return {"region": "Seattle metro", "publisher": "Zillow", **FORECAST}


def run_regression(model: str, horizon_years: int = 10, window_start: str = "2012-01-31",
                   window_end: Optional[str] = None) -> Dict[str, Any]:
    if model == "hedonic":
        d = SALES[SALES.city.isin(CITIES)]
        X = np.column_stack([np.ones(len(d)), np.log(d.sqft_living), d.grade, d.condition,
                             2015 - d.yr_built, (d.city == "Bellevue").astype(float),
                             (d.city == "Kent").astype(float)])
        y = np.log(d.price.to_numpy())
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        resid = y - X @ beta
        names = ["intercept", "log_sqft_living", "grade", "condition", "age_years",
                 "city_Bellevue", "city_Kent"]
        return {"model": "hedonic", "target": "log(price), 2014-15 sales", "n": int(len(d)),
                "coefficients": {n: round(float(b), 4) for n, b in zip(names, beta)},
                "city_effect_vs_Seattle_pct": {c: round(100 * (np.exp(beta[i]) - 1), 1)
                                               for c, i in (("Bellevue", 5), ("Kent", 6))},
                "r2": round(float(1 - resid.var() / y.var()), 3),
                "residual_sd_log": round(float(resid.std()), 3)}
    if model == "trend":
        s = ZHVI[window_start:window_end]
        t = ((s.index - s.index[0]).days / 365.25).to_numpy()
        slope, intercept = np.polyfit(t, s.to_numpy(), 1)
        fitted = intercept + slope * t
        r2 = 1 - ((s.to_numpy() - fitted) ** 2).sum() / ((s.to_numpy() - s.mean()) ** 2).sum()
        latest, t_latest = float(ZHVI.iloc[-1]), (ZHVI.index[-1] - s.index[0]).days / 365.25
        g12 = list(FORECAST["growth_pct"].values())[-1]
        h = int(horizon_years)
        return {"model": "linear_trend", "series": "Seattle-metro typical home value",
                "window": [str(s.index[0].date()), str(s.index[-1].date())],
                "slope_usd_per_year": int(slope), "r2": round(float(r2), 3), "horizon_years": h,
                "scenarios_usd_at_horizon": {
                    "trend_continues": int(intercept + slope * (t_latest + h)),
                    "flat": int(latest),
                    "published_12m_forecast_carried_forward": int(latest * (1 + g12 / 100) ** h)},
                "label": "scenarios under stated assumptions, not predictions"}
    raise ValueError("model must be 'hedonic' or 'trend'")


def hazard_profile(city: str) -> Dict[str, Any]:
    _check_city(city)
    return {"city": city, "source": "illustrative table",
            "hazards": _records(HAZARDS[HAZARDS.city == city][["hazard", "score_0_100", "notes"]])}


def insurance_scenarios(region: str) -> Dict[str, Any]:
    if region in SERVICE_DOWN or (region == "King County" and SERVICE_DOWN):
        raise ToolUnavailable(f"insurance data service: request for '{region}' timed out "
                              "(HTTP 503 Service Unavailable). The request may succeed if retried.")
    rows = INSURANCE[INSURANCE.region == region]
    if rows.empty:
        raise ValueError(f"unknown region '{region}'; use one of {sorted(INSURANCE.region.unique())}")
    return {"region": region, "source": "illustrative table",
            "scenarios": _records(rows.drop(columns="region"))}


def affordability(price_usd: float, down_payment_pct: float, annual_insurance_usd: float,
                  budget_usd: Optional[float] = None, rate_pct: float = ILLUSTRATIVE_RATE_PCT,
                  years: int = 30, property_tax_pct: float = 1.0) -> Dict[str, Any]:
    loan = price_usd * (1 - down_payment_pct / 100)
    r, n = rate_pct / 1200, years * 12
    pi = loan * r / (1 - (1 + r) ** -n)
    tax, ins = price_usd * property_tax_pct / 1200, annual_insurance_usd / 12
    return {"price_usd": round(price_usd), "down_payment_usd": round(price_usd - loan),
            "loan_usd": round(loan), "rate_pct": rate_pct, "rate_note": "illustrative rate",
            "monthly_principal_interest_usd": round(pi), "monthly_property_tax_usd": round(tax),
            "monthly_insurance_usd": round(ins), "monthly_total_usd": round(pi + tax + ins),
            "price_to_budget_pct": round(100 * price_usd / budget_usd, 1) if budget_usd else None}


@dataclass(frozen=True)
class Tool:
    fn: Optional[Callable]
    reads: Tuple[str, ...]          # evidence names, recorded as artifact inputs
    doc: str                        # what the agent is told the tool does


TOOLS: Dict[str, Tool] = {
    "query_sales": Tool(query_sales, ("king_county_sales", "zillow_zhvi"),
                        "2014-15 sales in a city, their estimated value today, and comparables within budget"),
    "metro_value_series": Tool(metro_value_series, ("zillow_zhvi",),
                               "Seattle-metro typical home value: latest, 2014, peak, 5-year change"),
    "market_heat": Tool(market_heat, ("zillow_market_heat",),
                        "Seattle-metro market heat index over the last `months` months"),
    "value_forecast": Tool(value_forecast, ("zillow_forecast",),
                           "Zillow's published 1, 3 and 12-month value forecast for the Seattle metro"),
    "run_regression": Tool(run_regression, ("king_county_sales", "zillow_zhvi", "zillow_forecast"),
                           "model='hedonic': log price on size, grade, condition, age and city; "
                           "model='trend': linear trend on the metro series with scenarios at the horizon"),
    "hazard_profile": Tool(hazard_profile, ("hazard_profiles",), "hazard scores 0-100 for a city"),
    "insurance_scenarios": Tool(insurance_scenarios, ("insurance_scenarios",),
                                "insurance premiums and coverage for a region: a city, or 'King County'"),
    "affordability": Tool(affordability, (), "monthly cost of a purchase: principal and interest, tax, insurance"),
    "read_artifacts": Tool(None, (), "the content of artifacts in this branch, by id"),
}


def tool_usage(name: str) -> str:
    """How the agent is told to call a tool: its signature and one line on what it returns."""
    sig = "ids: list of artifact ids" if name == "read_artifacts" else ", ".join(
        p.name if p.default is inspect.Parameter.empty else f"{p.name}={p.default!r}"
        for p in inspect.signature(TOOLS[name].fn).parameters.values())
    return f"{name}({sig}): {TOOLS[name].doc}"


def run_tool(name: str, args: Dict[str, Any], read_artifacts: Optional[Callable] = None) -> Dict[str, Any]:
    if name not in TOOLS:
        raise ValueError(f"unknown tool '{name}'")
    if name == "read_artifacts":
        return read_artifacts(list(args.get("ids", [])))
    try:
        inspect.signature(TOOLS[name].fn).bind(**args)
    except TypeError as e:
        raise ValueError(f"bad arguments ({e}). Usage: {tool_usage(name)}") from None
    return TOOLS[name].fn(**args)


def compact(result: Any, limit: int = 1800) -> str:
    text = json.dumps(result, separators=(",", ":"), default=str)
    return text if len(text) <= limit else text[:limit] + "...(truncated)"


COORD_SKILL = "home-purchase-research"
COORD_SCHEMA = json.loads((SKILLS / "home-purchase-research-state-schema.json").read_text())

AGENT_SKILL = {"market_research": "market-research", "quantitative_analysis": "quantitative-analysis",
               "environmental_risk": "environmental-risk", "insurance_finance": "insurance-finance",
               "synthesis": "synthesis"}
OUTPUT_KIND = {"market_research": "market_summary", "quantitative_analysis": "model_report",
               "environmental_risk": "risk_assessment", "insurance_finance": "affordability",
               "synthesis": "report"}
# Which constraints each agent is given. Recorded on its artifact, so a changed
# constraint invalidates exactly the artifacts that used it.
CONSUMES = {"market_research": ["budget_usd"], "quantitative_analysis": ["horizon_years"],
            "environmental_risk": [], "insurance_finance": ["budget_usd", "down_payment_pct"],
            "synthesis": ["budget_usd", "down_payment_pct", "horizon_years", "risk_tolerance"]}
UPSTREAM_KINDS = {"insurance_finance": ["risk_assessment", "market_summary"]}


def initial_coord_state() -> Dict[str, Any]:
    return {"phase": "constraints", "constraints": {}, "branch": {"city": None, "scenario": None},
            "tasks": {}, "artifacts": {}, "evidence_gaps": {}, "findings": {}, "pending_action": None}


# --- investigation invariants (checked on every merged state, before commit) ---------
# Each message says how to fix the state: on a rejected patch, this message is all the
# model is shown of what went wrong.
def done_tasks_cite_artifacts(s):
    return [f"task '{t}' has status 'done' but its `artifacts` list is empty: set tasks.{t}.artifacts to "
            f"the id(s) of the artifact it produced, e.g. [\"a_...\"], or set its status to 'failed'"
            for t, v in s.get("tasks", {}).items() if v.get("status") == "done" and not v.get("artifacts")]


def artifact_refs_resolve(s):
    known = set(s.get("artifacts", {}))
    return [f"task '{t}' cites artifact '{a}', which is not in `artifacts`: add it there with its kind "
            f"and summary, or remove it from the task" for t, v in s.get("tasks", {}).items()
            for a in v.get("artifacts", []) if a not in known]


def no_synthesis_while_running(s):
    running = [t for t, v in s.get("tasks", {}).items()
               if v.get("status") == "running" and v.get("agent") != "synthesis"]
    if s.get("phase") in ("synthesis", "approval", "done") and running:
        return [f"phase '{s['phase']}' while tasks {running} are running: record their results first, "
                f"or keep phase 'research'"]
    return []


def action_only_in_approval(s):
    if s.get("pending_action") and s.get("phase") != "approval":
        return ["pending_action may only be set in phase 'approval': set phase to 'approval', "
                "or set pending_action to null"]
    return []


COORD_CHECKS = [done_tasks_cite_artifacts, artifact_refs_resolve, no_synthesis_while_running,
                action_only_in_approval]

# --- an agent's working state -----------------------------------------------------
AGENT_SCHEMA = {
    "type": "object", "additionalProperties": False, "required": ["notes", "gaps"],
    "properties": {
        "notes": {"type": "object", "additionalProperties": {"type": "string"}},
        "gaps": {"type": "object", "additionalProperties": {"type": "string"}},
        "draft": {"type": "object", "additionalProperties": False,
                  "properties": {"kind": {"type": "string"}, "summary": {"type": "string"},
                                 "content": {}}},
    },
}


def initial_agent_state() -> Dict[str, Any]:
    return {"notes": {}, "gaps": {}}


def agent_checks(agent: str):
    def check(s):
        kind = (s.get("draft") or {}).get("kind")
        return [f"draft kind must be '{OUTPUT_KIND[agent]}', not '{kind}'"] \
            if kind and kind != OUTPUT_KIND[agent] else []
    check.__name__ = f"draft_kind_{agent}"
    return [check]


AGENT_PROTOCOL = """## How you act

One action per turn, written exactly as one of:
- `call <tool> <JSON arguments>`, for example `call hazard_profile {"city": "Kent"}`
- `finish`, once `draft` in your state holds the artifact

Tools you may call, with their arguments:
TOOLS

## Your state

- `notes`: {"<key>": "<what you learned>"}. Earlier tool results are not shown again,
  so record what you need from each result here.
- `gaps`: {"<key>": "<what is missing and what it prevents>"}.
- `draft`: {"kind": "KIND", "summary": "<one line>", "content": {...}}, the artifact
  you return."""

COORD_PROTOCOL = """## How you act

One action per turn, written exactly as one of:
- `dispatch_agent <agent> <task_id>`, where agent is one of market_research,
  quantitative_analysis, environmental_risk, insurance_finance, synthesis
- `request_approval <kind>` or `submit_inquiry <kind>`, kind mortgage_inquiry
- `wait`, when you need the buyer
- `done`, when the branch has its report or cannot proceed

A dispatch returns the agent's artifact id, kind and summary, or why it failed. Before
dispatching, add the task to `tasks` with status `running`. When the result arrives,
record it in the same patch in both places:

    "artifacts": {"a_123": {"kind": "market_summary", "summary": "..."}},
    "tasks": {"t1": {"agent": "market_research", "status": "done", "artifacts": ["a_123"]}}

A task the agent could not complete gets status `failed` and a `note`. Dispatching a
task id again returns that task's existing result; use a new task id to redo work.

An approval covers one exact inquiry: the city, scenario, constraints and
`pending_action.summary` at the time you request it. Changing any of them before
`submit_inquiry` voids the approval."""


def agent_instructions(agent: str, city: str, constraints: Dict, upstream: Dict[str, Dict],
                       granted: List[str], extra_facts: Optional[Dict] = None,
                       skill_ref: Optional[str] = None) -> Surface:
    facts = {"agent": agent, "city": city,
             **{k: constraints[k] for k in CONSUMES[agent] if k in constraints}}
    if upstream:
        facts["upstream artifacts"] = compact(upstream, 3000)
    facts.update(extra_facts or {})
    s = compile_surface(ALWAYS + [ARTIFACTS[skill_ref or AGENT_SKILL[agent]]], facts)
    tools = "\n".join(f"- {tool_usage(t)}" for t in sorted(granted))
    protocol = AGENT_PROTOCOL.replace("TOOLS", tools).replace("KIND", OUTPUT_KIND[agent])
    return Surface(s.text + "\n\n" + protocol, s.manifest)


def coordinator_instructions() -> Surface:
    s = compile_surface(ALWAYS + [ARTIFACTS[COORD_SKILL]])
    return Surface(s.text + "\n\n" + COORD_PROTOCOL, s.manifest)


def granted_tools(skill_name: str, implemented: set) -> List[str]:
    """requested by the skill, intersected with what this harness implements"""
    return sorted(set(ARTIFACTS[skill_name].requires.get("tools", [])) & implemented)


def artifact_id(kind: str, content: Any, inputs: Dict) -> str:
    raw = json.dumps([kind, content, inputs], sort_keys=True, default=str)
    return "a_" + hashlib.sha256(raw.encode()).hexdigest()[:10]


def parse_call(action: str) -> Optional[Tuple[str, Dict]]:
    m = re.match(r"^call\s+(\w+)\s*(\{.*\})?\s*$", action.strip(), re.S)
    if not m:
        return None
    try:
        return m.group(1), (json.loads(m.group(2)) if m.group(2) else {})
    except json.JSONDecodeError:
        return None


def describe_result(agent: str, task_id: str, r: Dict) -> str:
    gaps = "; ".join(f"{k}: {v}" for k, v in (r.get("gaps") or {}).items())
    if r["status"] == "done":
        text = (f"Task {task_id} ({agent}) is done. Artifact {r['artifact_id']} "
                f"({r['kind']}): {r['summary']}")
    else:
        text = f"Task {task_id} ({agent}) {r['status']}: {r.get('reason', 'no artifact produced')}"
    return text + (f" Evidence gaps reported: {gaps}" if gaps else "")


IMPLEMENTED_TOOLS = set(TOOLS)


class MemoryArtifacts:
    """A minimal in-memory artifact store for this section; section 3 replaces it with SQLite."""

    def __init__(self):
        self.items: Dict[str, Dict] = {}

    def put(self, kind, summary, content, inputs, produced_by) -> str:
        aid = artifact_id(kind, content, inputs)
        self.items.setdefault(aid, {"kind": kind, "summary": summary, "content": content,
                                    "inputs": inputs, "produced_by": produced_by})
        return aid

    def read(self, ids) -> Dict[str, Any]:
        return {i: {k: self.items[i][k] for k in ("kind", "summary", "content")}
                for i in ids if i in self.items}

    def latest_of_kinds(self, refs: Dict[str, Dict], kinds) -> Dict[str, Dict]:
        """The newest referenced artifact of each kind (refs come from the coordinator's state)."""
        out = {}
        for aid in refs:
            if aid in self.items and self.items[aid]["kind"] in kinds:
                out[self.items[aid]["kind"]] = {"id": aid, **self.read([aid])[aid]}
        return out


class AgentEnv:
    """The skill_state Environment one agent acts in: its actions are tool calls."""

    def __init__(self, agent, task_id, city, granted, read_artifacts):
        self.agent, self.task_id, self.city = agent, task_id, city
        self.granted, self.read_artifacts = set(granted), read_artifacts
        self.state, self.finished, self.tools_used, self.ids_read = None, False, set(), set()

    def capture(self, step):                      # on_step hook: the committed state
        self.state = step["state"]

    def reset(self) -> str:
        return f"Task {self.task_id}: {self.agent} for {self.city}. Begin."

    def step(self, action: str) -> Tuple[str, bool]:
        if action.strip() == "finish":
            self.finished = True
            return "Finished.", True
        call = parse_call(action)
        if call is None:
            return f"ERROR: unrecognised action '{action}'. Use `call <tool> <JSON>` or `finish`.", False
        name, args = call
        if name not in self.granted:
            return f"ERROR: tool '{name}' is not available to you.", False
        try:
            result = run_tool(name, args, self.read_artifacts)
        except ToolUnavailable as e:
            return f"ERROR (source unavailable): {e}", False
        except Exception as e:
            return f"ERROR: {type(e).__name__}: {e}", False
        self.tools_used.add(name)
        if name == "read_artifacts":
            self.ids_read.update(args.get("ids", []))
        return compact(result), False


def artifact_inputs(agent, city, constraints, tools_used, upstream_ids) -> Dict[str, Any]:
    evidence = sorted({e for t in tools_used for e in TOOLS[t].reads})
    return {"city": city,
            "constraints": {k: constraints[k] for k in CONSUMES[agent] if k in constraints},
            "evidence": {e: EVIDENCE[e]["sha256"] for e in evidence},
            "upstream": sorted(upstream_ids)}


def run_agent_paper(agent, task_id, city, constraints, store: MemoryArtifacts, refs) -> Dict:
    upstream = store.latest_of_kinds(refs, UPSTREAM_KINDS.get(agent, []))
    extra = {}
    if agent == "synthesis":
        extra["artifacts in this branch"] = compact({i: store.items[i]["kind"] for i in refs
                                                    if i in store.items})
    granted = granted_tools(AGENT_SKILL[agent], IMPLEMENTED_TOOLS)
    surface = agent_instructions(agent, city, constraints, upstream, granted, extra)
    env = AgentEnv(agent, task_id, city, granted, store.read)
    skill = Skill(name=agent, instructions=surface.text, schema=AGENT_SCHEMA,
                  initial_state=initial_agent_state(), checks=agent_checks(agent))
    SkillStateRuntime(skill, llm.with_label(f"paper:{agent}:{city}"), max_retries=2,
                      on_step=env.capture).run(env, max_steps=12)
    state = env.state or initial_agent_state()
    draft = state.get("draft") or {}
    if not (env.finished and draft.get("kind") == OUTPUT_KIND[agent]):
        return {"status": "failed", "reason": "no artifact within 12 steps", "gaps": state.get("gaps")}
    ids = {u["id"] for u in upstream.values()} | env.ids_read
    aid = store.put(draft["kind"], draft.get("summary", ""), draft.get("content"),
                    artifact_inputs(agent, city, constraints, env.tools_used, ids),
                    {"agent": agent, "skill": AGENT_SKILL[agent],
                     "version": ARTIFACTS[AGENT_SKILL[agent]].version})
    return {"status": "done", "artifact_id": aid, "kind": draft["kind"],
            "summary": draft.get("summary", ""), "gaps": state.get("gaps")}


class CoordinatorEnv:
    """The skill_state Environment for the coordinator: its actions run agents."""

    def __init__(self, city, constraints, store, agent_runner):
        self.city, self.constraints, self.store, self.run_agent = city, constraints, store, agent_runner
        self.refs: Dict[str, Dict] = {}          # artifacts this branch has produced (harness record)

    def reset(self) -> str:
        return (f"Buyer constraints: {json.dumps(self.constraints)}. This branch researches "
                f"{self.city}. Record the constraints and the branch, then run the research.")

    def step(self, action: str) -> Tuple[str, bool]:
        parts = action.split()
        verb = parts[0] if parts else ""
        if verb == "done":
            return "Branch finished.", True
        if verb == "wait":
            return "Nothing new has arrived. Use `done` if the branch cannot proceed.", False
        if verb == "dispatch_agent" and len(parts) == 3 and parts[1] in AGENT_SKILL:
            r = self.run_agent(parts[1], parts[2], self.city, self.constraints, self.store, self.refs)
            if r["status"] == "done":
                self.refs[r["artifact_id"]] = {"kind": r["kind"]}
            return describe_result(parts[1], parts[2], r), False
        return f"ERROR: action '{action}' is not available in this run.", False


AGENT_CACHE: Dict[Tuple, Dict] = {}


def cached_agent_runner(agent, task_id, city, constraints, store, refs) -> Dict:
    """Reuse an agent's result when agent, city, constraints and upstream refs are unchanged.
    Used by the transcript comparison, so both runs see the same agent results."""
    upstream = store.latest_of_kinds(refs, UPSTREAM_KINDS.get(agent, []))
    key = (agent, city, json.dumps({k: constraints.get(k) for k in CONSUMES[agent]}, sort_keys=True),
           tuple(sorted(u["id"] for u in upstream.values())))
    if key not in AGENT_CACHE:
        AGENT_CACHE[key] = run_agent_paper(agent, task_id, city, constraints, store, refs)
    return AGENT_CACHE[key]

In [ ]:
# @title Harness from notebook 1: middleware, store, LangGraph kernel, agents and coordinator
from collections import Counter


class Status(str, Enum):
    CREATED = "CREATED"
    RUNNING = "RUNNING"
    INTERRUPTED = "INTERRUPTED"
    RECOVERING = "RECOVERING"
    SUSPENDED = "SUSPENDED"
    COMPLETED = "COMPLETED"
    FAILED = "FAILED"


# The only status changes the kernel may make. FAILED is terminal; a SUSPENDED or COMPLETED
# execution is reopened through RECOVERING; RUNNING -> RECOVERING is a restart after a failure.
TRANSITIONS = {
    Status.CREATED: {Status.RUNNING},
    Status.RUNNING: {Status.INTERRUPTED, Status.SUSPENDED, Status.COMPLETED, Status.RECOVERING},
    Status.INTERRUPTED: {Status.RECOVERING, Status.SUSPENDED, Status.FAILED},
    Status.RECOVERING: {Status.RUNNING},
    Status.SUSPENDED: {Status.RECOVERING},
    Status.COMPLETED: {Status.RECOVERING},
    Status.FAILED: set(),
}


class Kind(str, Enum):
    LOOP_DETECTED = "LOOP_DETECTED"
    SCHEMA_VIOLATION = "SCHEMA_VIOLATION"
    POLICY_DENIED = "POLICY_DENIED"
    BUDGET_EXCEEDED = "BUDGET_EXCEEDED"
    STATE_CONFLICT = "STATE_CONFLICT"
    APPROVAL_REQUIRED = "APPROVAL_REQUIRED"
    STREAM_ABORTED = "STREAM_ABORTED"
    SOURCE_UNAVAILABLE = "SOURCE_UNAVAILABLE"
    OUTCOME_UNKNOWN = "OUTCOME_UNKNOWN"
    PIN_MISMATCH = "PIN_MISMATCH"


class SemanticInterrupt(Exception):
    """Raised by middleware, by an action, or by the kernel when execution must stop or change course."""

    def __init__(self, kind: Kind, message: str, hook: str, **detail):
        super().__init__(f"{kind.value}: {message}")
        self.kind, self.message, self.hook, self.detail = kind, message, hook, detail


class ProcessKilled(BaseException):
    """A simulated process failure. Not an interrupt: nothing in the kernel handles it."""


# --- what an action does, and what recovery may therefore do with it ------------------
class Effect(str, Enum):
    READ = "read"               # no side effects
    IDEMPOTENT = "idempotent"   # side effects the harness keys and deduplicates
    EXTERNAL = "external"       # reaches a system outside the harness
    CONTROL = "control"         # changes only this execution's own status
    UNKNOWN = "unknown"         # not classified: its outcome cannot be established


RECOVERY = {Effect.READ: "replay", Effect.IDEMPOTENT: "replay", Effect.CONTROL: "replay",
            Effect.EXTERNAL: "reconcile", Effect.UNKNOWN: "escalate"}


def _canonical(value):
    if isinstance(value, float) and value.is_integer():
        return int(value)
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, dict):
        return {k.strip(): _canonical(v) for k, v in value.items()}
    if isinstance(value, list):
        return [_canonical(v) for v in value]
    return value


def action_name(action: str) -> str:
    call = parse_call(action)
    return call[0] if call else (action.split() or [""])[0]


def normalise_action(action: str) -> str:
    """One spelling per call: key order, whitespace and 850000 vs 850000.0 do not matter;
    any difference in a value does."""
    call = parse_call(action)
    if call:
        return f"call {call[0]} {json.dumps(_canonical(call[1]), sort_keys=True)}"
    return " ".join(action.split())


def digest_text(text: str) -> str:
    return hashlib.sha256((text or "").encode()).hexdigest()[:12]


class Middleware:
    """Synchronous hooks. Each receives the kernel, which exposes the current state, the
    step number, the harness's record of completed actions, and the store."""

    def on_pre_inference(self, k): ...
    def on_stream_start(self, k): ...
    def on_stream_chunk(self, k, piece: str): ...
    def on_post_inference(self, k, text: str): ...
    def on_pre_commit(self, k, candidate: Dict, patch: Dict): ...
    def on_pre_action(self, k, action: str): ...


class BudgetMiddleware(Middleware):
    def __init__(self, max_steps: int, max_prompt_tokens: Optional[int] = None):
        self.max_steps, self.max_prompt_tokens = max_steps, max_prompt_tokens

    def on_pre_inference(self, k):
        if k.step_no >= self.max_steps:
            raise SemanticInterrupt(Kind.BUDGET_EXCEEDED, f"{k.step_no} of {self.max_steps} steps used",
                                    "pre_inference")
        if self.max_prompt_tokens and k.prompt_tokens_used >= self.max_prompt_tokens:
            raise SemanticInterrupt(Kind.BUDGET_EXCEEDED,
                                    f"{k.prompt_tokens_used} of {self.max_prompt_tokens} prompt tokens used",
                                    "pre_inference")


class StreamGuardMiddleware(Middleware):
    """Stops a generation that runs away or repeats itself, while it is being produced.

    Each piece is scanned once, from left to right, with str.find; nothing is re-read or
    re-concatenated. Memory is bounded whatever the output: the current line keeps at most
    `max_line` characters, and complete lines are counted by a short digest, not stored.
    Pieces arrive as decoded text (the client parses each stream event as JSON), so a piece
    never ends inside a character; line endings can still be \\n, \\r\\n or \\r, and a
    \\r\\n can be split across two pieces."""

    def __init__(self, max_chars: int = 8000, max_repeats: int = 8, min_line: int = 40, max_line: int = 2000):
        self.max_chars, self.max_repeats, self.min_line, self.max_line = max_chars, max_repeats, min_line, max_line

    def on_stream_start(self, k):
        self.chars, self.line, self.line_len, self.after_cr, self.lines = 0, [], 0, False, Counter()

    def on_stream_chunk(self, k, piece: str):
        self.chars += len(piece)
        if self.chars > self.max_chars:
            raise SemanticInterrupt(Kind.STREAM_ABORTED, f"output exceeded {self.max_chars} characters", "stream")
        i, n = 0, len(piece)
        if self.after_cr and n and piece[0] == "\n":       # the second half of a \r\n split across pieces
            i = 1
        self.after_cr = False
        while i < n:
            j = min((x for x in (piece.find("\n", i), piece.find("\r", i)) if x >= 0), default=-1)
            if j < 0:
                self._extend(piece, i, n)
                break
            self._extend(piece, i, j)
            self._end_line(k)
            if piece[j] == "\r":
                if j + 1 < n and piece[j + 1] == "\n":
                    j += 1
                elif j + 1 == n:
                    self.after_cr = True
            i = j + 1

    def _extend(self, piece: str, i: int, j: int):
        room = self.max_line - self.line_len
        if room > 0:
            self.line.append(piece[i:min(j, i + room)])
        self.line_len += j - i

    def _end_line(self, k):
        text = "".join(self.line).strip()
        self.line, self.line_len = [], 0
        if len(text) > self.min_line:
            key = hashlib.blake2b(text.encode(), digest_size=8).digest()
            self.lines[key] += 1
            if self.lines[key] >= self.max_repeats:
                raise SemanticInterrupt(Kind.STREAM_ABORTED,
                                        f"a line repeated {self.max_repeats} times in the output", "stream")


class PermissionMiddleware(Middleware):
    def __init__(self, granted: set):
        self.granted = set(granted)

    def on_pre_action(self, k, action: str):
        name = action_name(action)
        if name not in self.granted:
            raise SemanticInterrupt(Kind.POLICY_DENIED, f"'{name}' is not granted to {k.role}",
                                    "pre_action", action=action)


class LoopDetectorMiddleware(Middleware):
    """Repetition of one call (after normalisation), judged by what the earlier attempts returned:

    - it failed every time before: refused at the `fail_limit`-th attempt (a retry loop);
    - it returned the same result every time before: refused at the `stall_limit`-th attempt
      (no new information);
    - it returned different results: allowed, as polling a changing status is, up to `poll_limit`.

    The history is the completed actions in the store. An action left unfinished by a process
    failure has no completion, so its replay on resume is counted once, when it completes."""

    def __init__(self, fail_limit: int = 3, stall_limit: int = 3, poll_limit: int = 10):
        self.fail_limit, self.stall_limit, self.poll_limit = fail_limit, stall_limit, poll_limit

    def on_pre_action(self, k, action: str):
        norm = normalise_action(action)
        prior = [h for h in k.history if h["norm"] == norm]
        if not prior:
            return
        n = len(prior) + 1
        failed = [h for h in prior if h["observation"].startswith("ERROR")]
        if len(failed) == len(prior) and n >= self.fail_limit:
            raise SemanticInterrupt(
                Kind.LOOP_DETECTED, f"'{action[:80]}' failed {len(prior)} times before", "pre_action",
                action=action, pattern="repeated_failure",
                source_unavailable=prior[-1]["observation"].startswith("ERROR (source unavailable)"))
        if not failed and len({h["digest"] for h in prior}) == 1 and n >= self.stall_limit:
            raise SemanticInterrupt(Kind.LOOP_DETECTED,
                                    f"'{action[:80]}' returned the same result {len(prior)} times",
                                    "pre_action", action=action, pattern="no_new_information")
        if n > self.poll_limit:
            raise SemanticInterrupt(Kind.LOOP_DETECTED, f"'{action[:80]}' attempted {n} times",
                                    "pre_action", action=action, pattern="poll_limit")


class FinishCheckMiddleware(Middleware):
    """An agent may only finish with a complete draft of the artifact kind its skill promises."""

    def __init__(self, agent: str):
        self.agent = agent

    def on_pre_action(self, k, action: str):
        if action.strip() != "finish":
            return
        d = k.candidate.get("draft") or {}
        if d.get("kind") != OUTPUT_KIND[self.agent] or not d.get("summary") or d.get("content") in (None, {}, []):
            raise SemanticInterrupt(Kind.STATE_CONFLICT,
                                    f"finish needs a draft with kind '{OUTPUT_KIND[self.agent]}', a summary "
                                    "and content", "pre_action")


class ApprovalGateMiddleware(Middleware):
    """External submissions need a person's approval of exactly the payload that would be
    sent, recorded in the store. The state's own fields are not evidence: the model writes
    the state. `payload_for(state, kind)` says what would be sent."""

    GATED = {"submit_inquiry"}

    def __init__(self, payload_for: Callable[[Dict, str], Dict]):
        self.payload_for = payload_for

    def on_pre_action(self, k, action: str):
        parts = action.split()
        if parts and parts[0] in self.GATED:
            kind = parts[1] if len(parts) > 1 else ""
            payload = self.payload_for(k.candidate, kind)
            ok, reason = k.store.check_approval(k.execution_id, f"{parts[0]} {kind}", payload,
                                                now=getattr(k, "now", time.time)())
            if not ok:
                raise SemanticInterrupt(Kind.POLICY_DENIED, reason, "pre_action", action=action)


# What the kernel does with each kind of interrupt.
DEFAULT_POLICY = {
    Kind.SCHEMA_VIOLATION: "escalate",              # only raised once the in-step retries are spent
    Kind.STATE_CONFLICT: "redirect_or_escalate",    # escalate when the in-step retries are spent
    Kind.POLICY_DENIED: "redirect",
    Kind.LOOP_DETECTED: "suspend_or_redirect",   # suspend if the source is down, else redirect
    Kind.BUDGET_EXCEEDED: "suspend",
    Kind.APPROVAL_REQUIRED: "escalate",
    Kind.STREAM_ABORTED: "retry",
    Kind.SOURCE_UNAVAILABLE: "suspend",
    Kind.OUTCOME_UNKNOWN: "escalate",
    Kind.PIN_MISMATCH: "escalate",
}
MAX_REDIRECTS, MAX_RETRIES = 2, 1


class Faults:
    """Simulated process failures for the demonstrations. ProcessKilled is not handled by
    the kernel: the Python objects are lost, and only what reached SQLite survives."""

    def __init__(self, kill_after_commits: Optional[int] = None, kill_after_send: bool = False):
        self.kill_after_commits, self.kill_after_send, self.commits = kill_after_commits, kill_after_send, 0

    def after_commit(self, k):
        self.commits += 1
        if self.kill_after_commits and self.commits == self.kill_after_commits:
            raise ProcessKilled(f"process killed after commit {self.commits} ({k.role}, step {k.step_no})")

    def after_send(self):
        if self.kill_after_send:
            self.kill_after_send = False
            raise ProcessKilled("process killed after the request was sent, before its result was recorded")


class Store:
    """The harness's own records, in the same SQLite file as LangGraph's checkpoints: branches,
    executions, the action log, artifacts, events, the journal and approvals. LangGraph's
    SqliteSaver keeps the checkpoints (its tables `checkpoints` and `writes`)."""

    DDL = """
    CREATE TABLE IF NOT EXISTS branches (name TEXT PRIMARY KEY, parent TEXT, from_checkpoint TEXT,
        assumptions TEXT, created REAL);
    CREATE TABLE IF NOT EXISTS executions (id TEXT PRIMARY KEY, branch TEXT, role TEXT,
        parent_execution TEXT, task_id TEXT, status TEXT, config TEXT, initial_state TEXT, created REAL);
    CREATE TABLE IF NOT EXISTS actions (commit_id TEXT PRIMARY KEY, execution_id TEXT, step INTEGER,
        action TEXT, status TEXT, observation TEXT, outcome TEXT, started REAL, finished REAL);
    CREATE TABLE IF NOT EXISTS artifacts (id TEXT PRIMARY KEY, kind TEXT, branch TEXT, summary TEXT,
        content TEXT, inputs TEXT, produced_by TEXT, created REAL);
    CREATE TABLE IF NOT EXISTS events (seq INTEGER PRIMARY KEY AUTOINCREMENT, execution_id TEXT,
        branch TEXT, role TEXT, kind TEXT, detail TEXT, created REAL);
    CREATE TABLE IF NOT EXISTS journal (key TEXT PRIMARY KEY, execution_id TEXT, action TEXT,
        payload TEXT, status TEXT, result TEXT, created REAL, updated REAL);
    CREATE TABLE IF NOT EXISTS approvals (id TEXT PRIMARY KEY, execution_id TEXT, branch TEXT,
        action TEXT, payload TEXT, payload_digest TEXT, status TEXT, decided_by TEXT, created REAL,
        decided REAL, expires REAL);
    """

    def __init__(self, path: Path):
        self.path = Path(path)
        self.db = sqlite3.connect(self.path, check_same_thread=False, timeout=30)
        self.db.row_factory = sqlite3.Row
        self.db.execute("PRAGMA journal_mode=WAL")
        self.db.executescript(self.DDL)
        self._seq: Counter = Counter()

    # -- helpers ------------------------------------------------------------------
    def _x(self, sql, *args):
        cur = self.db.execute(sql, args)
        self.db.commit()
        return cur

    def _one(self, sql, *args) -> Optional[Dict]:
        row = self.db.execute(sql, args).fetchone()
        return dict(row) if row else None

    def df(self, table: str) -> pd.DataFrame:
        return pd.read_sql_query(f"SELECT * FROM {table}", self.db)

    def _id(self, prefix: str) -> str:
        """Sequential ids. A run that repeats the same steps gets the same ids, which is
        what lets a recorded run be replayed: ids appear in what the model reads."""
        self._seq[prefix] += 1
        return f"{prefix}_{self._seq[prefix]:05d}"

    # -- branches and executions ----------------------------------------------------
    def add_branch(self, name, parent=None, from_checkpoint=None, assumptions=None):
        self._x("INSERT OR IGNORE INTO branches VALUES (?,?,?,?,?)", name, parent, from_checkpoint,
                json.dumps(assumptions or {}), time.time())

    def lineage(self, branch: str) -> List[str]:
        out = []
        while branch:
            out.append(branch)
            row = self._one("SELECT parent FROM branches WHERE name=?", branch)
            branch = row["parent"] if row else None
        return out

    def new_execution(self, branch, role, config, initial_state, parent_execution=None, task_id=None) -> str:
        eid = self._id("ex")
        self._x("INSERT INTO executions VALUES (?,?,?,?,?,?,?,?,?)", eid, branch, role, parent_execution,
                task_id, Status.CREATED.value, json.dumps(config), json.dumps(initial_state), time.time())
        return eid

    def execution(self, eid) -> Dict:
        row = self._one("SELECT * FROM executions WHERE id=?", eid)
        row["config"], row["initial_state"] = json.loads(row["config"]), json.loads(row["initial_state"])
        return row

    def executions(self, **where) -> List[Dict]:
        sql = "SELECT * FROM executions" + (" WHERE " + " AND ".join(f"{k}=?" for k in where) if where else "")
        rows = [dict(r) for r in self.db.execute(sql + " ORDER BY created", tuple(where.values()))]
        for r in rows:
            r["config"], r["initial_state"] = json.loads(r["config"]), json.loads(r["initial_state"])
        return rows

    def set_status(self, eid, status: Status):
        self._x("UPDATE executions SET status=? WHERE id=?", status.value, eid)

    # -- the action log: one row per committed action, keyed by the commit's checkpoint id ---
    def action_row(self, commit_id) -> Optional[Dict]:
        return self._one("SELECT * FROM actions WHERE commit_id=?", commit_id)

    def action_started(self, commit_id, eid, step, action):
        self._x("INSERT INTO actions VALUES (?,?,?,?,?,?,?,?,?)", commit_id, eid, step, action, "started",
                None, None, time.time(), None)

    def action_completed(self, commit_id, observation, outcome):
        self._x("UPDATE actions SET status='completed', observation=?, outcome=?, finished=? WHERE commit_id=?",
                observation, outcome, time.time(), commit_id)

    # -- artifacts --------------------------------------------------------------------
    def put_artifact(self, kind, branch, summary, content, inputs, produced_by) -> str:
        aid = artifact_id(kind, content, inputs)
        self._x("INSERT OR IGNORE INTO artifacts VALUES (?,?,?,?,?,?,?,?)", aid, kind, branch, summary,
                json.dumps(content, default=str), json.dumps(inputs), json.dumps(produced_by), time.time())
        return aid

    def artifact(self, aid) -> Optional[Dict]:
        row = self._one("SELECT * FROM artifacts WHERE id=?", aid)
        if row:
            for k in ("content", "inputs", "produced_by"):
                row[k] = json.loads(row[k])
        return row

    def artifact_of_execution(self, eid) -> Optional[Dict]:
        row = self._one("SELECT id FROM artifacts WHERE json_extract(produced_by, '$.execution')=?", eid)
        return self.artifact(row["id"]) if row else None

    def visible(self, branch, aid) -> bool:
        a = self._one("SELECT branch FROM artifacts WHERE id=?", aid)
        return bool(a) and a["branch"] in self.lineage(branch)

    def read_artifacts(self, branch, ids) -> Dict[str, Any]:
        return {i: {k: a[k] for k in ("kind", "summary", "content")}
                for i in ids if self.visible(branch, i) and (a := self.artifact(i))}

    def latest_of_kinds(self, branch, ids, kinds) -> Dict[str, Dict]:
        found = {}
        for aid in ids:
            a = self.artifact(aid) if self.visible(branch, aid) else None
            if a and a["kind"] in kinds and (a["kind"] not in found or a["created"] > found[a["kind"]]["created"]):
                found[a["kind"]] = a
        return {k: {"id": a["id"], "summary": a["summary"], "content": a["content"]} for k, a in found.items()}

    # -- events, journal, approvals --------------------------------------------------
    def event(self, eid, branch, role, event_kind, **detail):
        self._x("INSERT INTO events (execution_id, branch, role, kind, detail, created) VALUES (?,?,?,?,?,?)",
                eid, branch, role, event_kind, json.dumps(detail, default=str), time.time())

    def journal_get(self, key) -> Optional[Dict]:
        row = self._one("SELECT * FROM journal WHERE key=?", key)
        if row:
            row["payload"] = json.loads(row["payload"])
            row["result"] = json.loads(row["result"]) if row["result"] else None
        return row

    def journal_intent(self, key, eid, action, payload):
        self._x("INSERT INTO journal VALUES (?,?,?,?,?,?,?,?)", key, eid, action, json.dumps(payload),
                "intent", None, time.time(), time.time())

    def journal_done(self, key, result):
        self._x("UPDATE journal SET status='done', result=?, updated=? WHERE key=?", json.dumps(result),
                time.time(), key)

    def request_approval(self, eid, branch, action, payload) -> str:
        """A request for a person to approve one exact payload. A pending request for a
        different payload of the same action is superseded."""
        digest = payload_digest(payload)
        pending = self.db.execute("SELECT id, payload_digest FROM approvals WHERE execution_id=? AND action=? "
                                  "AND status='pending'", (eid, action)).fetchall()
        for r in pending:
            if r["payload_digest"] == digest:
                return r["id"]
            self._x("UPDATE approvals SET status='superseded' WHERE id=?", r["id"])
        aid = self._id("ap")
        self._x("INSERT INTO approvals VALUES (?,?,?,?,?,?,?,?,?,?,?)", aid, eid, branch, action,
                json.dumps(payload, sort_keys=True), digest, "pending", None, time.time(), None, None)
        return aid

    def approve(self, approval_id, decided_by: str, valid_for_s: float = 24 * 3600):
        now = time.time()
        self._x("UPDATE approvals SET status='approved', decided_by=?, decided=?, expires=? WHERE id=? "
                "AND status='pending'", decided_by, now, now + valid_for_s, approval_id)

    def approval(self, approval_id) -> Optional[Dict]:
        row = self._one("SELECT * FROM approvals WHERE id=?", approval_id)
        if row:
            row["payload"] = json.loads(row["payload"])
        return row

    def check_approval(self, eid, action, payload, now=None) -> Tuple[bool, str]:
        """Is there an approval by a person, for this execution, this action and exactly this
        payload, that has not expired? Returns (ok, reason)."""
        now = time.time() if now is None else now
        rows = [dict(r) for r in self.db.execute(
            "SELECT * FROM approvals WHERE execution_id=? AND action=? AND status='approved' "
            "ORDER BY decided DESC", (eid, action))]
        if not rows:
            return False, f"no recorded approval for '{action}' in this execution"
        digest = payload_digest(payload)
        match = [r for r in rows if r["payload_digest"] == digest]
        if not match:
            return False, (f"the approved payload ({rows[0]['payload_digest']}) differs from the one "
                           f"about to be sent ({digest})")
        live = [r for r in match if r["expires"] and r["expires"] > now]
        if not live:
            return False, f"the approval {match[0]['id']} expired"
        return True, f"approved by {live[0]['decided_by']} ({live[0]['id']}, payload {digest})"

    def valid_approval(self, eid, action, payload, now=None) -> Optional[Dict]:
        ok, _ = self.check_approval(eid, action, payload, now)
        if not ok:
            return None
        return self._one("SELECT * FROM approvals WHERE execution_id=? AND action=? AND status='approved' "
                         "AND payload_digest=? ORDER BY decided DESC LIMIT 1", eid, action, payload_digest(payload))

    def pending_approvals(self) -> List[Dict]:
        return [dict(r) for r in self.db.execute("SELECT * FROM approvals WHERE status='pending'")]


def payload_digest(payload: Dict) -> str:
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]


def history_entry(action: str, observation: str) -> Dict[str, str]:
    return {"action": action, "norm": normalise_action(action), "observation": (observation or "")[:300],
            "digest": digest_text(observation)}


import contextvars
from typing import TypedDict

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


def schema_digest(schema: Dict) -> str:
    return hashlib.sha256(json.dumps(schema, sort_keys=True).encode()).hexdigest()[:12]


def pin_config(skill_name: str, schema: Dict, instructions: str) -> Dict[str, Any]:
    """What an execution is pinned to: the model, the skill version and digest, the always-loaded
    artifacts, the state schema, and the exact instructions compiled from them."""
    a = ARTIFACTS[skill_name]
    return {"model": MODEL, "skill": {"name": a.name, "version": a.version, "digest": a.digest},
            "always": {x.name: x.digest for x in ALWAYS}, "schema_digest": schema_digest(schema),
            "instructions_digest": digest_text(instructions)}


def pin_mismatches(pinned: Dict, current: Dict) -> List[Dict[str, Any]]:
    """Every pinned field whose current value differs."""
    out = []
    for key in ("model", "schema_digest", "instructions_digest"):
        if pinned.get(key) != current.get(key):
            out.append({"pin": key, "pinned": pinned.get(key), "current": current.get(key)})
    if pinned["skill"] != current["skill"]:
        out.append({"pin": "skill", "pinned": pinned["skill"], "current": current["skill"]})
    for name in sorted(set(pinned["always"]) | set(current["always"])):
        if pinned["always"].get(name) != current["always"].get(name):
            out.append({"pin": f"always:{name}", "pinned": pinned["always"].get(name),
                        "current": current["always"].get(name)})
    return out


@dataclass
class Diagnostic:
    code: str          # E1xx response and JSON, E2xx schema, E3xx invariants
    where: str
    message: str
    hint: str = ""
    excerpt: List[str] = field(default_factory=list)

    def render(self) -> str:
        lines = [f"{self.code} {self.where}: {self.message}"] + [f"    {x}" for x in self.excerpt]
        return "\n".join(lines + ([f"    help: {self.hint}"] if self.hint else []))


_FENCED = re.compile(r"```(?:json)?\s*\n?(.*?)```", re.S | re.I)


def _schema_at(schema: Dict, parts: List[str]) -> Optional[Dict]:
    """The sub-schema at a state path such as ['tasks', 't2'], or None."""
    for part in parts:
        if schema is None:
            return None
        if part.startswith("["):
            schema = schema.get("items")
        elif part in schema.get("properties", {}):
            schema = schema["properties"][part]
        elif isinstance(schema.get("additionalProperties"), dict):
            schema = schema["additionalProperties"]
        else:
            return None
    return schema


def _schema_diagnostic(error: str, schema: Dict) -> Diagnostic:
    path, _, message = error.partition(": ")
    parts = re.findall(r"\[\d+\]|[^.\[\]$]+", path)
    where = "state." + ".".join(parts) if parts else "state"
    if message == "key not allowed by schema":
        parent = _schema_at(schema, parts[:-1]) or {}
        allowed = sorted(parent.get("properties", {}))
        return Diagnostic("E202", where, "key not allowed by the state schema",
                          f"remove '{parts[-1]}'; allowed here: {', '.join(allowed) or 'none'}")
    if message.startswith("missing required key"):
        return Diagnostic("E203", where, message, "add the key in the same patch")
    if " not in " in message:
        return Diagnostic("E204", where, message, "use one of the listed values")
    if message.startswith("expected "):
        return Diagnostic("E205", where, message, "change the value's type")
    if "minimum" in message or "maximum" in message:
        return Diagnostic("E206", where, message, "keep the value within the bounds")
    return Diagnostic("E201", where, message)


def diagnose_response(text: str, state: Dict, schema: Dict, checks: List[Callable]):
    """Parse, merge, validate and check one response. Returns (parsed, candidate, diagnostics, kind):
    either a valid candidate state, or diagnostics that say where the response is wrong and how
    to fix it. This is all the model is shown on a retry."""
    try:
        parsed = parse_response(text)
    except ParseError as e:
        blocks = _FENCED.findall(text)
        if not blocks:
            return None, None, [Diagnostic("E101", "response", "no ```json block found",
                                           'end the response with one ```json block holding "state_patch" and "action"')], \
                Kind.SCHEMA_VIOLATION
        raw = blocks[-1].strip()
        try:
            json.loads(raw)
        except json.JSONDecodeError as je:
            src = raw.splitlines() or [""]
            line = src[min(je.lineno, len(src)) - 1]
            return None, None, [Diagnostic(
                "E102", f"json line {je.lineno}, column {je.colno}", je.msg, "fix the JSON syntax; no comments, no trailing commas",
                [f"{je.lineno:>3} | {line[:160]}", f"    | {' ' * min(je.colno - 1, 160)}^"])], Kind.SCHEMA_VIOLATION
        return None, None, [Diagnostic("E103", "json block", str(e),
                                       'the block must be {"state_patch": {...}, "action": "..."} and nothing else')], \
            Kind.SCHEMA_VIOLATION
    try:
        candidate = merge(state, parsed.state_patch)
    except Exception as e:
        return parsed, None, [Diagnostic("E104", "state_patch", f"cannot be merged: {e}")], Kind.SCHEMA_VIOLATION
    errors = validate(candidate, schema)
    if errors:
        return parsed, candidate, [_schema_diagnostic(e, schema) for e in errors], Kind.SCHEMA_VIOLATION
    diags = [Diagnostic("E301", f"invariant {c.__name__}", e) for c in checks for e in c(candidate)]
    return parsed, candidate, diags, (Kind.STATE_CONFLICT if diags else None)


def render_diagnostics(diags: List[Diagnostic], limit: int = 4) -> str:
    more = f"\n(and {len(diags) - limit} more)" if len(diags) > limit else ""
    return "\n" + "\n".join(d.render() for d in diags[:limit]) + more


def start_execution(store: Store, branch: str, role: str, skill_name: str, initial_state: Dict,
                    schema: Dict, instructions: str, parent_execution=None, task_id=None) -> str:
    """Register an execution and what it is pinned to. Its LangGraph thread (thread_id = the
    execution id) starts from `initial_state` on the first run."""
    return store.new_execution(branch, role, pin_config(skill_name, schema, instructions), initial_state,
                               parent_execution, task_id)


class ExecState(TypedDict, total=False):
    """One execution's graph state. `sigma` is Σ, the only part the model ever sees. The other
    fields are the harness's: checkpointed with Σ, restored with Σ, never put in a prompt."""
    sigma: Dict[str, Any]
    observation: str
    action: Optional[str]           # committed by `decide`, executed by `act`
    commit: Optional[str]           # id of the commit that holds the pending action
    outcome: Optional[str]
    step: int
    status: str
    history: List[Dict[str, str]]   # completed actions on this path, read by the loop detector
    scratch: Dict[str, Any]         # per-role harness memory, e.g. the tools an agent used
    redirects: int
    retries: Dict[str, int]
    run_steps: int
    run_budget: int
    pending: Optional[Dict[str, Any]]     # an interrupt waiting for the recovery policy
    last_interrupt: Optional[Dict[str, Any]]
    decision: Optional[str]
    hold: Optional[Dict[str, Any]]        # why the execution is suspended
    after_action: Optional[str]           # "hold": return control to the harness once the action completes
    prompt_tokens: int


def implicate(si: SemanticInterrupt, k: "HarnessKernel") -> List[Dict[str, Any]]:
    """Which components an interrupt points at, as candidates, not as a finding. Where an interrupt
    is observed (this execution and its skill) is often not where it was caused: a dispatch fails
    because an agent's data source is down, a loop repeats a call to a failing tool."""
    skill = {"kind": "skill_output", "id": "{name}@{version}".format(**k.config["skill"])}
    d = si.detail
    if si.kind is Kind.SOURCE_UNAVAILABLE:
        return [{"kind": "delegated_agent", "id": d["agent"], "task": d.get("task")}] if d.get("agent") \
            else [{"kind": "data_source", "id": d.get("tool")}]
    if si.kind is Kind.LOOP_DETECTED:
        if d.get("pattern") == "repeated_failure":       # a failing tool, and instructions that keep retrying it
            return [{"kind": "tool", "id": action_name(d.get("action", ""))}, skill]
        return [skill]
    if si.kind in (Kind.SCHEMA_VIOLATION, Kind.STATE_CONFLICT, Kind.STREAM_ABORTED):
        return [skill, {"kind": "model", "id": getattr(k.llm, "model", None)}]
    if si.kind is Kind.POLICY_DENIED:
        return [{"kind": "approval_boundary", "id": d.get("action")}] if "approv" in si.message else [skill]
    if si.kind is Kind.PIN_MISMATCH:
        return [{"kind": "configuration", "id": k.branch}]
    if si.kind is Kind.OUTCOME_UNKNOWN:
        return [{"kind": "external_system", "id": d.get("action")}]
    if si.kind is Kind.APPROVAL_REQUIRED:
        return [{"kind": "human_decision", "id": d.get("approval")}]
    if si.kind is Kind.BUDGET_EXCEEDED:
        return [{"kind": "execution_budget", "id": None}, skill]
    return [skill]


def si_to_dict(si: SemanticInterrupt) -> Dict[str, Any]:
    return {"kind": si.kind.value, "message": si.message, "hook": si.hook, "detail": si.detail}


def si_from_dict(d: Optional[Dict]) -> Optional[SemanticInterrupt]:
    return SemanticInterrupt(Kind(d["kind"]), d["message"], d["hook"], **d["detail"]) if d else None


SAVER = None       # LangGraph's SqliteSaver on the harness database, set with the store


class HarnessKernel:
    """One execution as a LangGraph thread. The SKILL.state step contract runs in two nodes:

      begin ──> decide ──(checkpoint: Σ_{t+1} and the action are durable)──> act ──> decide ...
                  │                                                           │
                  └──────────────> recover <──────────── interrupts ──────────┘
                                      │
                            redirect / retry ──> decide       suspend / escalate ──> hold (interrupt())

    `decide` makes the model call and validates; LangGraph checkpoints its result before `act`
    runs, which is the commit boundary. `act` executes the action, and recovers an action whose
    commit it finds already started in the action log, according to the action's effect. `hold` pauses the thread
    with LangGraph's interrupt(); resuming it continues where it stopped."""

    def __init__(self, store: Store, execution_id: str, *, role: str, skill_name: str, instructions: str,
                 schema: Dict, checks: List[Callable], llm, middleware: List[Middleware],
                 act: Callable, effect_of: Callable[[str], Effect],
                 reconcile: Optional[Callable] = None, policy: Optional[Dict] = None,
                 max_retries: int = 2, faults: Optional[Faults] = None):
        self.store, self.execution_id, self.role, self.skill_name = store, execution_id, role, skill_name
        self.instructions, self.schema, self.checks, self.llm = instructions, schema, checks, llm
        self.middleware, self.act, self.effect_of, self.reconcile = middleware, act, effect_of, reconcile
        self.policy, self.max_retries, self.faults = policy or DEFAULT_POLICY, max_retries, faults
        ex = store.execution(execution_id)
        self.branch, self.config = ex["branch"], ex["config"]
        self.cfg = {"configurable": {"thread_id": execution_id}, "recursion_limit": 1000}
        self.app = build_kernel_graph(self).compile(checkpointer=SAVER)
        self.calls: List[int] = []
        self.candidate: Dict = {}
        self._resume_reason: Optional[str] = None
        self._sync(self.app.get_state(self.cfg).values or {"status": ex["status"]})
        self.reconcile_status(ex["status"])

    def reconcile_status(self, stored: str) -> None:
        """The checkpoint's status is authoritative. `executions.status` is a copy for queries,
        written in its own transaction; a failure between the two writes can leave it behind."""
        if self.snapshot().values and stored != self.status.value:
            self.event("status_reconciled", stored=stored, checkpoint=self.status.value)
            self.store.set_status(self.execution_id, self.status)

    # -- the view middleware and actions read ------------------------------------------
    def _sync(self, s: Dict):
        self.state = s.get("sigma", {})
        self.step_no, self.history = s.get("step", 0), s.get("history", [])
        self.status = Status(s.get("status", Status.CREATED.value))
        self.scratch = json.loads(json.dumps(s.get("scratch", {})))
        self.prompt_tokens_used = s.get("prompt_tokens", 0)
        self.last_interrupt = si_from_dict(s.get("last_interrupt"))

    def snapshot(self, config: Optional[Dict] = None):
        return self.app.get_state(config or self.cfg)

    # -- records -------------------------------------------------------------------
    def event(self, name: str, **detail):
        self.store.event(self.execution_id, self.branch, self.role, name, skill=self.config["skill"], **detail)

    def set_status(self, status: Status, **detail) -> str:
        if status is not self.status:
            if status not in TRANSITIONS[self.status]:
                raise RuntimeError(f"illegal status change {self.status.value} -> {status.value}")
            self.event("status", frm=self.status.value, to=status.value, **detail)
            self.status = status
            self.store.set_status(self.execution_id, status)
        return status.value

    def current_pins(self) -> Dict[str, Any]:
        return {**pin_config(self.skill_name, self.schema, self.instructions), "model": self.llm.model}

    def pin_mismatches(self) -> List[Dict[str, Any]]:
        return pin_mismatches(self.config, self.current_pins())

    def _pin_interrupt(self) -> Optional[Dict]:
        diff = self.pin_mismatches()
        if not diff:
            return None
        return si_to_dict(SemanticInterrupt(Kind.PIN_MISMATCH, "configuration differs from the pinned one: "
                                            + ", ".join(d["pin"] for d in diff), "pre_inference", mismatches=diff))

    # -- running -------------------------------------------------------------------
    def run(self, observation: str = "", max_steps: int = 40) -> Status:
        """Start, resume or reopen this execution, whichever its thread needs."""
        snap = self.snapshot()
        if not snap.values:                                   # first run of a new thread
            inp = {"sigma": self.store.execution(self.execution_id)["initial_state"], "observation": observation,
                   "action": None, "commit": None, "outcome": None, "step": 0, "status": Status.CREATED.value,
                   "history": [], "scratch": {}, "redirects": 0, "retries": {}, "run_steps": 0,
                   "run_budget": max_steps, "pending": None, "last_interrupt": None, "decision": None,
                   "hold": None, "after_action": None, "prompt_tokens": 0}
        elif snap.interrupts:                                 # suspended at `hold`
            inp = Command(resume={"observation": observation, "budget": max_steps})
        elif snap.next == ("act",) and not self.store.action_row(snap.values.get("commit")):
            inp = None                                        # a commit the harness wrote; not yet run
        elif snap.next:                                       # stopped between nodes: a process failure
            self._resume_reason, inp = "restart", None
        else:                                                 # finished: reopen with a new observation
            inp = {"observation": observation, "run_budget": max_steps}
        return self._invoke(inp, self.cfg)

    def commit_transition(self, patch: Dict, action: str, reason: str) -> str:
        """A transition the harness makes, not the model: commit Σ ⊕ patch and `action` exactly as
        `decide` would, through the same validation and the same pre-action gates, then let `act`
        run it on the next run. When the action completes, control returns to the harness
        (the execution suspends) instead of to the model."""
        snap = self.snapshot()
        s = snap.values
        self._sync(s)
        if s.get("action") and snap.next == ("act",):
            raise RuntimeError(f"an action is already committed and pending: {s['action']}")
        candidate, diags = merge(self.state, patch), []
        diags = [_schema_diagnostic(e, self.schema) for e in validate(candidate, self.schema)] + \
                [Diagnostic("E301", f"invariant {c.__name__}", e) for c in self.checks for e in c(candidate)]
        if diags:
            raise ValueError("the transition would commit an invalid state:" + render_diagnostics(diags))
        self.candidate = candidate
        for m in self.middleware:                 # the same gates a model-chosen action passes
            m.on_pre_action(self, action)
        if self.status in (Status.COMPLETED, Status.SUSPENDED):
            self.set_status(Status.RECOVERING, reason="harness transition")
        status = self.set_status(Status.RUNNING)
        commit = f"{self.execution_id}/{s['step'] + 1}/{digest_text(json.dumps([candidate, action], sort_keys=True))}"
        self.app.update_state(snap.config, {
            "sigma": candidate, "action": action, "commit": commit, "step": s["step"] + 1, "status": status,
            "outcome": None, "pending": None, "hold": None, "after_action": "hold", "run_steps": 0,
            "run_budget": max(s.get("run_budget", 1), 1)}, as_node="decide")
        self.event("harness_transition", action=action, reason=reason, commit=commit)
        return commit

    def run_from(self, config: Dict) -> Status:
        """Time travel: execute again from an earlier checkpoint of this thread. LangGraph forks
        the thread there; the original path stays in its history."""
        self._resume_reason = "time_travel"
        self.event("time_travel", from_checkpoint=config["configurable"]["checkpoint_id"])
        return self._invoke(None, {**config, "recursion_limit": 1000})

    def _invoke(self, inp, config) -> Status:
        # A fresh context: an agent's graph run from inside the coordinator's `act` node is its
        # own thread, not a subgraph of the coordinator.
        contextvars.Context().run(self.app.invoke, inp, config, durability="sync")
        self._sync(self.snapshot().values)
        return self.status

    # -- nodes -----------------------------------------------------------------------
    def _begin(self, s: Dict) -> Dict:
        self._sync(s)
        if self.status is Status.COMPLETED:
            self.event("reopened", observation=s.get("observation", "")[:200])
            self.set_status(Status.RECOVERING)
        out = {"status": self.set_status(Status.RUNNING), "run_steps": 0, "outcome": None, "redirects": 0,
               "pending": self._pin_interrupt()}
        return out

    def _decide(self, s: Dict) -> Dict:
        self._sync(s)
        try:
            if s["run_steps"] >= s["run_budget"]:
                raise SemanticInterrupt(Kind.BUDGET_EXCEEDED, f"{s['run_budget']} steps in one run", "pre_inference")
            for m in self.middleware:
                m.on_pre_inference(self)
            retry_error, retry_kind = None, Kind.SCHEMA_VIOLATION
            for _ in range(self.max_retries + 1):
                prompt = build_prompt(self.instructions, self.state, s["observation"], retry_error)
                text = self._generate(prompt)
                for m in self.middleware:
                    m.on_post_inference(self, text)
                parsed, candidate, diags, kind = diagnose_response(text, self.state, self.schema, self.checks)
                if not diags:
                    try:
                        for m in self.middleware:
                            m.on_pre_commit(self, candidate, parsed.state_patch)
                    except SemanticInterrupt as si:
                        if si.kind not in (Kind.SCHEMA_VIOLATION, Kind.STATE_CONFLICT):
                            raise
                        diags, kind = [Diagnostic("E302", f"pre-commit {si.hook}", si.message)], si.kind
                if diags:                        # the paper's rollback-retry: Σ unchanged, diagnostics shown
                    retry_error, retry_kind = render_diagnostics(diags), kind
                    self.event("rejected_patch", kind=kind.value, codes=[d.code for d in diags], error=retry_error)
                    continue
                break
            else:
                raise SemanticInterrupt(retry_kind, f"no valid state patch in {self.max_retries + 1} attempts:"
                                        f"{retry_error}", "pre_commit", exhausted=True)
            self.candidate = candidate
            for m in self.middleware:
                m.on_pre_action(self, parsed.action)
        except SemanticInterrupt as si:
            return {"pending": si_to_dict(si), "prompt_tokens": self.prompt_tokens_used}
        # Returning commits: LangGraph checkpoints Σ_{t+1} and the action before `act` runs.
        # The commit id is derived from what is committed, so a replay produces the same ids.
        commit = (f"{self.execution_id}/{s['step'] + 1}/"
                  f"{digest_text(json.dumps([candidate, parsed.action], sort_keys=True))}")
        return {"sigma": candidate, "action": parsed.action, "commit": commit, "step": s["step"] + 1,
                "run_steps": s["run_steps"] + 1, "pending": None, "prompt_tokens": self.prompt_tokens_used}

    def _act(self, s: Dict) -> Dict:
        self._sync(s)
        action, commit = s["action"], s["commit"]
        row, handler = self.store.action_row(commit), self.act
        if row is None:
            self.store.action_started(commit, self.execution_id, s["step"], action)
            if self.faults:
                self.faults.after_commit(self)        # a failure between the commit and the action
        else:
            # This commit's action has run before: cut off by a failure, stopped by an interrupt,
            # or executed again by time travel. What happens now depends on what it does.
            reason = "re-executed" if row["status"] == "completed" else "unfinished"
            if self._resume_reason:
                self.event("resumed", previous_status=self.status.value, cause=self._resume_reason,
                           unfinished_action=action)
                self.set_status(Status.RECOVERING)
                self.set_status(Status.RUNNING)
                self._resume_reason = None
            effect = self.effect_of(action)
            plan = RECOVERY[effect] if effect is not Effect.EXTERNAL or self.reconcile else "escalate"
            self.event("recovery_plan", action=action, checkpoint=commit, effect=effect.value, plan=plan,
                       reason=reason)
            pins = self._pin_interrupt()
            if pins:
                return {"pending": pins, "commit": commit, "status": self.status.value}
            if plan == "escalate":
                return {"pending": si_to_dict(SemanticInterrupt(
                    Kind.OUTCOME_UNKNOWN, f"'{action[:80]}' ({effect.value}) may or may not have taken effect",
                    "recovery", action=action, checkpoint=commit)), "commit": commit, "status": self.status.value}
            if plan == "reconcile":
                handler = self.reconcile
        try:
            obs, outcome = handler(action, self)
        except SemanticInterrupt as si:            # the action stays committed and not completed
            return {"pending": si_to_dict(si), "commit": commit, "scratch": self.scratch,
                    "status": self.status.value}
        self.store.action_completed(commit, obs, outcome)
        out = {"observation": obs, "outcome": outcome, "action": None, "commit": None, "pending": None,
               "history": s["history"] + [{**history_entry(action, obs), "commit": commit}],
               "scratch": self.scratch, "status": self.status.value}
        if outcome == "wait":
            out.update(status=self.set_status(Status.SUSPENDED, reason="waiting for input"),
                       hold={"reason": "waiting for input", "needs_human": True})
        elif s.get("after_action") == "hold":       # a transition the harness made: control returns to it
            out.update(status=self.set_status(Status.SUSPENDED, reason="harness transition complete"),
                       hold={"reason": "waiting for the harness", "needs_human": False}, after_action=None)
        return out

    def _recover(self, s: Dict) -> Dict:
        self._sync(s)
        si = si_from_dict(s["pending"])
        self.event("interrupt", kind=si.kind.value, hook=si.hook, message=si.message,
                   observed_at={"execution": self.execution_id, "role": self.role,
                                "skill": "{name}@{version}".format(**self.config["skill"])},
                   implicated=implicate(si, self), **{k: v for k, v in si.detail.items() if k != "kind"})
        self.set_status(Status.INTERRUPTED, kind=si.kind.value)
        rule = self.policy.get(si.kind, "fail")
        if rule == "suspend_or_redirect":
            rule = "suspend" if si.detail.get("source_unavailable") else "redirect"
        if rule == "redirect_or_escalate":
            rule = "escalate" if si.detail.get("exhausted") else "redirect"
        if rule == "redirect" and s["redirects"] >= MAX_REDIRECTS:
            rule = "escalate"                       # redirects spent: a person looks at it
        retries = dict(s.get("retries", {}))
        if rule == "retry" and retries.get(si.kind.value, 0) >= MAX_RETRIES:
            rule = "fail"
        self.event("recovery", kind=si.kind.value, decision=rule)
        out = {"pending": None, "last_interrupt": si_to_dict(si), "decision": rule}
        if rule in ("redirect", "retry"):
            self.set_status(Status.RECOVERING)
            out["status"] = self.set_status(Status.RUNNING)
            if rule == "retry":
                retries[si.kind.value] = retries.get(si.kind.value, 0) + 1
                out["retries"] = retries
            else:                                   # keep what the model was shown, add the refusal
                out["redirects"] = s["redirects"] + 1
                out["observation"] = (f"{s['observation']}\n\nYour last response was refused ({si.kind.value}): "
                                      f"{si.message}. The state is unchanged. Respond again to the observation above.")
                if s.get("action") and s.get("commit"):     # an action refused while it ran
                    self.store.action_completed(s["commit"], f"refused: {si.message}", None)
                    out.update(action=None, commit=None)
            return out
        if rule in ("suspend", "escalate"):
            out["status"] = self.set_status(Status.SUSPENDED, reason=si.kind.value, needs_human=rule == "escalate")
            out["hold"] = {"reason": si.kind.value, "message": si.message, "needs_human": rule == "escalate"}
        else:
            out["status"] = self.set_status(Status.FAILED, reason=si.kind.value)
        return out

    def _hold(self, s: Dict) -> Dict:
        reply = interrupt(s["hold"])               # the run stops here; Command(resume=...) continues it
        self._sync(s)
        self.event("resumed", previous_status=Status.SUSPENDED.value, cause=s["hold"]["reason"],
                   unfinished_action=s.get("action"))
        self.set_status(Status.RECOVERING)
        status = self.set_status(Status.RUNNING)
        return {"observation": reply.get("observation") or s["observation"], "hold": None, "status": status,
                "scratch": {**s.get("scratch", {}), "unavailable": 0}, "run_steps": 0,
                "run_budget": reply.get("budget", s["run_budget"]), "redirects": 0, "pending": self._pin_interrupt()}

    def _finish(self, s: Dict) -> Dict:
        self._sync(s)
        return {"status": self.set_status(Status.COMPLETED)}

    # -- the model call ---------------------------------------------------------------
    def _generate(self, prompt: str) -> str:
        gen, pieces = self.llm.start(prompt), []
        gen.record["execution_id"] = self.execution_id
        for m in self.middleware:
            m.on_stream_start(self)
        try:
            for piece in gen:
                pieces.append(piece)
                for m in self.middleware:
                    m.on_stream_chunk(self, piece)
        finally:
            gen.close()                             # stops the generation if it is still running
            self.calls.append(gen.record["call"])
            self.prompt_tokens_used += gen.record["prompt_tokens"]
        return "".join(pieces)


def build_kernel_graph(k: Optional[HarnessKernel]) -> StateGraph:
    """The same graph for every execution; the kernel supplies what each node does. With
    k=None the nodes are inert, which is enough to read any thread's checkpoints."""
    g = StateGraph(ExecState)
    nodes = {"begin": "_begin", "decide": "_decide", "act": "_act", "recover": "_recover", "hold": "_hold",
             "finish": "_finish"}
    for name, method in nodes.items():
        g.add_node(name, getattr(k, method) if k else (lambda s: {}))
    g.add_edge(START, "begin")
    g.add_conditional_edges("begin", lambda s: "recover" if s.get("pending") else "decide", ["recover", "decide"])
    g.add_conditional_edges("decide", lambda s: "recover" if s.get("pending") else "act", ["recover", "act"])
    g.add_conditional_edges(
        "act", lambda s: "recover" if s.get("pending") else "hold" if s.get("hold") else
        "finish" if s.get("outcome") == "done" else "decide", ["recover", "finish", "hold", "decide"])
    g.add_conditional_edges(
        "recover", lambda s: {"redirect": "decide", "retry": "decide", "suspend": "hold", "escalate": "hold"}.get(s["decision"], END),
        ["decide", "hold", END])
    g.add_conditional_edges(
        "hold", lambda s: "recover" if s.get("pending") else ("act" if s.get("action") else "decide"),
        ["recover", "act", "decide"])
    g.add_edge("finish", END)
    return g


def thread(eid: str) -> Dict:
    return {"configurable": {"thread_id": eid}}


def head(eid: str):
    """The latest checkpoint of an execution's thread, read without building its kernel."""
    return READER.get_state(thread(eid))


def sigma_of(eid: str) -> Dict:
    return head(eid).values.get("sigma") or STORE.execution(eid)["initial_state"]


IMPLEMENTED_VERBS = {"dispatch_agent", "request_approval", "submit_inquiry"}
CONTROL_VERBS = {"wait", "done"}


class MortgageInquiryEndpoint:
    """Stands in for a lender's API: records each request it receives, sends nothing anywhere."""

    def __init__(self):
        self.received: List[Dict] = []

    def submit(self, key: str, payload: Dict) -> Dict:
        receipt = {"confirmation": "MQ-" + key[:8].upper(), "received_at": time.strftime("%Y-%m-%dT%H:%M:%S")}
        self.received.append({"idempotency_key": key, "payload": payload, **receipt})
        return receipt

    def lookup(self, key: str) -> Optional[Dict]:
        hit = [r for r in self.received if r["idempotency_key"] == key]
        return {"confirmation": hit[0]["confirmation"], "received_at": hit[0]["received_at"]} if hit else None


# What each action does. Recovery reads this before it touches an unfinished action;
# anything not listed is UNKNOWN and goes to a person.
TOOL_EFFECTS = {"query_sales": Effect.READ, "metro_value_series": Effect.READ, "market_heat": Effect.READ,
                "value_forecast": Effect.READ, "run_regression": Effect.READ, "hazard_profile": Effect.READ,
                "insurance_scenarios": Effect.READ, "affordability": Effect.READ, "read_artifacts": Effect.READ}
COORD_EFFECTS = {"dispatch_agent": Effect.IDEMPOTENT,     # keyed by (coordinator execution, task id)
                 "request_approval": Effect.IDEMPOTENT,   # a pending request for one payload is reused
                 "submit_inquiry": Effect.EXTERNAL,
                 "wait": Effect.CONTROL, "done": Effect.CONTROL}


def agent_effect(action: str) -> Effect:
    if action.strip() == "finish":
        return Effect.CONTROL
    call = parse_call(action)
    return TOOL_EFFECTS.get(call[0], Effect.UNKNOWN) if call else Effect.UNKNOWN


def coordinator_effect(action: str) -> Effect:
    return COORD_EFFECTS.get(action_name(action), Effect.UNKNOWN)


def inquiry_payload(state: Dict, kind: str) -> Dict[str, Any]:
    """Exactly what `submit_inquiry <kind>` would send from this state. The approval binds to it."""
    return {"action": f"submit_inquiry {kind}", "city": state["branch"]["city"],
            "scenario": state["branch"].get("scenario"), "constraints": state["constraints"],
            "summary": (state.get("pending_action") or {}).get("summary", "")}


UNAVAILABLE_LIMIT = 2      # failed calls to unavailable sources before the harness suspends the agent


def agent_kernel(store: Store, eid: str, agent: str, instructions: str, granted: List[str],
                 skill_ref: Optional[str] = None) -> HarnessKernel:
    branch = store.execution(eid)["branch"]

    def act(action: str, k: HarnessKernel):
        if action.strip() == "finish":
            return "Finished.", "done"
        call = parse_call(action)
        if call is None:
            return f"ERROR: unrecognised action '{action}'. Use `call <tool> <JSON>` or `finish`.", None
        name, args = call
        mem = k.scratch                   # checkpointed with Σ, so it survives suspensions and failures
        try:
            result = run_tool(name, args, lambda ids: store.read_artifacts(branch, ids))
        except ToolUnavailable as e:
            mem["unavailable"] = mem.get("unavailable", 0) + 1
            k.event("source_unavailable", tool=name, args=args, error=str(e), count=mem["unavailable"])
            if mem["unavailable"] >= UNAVAILABLE_LIMIT:
                raise SemanticInterrupt(Kind.SOURCE_UNAVAILABLE,
                                        f"{mem['unavailable']} calls to unavailable sources, last: {name}",
                                        "action", tool=name, args=args)
            return f"ERROR (source unavailable): {e}", None
        except Exception as e:
            return f"ERROR: {type(e).__name__}: {e}", None
        mem["tools_used"] = sorted(set(mem.get("tools_used", [])) | {name})
        if name == "read_artifacts":
            mem["ids_read"] = sorted(set(mem.get("ids_read", [])) | set(args.get("ids", [])))
        return compact(result), None

    return HarnessKernel(store, eid, role=agent, skill_name=skill_ref or AGENT_SKILL[agent], instructions=instructions,
                         schema=AGENT_SCHEMA, checks=agent_checks(agent),
                         llm=llm.with_label(f"kernel:{agent}:{branch}"),
                         middleware=[BudgetMiddleware(14), StreamGuardMiddleware(),
                                     PermissionMiddleware(set(granted) | {"finish"}),
                                     LoopDetectorMiddleware(), FinishCheckMiddleware(agent)],
                         act=act, effect_of=agent_effect)


def resolve_agent(coord: HarnessKernel, agent: str) -> Tuple[str, List[str]]:
    """Which skill an agent runs under, and which tools it is granted. Here: the skill file on
    disk and the tools it requests that this harness implements. Notebook 2 replaces this with
    resolution from a registry."""
    skill = AGENT_SKILL[agent]
    return skill, granted_tools(skill, IMPLEMENTED_TOOLS)


def run_agent(store: Store, coord: HarnessKernel, agent: str, task_id: str) -> Dict:
    s, branch = coord.state, coord.branch
    city, constraints = s["branch"]["city"], s["constraints"]
    refs = list(s.get("artifacts", {}))
    upstream = store.latest_of_kinds(branch, refs, UPSTREAM_KINDS.get(agent, []))
    extra = {}
    if agent == "synthesis":
        extra["artifacts in this branch"] = compact({i: v.get("kind") for i, v in s.get("artifacts", {}).items()})
    skill_ref, granted = resolve_agent(coord, agent)
    surface = agent_instructions(agent, city, constraints, upstream, granted, extra, skill_ref)
    # Dispatch is idempotent per (coordinator execution, task id): a finished child is reused,
    # an unfinished one is resumed from its thread's latest checkpoint, a failed one is replaced.
    children = store.executions(parent_execution=coord.execution_id, task_id=task_id, role=agent)
    live = [e for e in children if e["status"] != Status.FAILED.value]
    if live and live[-1]["status"] == Status.COMPLETED.value:
        a = store.artifact_of_execution(live[-1]["id"])
        if a:
            coord.event("dispatch_reconciled", task=task_id, execution=live[-1]["id"], artifact=a["id"])
            return {"status": "done", "artifact_id": a["id"], "kind": a["kind"], "summary": a["summary"],
                    "gaps": {}, "execution": live[-1]["id"]}
    if live and live[-1]["status"] != Status.COMPLETED.value:
        eid = live[-1]["id"]
        observation = f"Task {task_id} resumes after an interruption. Continue."
    else:
        eid = start_execution(store, branch, agent, skill_ref, initial_agent_state(), AGENT_SCHEMA,
                              surface.text, parent_execution=coord.execution_id, task_id=task_id)
        observation = f"Task {task_id}: {agent} for {city}. Begin."
    k = agent_kernel(store, eid, agent, surface.text, granted, skill_ref)
    status = k.run(observation, max_steps=16)
    draft, gaps = k.state.get("draft") or {}, k.state.get("gaps") or {}
    if status is Status.COMPLETED:
        ids = {u["id"] for u in upstream.values()} | set(k.scratch.get("ids_read", []))
        aid = store.put_artifact(
            draft["kind"], branch, draft.get("summary", ""), draft.get("content"),
            artifact_inputs(agent, city, constraints, k.scratch.get("tools_used", []), ids),
            {"agent": agent, "execution": eid, "skill": k.config["skill"],
             "surface_digest": surface.manifest["surface_digest"],
             "sections": [f"{x['artifact']}:{x['section']}" for x in surface.manifest["sections"]]})
        return {"status": "done", "artifact_id": aid, "kind": draft["kind"],
                "summary": draft.get("summary", ""), "gaps": gaps, "execution": eid}
    si = k.last_interrupt
    if status is Status.SUSPENDED and si and (si.kind is Kind.SOURCE_UNAVAILABLE or
                                              si.detail.get("source_unavailable")):
        return {"status": "suspended", "reason": si.message, "gaps": gaps, "execution": eid}
    return {"status": "failed", "reason": si.message if si else f"ended {status.value} without an artifact",
            "gaps": gaps, "execution": eid}


def submit_with_journal(store: Store, endpoint, k: HarnessKernel, kind: str,
                        faults: Optional[Faults]) -> str:
    """Send at most once. Also the reconciliation path: run again after a failure, it finds the
    journal entry and asks the endpoint before sending anything."""
    action, payload = f"submit_inquiry {kind}", inquiry_payload(k.state, kind)
    approval = store.valid_approval(k.execution_id, action, payload)
    if approval is None:                           # re-checked here, not only at the gate
        raise SemanticInterrupt(Kind.POLICY_DENIED, store.check_approval(k.execution_id, action, payload)[1],
                                "action", action=action)
    digest = payload_digest(payload)
    key = hashlib.sha256(f"{k.execution_id}|{action}|{approval['id']}|{digest}".encode()).hexdigest()[:16]
    entry = store.journal_get(key)
    if entry and entry["status"] == "done":
        return f"Already submitted: confirmation {entry['result']['confirmation']} (key {key})."
    if entry is None:
        store.journal_intent(key, k.execution_id, action, payload)
    already = endpoint.lookup(key) if entry else None     # did it arrive before we lost track?
    receipt = already or endpoint.submit(key, {**payload, "payload_digest": digest, "approval": approval["id"]})
    if faults:
        faults.after_send()
    store.journal_done(key, receipt)
    k.event("external_action", action=action, key=key, payload_digest=digest, approval=approval["id"],
            reconciled=bool(already), confirmation=receipt["confirmation"])
    note = " It had already been received; it was not sent again." if already else ""
    return f"Submitted {kind}: confirmation {receipt['confirmation']} (key {key}).{note}"


def coordinator_kernel(store: Store, eid: str, endpoint, faults: Optional[Faults] = None) -> HarnessKernel:
    def act(action: str, k: HarnessKernel):
        parts = action.split()
        verb = parts[0] if parts else ""
        kind = parts[1] if len(parts) > 1 else ""
        if verb == "done":
            return "Branch finished.", "done"
        if verb == "wait":
            return "Waiting for the buyer.", "wait"
        if verb == "dispatch_agent":
            if len(parts) != 3 or parts[1] not in AGENT_SKILL:
                return "ERROR: use `dispatch_agent <agent> <task_id>` with a known agent.", None
            if not k.state["branch"].get("city"):
                return "ERROR: this branch has no city yet. `wait` for the buyer.", None
            r = run_agent(store, k, parts[1], parts[2])
            if r["status"] == "suspended":
                raise SemanticInterrupt(Kind.SOURCE_UNAVAILABLE, describe_result(parts[1], parts[2], r),
                                        "action", agent=parts[1], task=parts[2])
            return describe_result(parts[1], parts[2], r), None
        if verb == "request_approval":
            payload = inquiry_payload(k.state, kind)
            ok, reason = store.check_approval(k.execution_id, f"submit_inquiry {kind}", payload)
            if ok:
                return f"The buyer's approval is recorded: {reason}.", None
            aid = store.request_approval(k.execution_id, k.branch, f"submit_inquiry {kind}", payload)
            raise SemanticInterrupt(Kind.APPROVAL_REQUIRED, f"{kind} needs the buyer's approval", "action",
                                    approval=aid, payload_digest=payload_digest(payload))
        if verb == "submit_inquiry":
            return submit_with_journal(store, endpoint, k, kind, faults), None
        return f"ERROR: unknown action '{action}'.", None

    def reconcile(action: str, k: HarnessKernel):
        if action_name(action) == "submit_inquiry" and hasattr(endpoint, "lookup"):
            return act(action, k)                  # the journalled path checks before it sends
        raise SemanticInterrupt(Kind.OUTCOME_UNKNOWN, f"cannot establish whether '{action}' took effect",
                                "recovery", action=action)

    granted = set(granted_tools(COORD_SKILL, IMPLEMENTED_VERBS)) | CONTROL_VERBS
    return HarnessKernel(store, eid, role="coordinator", skill_name=COORD_SKILL,
                         instructions=coordinator_instructions().text, schema=COORD_SCHEMA,
                         checks=COORD_CHECKS, llm=llm.with_label(f"kernel:coordinator:{store.execution(eid)['branch']}"),
                         middleware=[BudgetMiddleware(40), StreamGuardMiddleware(), PermissionMiddleware(granted),
                                     LoopDetectorMiddleware(), ApprovalGateMiddleware(inquiry_payload)],
                         act=act, effect_of=coordinator_effect,
                         reconcile=reconcile if hasattr(endpoint, "lookup") else None, faults=faults)


STORE = Store(WORK / "harness.db")
SAVER = SqliteSaver(sqlite3.connect(STORE.path, check_same_thread=False, timeout=30))
READER = build_kernel_graph(None).compile(checkpointer=SAVER)     # reads any execution's thread
ENDPOINT = MortgageInquiryEndpoint()

In [ ]:
# @title Harness from notebook 1: events, forks, audits and invalidation
from types import SimpleNamespace


def events(execution_id: Optional[str] = None, branch: Optional[str] = None,
           kinds: Optional[List[str]] = None) -> pd.DataFrame:
    df = STORE.df("events")
    if execution_id:
        df = df[df.execution_id == execution_id]
    if branch:
        df = df[df.branch == branch]
    if kinds:
        df = df[df.kind.isin(kinds)]
    df = df.copy()
    df["detail"] = df.detail.map(lambda d: {k: v for k, v in json.loads(d).items() if k != "skill"})
    return df[["seq", "branch", "role", "kind", "detail"]]


def show_events(df: pd.DataFrame, width: int = 110) -> None:
    for r in df.itertuples():
        print(f"  {r.seq:>4} {r.role:18}{r.kind:19}{compact(r.detail, width)}")


def ending(k: HarnessKernel) -> str:
    """How a run ended. SUSPENDED is a wait at a boundary, so say what it waits for."""
    hold = (k.snapshot().values or {}).get("hold")
    if k.status is Status.SUSPENDED and hold:
        why = {"waiting for input": "waiting for the buyer's next instruction"}.get(hold["reason"], hold["reason"])
        return (f"SUSPENDED, {why}{' (a person decides)' if hold.get('needs_human') else ''}"
                + (f": {hold['message'][:140]}" if hold.get("message") else ""))
    si = k.last_interrupt
    return f"{k.status.value}" + (f" after {si.kind.value}: {si.message[:160]}" if si else "")


def resumable(eid: str) -> bool:
    """FAILED is terminal. Say what ended an execution instead of trying to reopen it."""
    ex = STORE.execution(eid)
    if ex["status"] != Status.FAILED.value:
        return True
    print(f"{ex['branch']} ({ex['role']}) is FAILED, which is terminal; it is not resumed. What ended it:")
    show_events(events(eid, kinds=["rejected_patch", "interrupt", "recovery"]))
    return False


def fork(store: Store, from_config: Dict, branch: str, patch: Dict) -> str:
    """A new branch from any checkpoint of any thread: copy its Σ, apply the branch's assumptions,
    validate, and start a new execution (a new thread) from the result."""
    snap = READER.get_state(from_config)
    # A task running in the source was dispatched by the source's execution; that dispatch does
    # not exist in the new thread. The new branch starts it again as pending.
    in_flight = {t: {"status": "pending", "artifacts": [], "note": "running in the source when forked"}
                 for t, v in snap.values["sigma"].get("tasks", {}).items() if v.get("status") == "running"}
    state = merge(merge(snap.values["sigma"], {"tasks": in_flight} if in_flight else {}), patch)
    errors = validate(state, COORD_SCHEMA) + [e for c in COORD_CHECKS for e in c(state)]
    if errors:
        raise ValueError(f"fork '{branch}' would start from an invalid state: {errors}")
    parent = store.execution(snap.config["configurable"]["thread_id"])["branch"]
    store.add_branch(branch, parent=parent, from_checkpoint=snap.config["configurable"]["checkpoint_id"],
                     assumptions=patch)
    return start_execution(store, branch, "coordinator", COORD_SKILL, state, COORD_SCHEMA,
                           coordinator_instructions().text)


def brief(city: str) -> str:
    return (f"This branch researches {city} under the recorded constraints. Dispatch the agents, record "
            "their artifacts, run synthesis last, then `done`.")


city_eids: Dict[str, str] = {}


def audit_execution(store: Store, eid: str, endpoint=None) -> pd.DataFrame:
    ex = store.execution(eid)
    hist = list(READER.get_state_history(thread(eid)))           # LangGraph's checkpoints, newest first
    cid = lambda snap: snap.config["configurable"]["checkpoint_id"]
    ids = {cid(h) for h in hist}
    parent = {cid(h): (h.parent_config or {}).get("configurable", {}).get("checkpoint_id") for h in hist}
    tips = [h for h in hist if cid(h) not in set(parent.values())]
    commit_ids = {h.values.get("commit") for h in hist if h.next == ("act",)}
    rows = [("checkpoints form one tree, every parent in this thread",
             sum(p is None for p in parent.values()) <= 1 and all(p is None or p in ids for p in parent.values()),
             f"{len(hist)} checkpoints, {len(tips)} branch tip(s)")]
    acts = store.df("actions").query("execution_id == @eid")
    pending = {h.values.get("commit") for h in tips if h.values.get("action")}
    unfinished = set(acts[acts.status == "started"].commit_id)
    rows.append(("every logged action belongs to a commit of this thread", set(acts.commit_id) <= commit_ids,
                 f"{len(acts)} actions, {len(commit_ids)} commits"))
    rows.append(("only a branch tip's pending action may lack a completion", unfinished <= pending,
                 f"{len(unfinished)} without a completion"))
    ev = events(eid)
    st = [r.detail for r in ev[ev.kind == "status"].itertuples()]
    # Time travel restarts the status from the checkpoint it goes back to: a chain may break there.
    seq = [r for r in ev[ev.kind.isin(["status", "time_travel", "status_reconciled"])].itertuples()]
    restart = ("time_travel", "status_reconciled")       # the status restarts from the checkpoint here
    chained = all(seq[i].kind in restart or seq[i - 1].kind in restart
                  or seq[i].detail["frm"] == seq[i - 1].detail["to"] for i in range(1, len(seq)))
    last = [r for r in seq if r.kind != "time_travel"][-1:] or [None]
    final = (last[0].detail["to"] if last[0].kind == "status" else last[0].detail["checkpoint"]) if last[0] else None
    rows.append(("status changes legal, chained, and match the stored status",
                 all(Status(d["to"]) in TRANSITIONS[Status(d["frm"])] for d in st) and chained
                 and final in (None, ex["status"]),
                 " > ".join([st[0]["frm"]] + [d["to"] for d in st]) if st else ex["status"]))
    graph_status = hist[0].values.get("status") if hist else None
    rows.append(("stored status is the checkpoint's status", graph_status in (None, ex["status"]),
                 f"checkpoint {graph_status}, executions table {ex['status']}"))
    children = store.executions(parent_execution=eid)
    if children:
        per_task = Counter(c["task_id"] for c in children)
        rows.append(("one agent execution per task", all(n == 1 for n in per_task.values()),
                     f"{len(children)} agent executions"))
    on_path = [h.get("commit") for h in (hist[0].values.get("history") or [])] if hist else []
    for r in ev[ev.kind == "recovery_plan"].itertuples():
        c = r.detail["checkpoint"]
        row = store.action_row(c)
        rows.append((f"recovered action ran once: {r.detail['action'][:35]}",
                     row is not None and on_path.count(c) <= 1,
                     f"{r.detail['reason']}, plan {r.detail['plan']}, action {row['status'] if row else None}, "
                     f"in history {on_path.count(c)}"))
    journal = store.df("journal").query("execution_id == @eid")
    for j in journal.itertuples():
        sent = sum(x["idempotency_key"] == j.key for x in endpoint.received) if endpoint else None
        rows.append((f"external '{j.action}' done and received once", j.status == "done" and sent in (None, 1),
                     f"journal {j.status}, received {sent}"))
    return pd.DataFrame(rows, columns=["check", "ok", "detail"])


def show_audit(store: Store, eid: str, endpoint=None) -> bool:
    a = audit_execution(store, eid, endpoint)
    ex = store.execution(eid)
    print(f"audit of {ex['role']} on {ex['branch']} ({eid}):")
    for r in a.itertuples():
        print(f"  {'ok  ' if r.ok else 'FAIL'}  {r.check[:62]:64}{r.detail}")
    return bool(a.ok.all())


def stale_artifacts(store: Store, branch: str, ids: List[str], constraints: Dict) -> Dict[str, str]:
    """Which artifacts no longer hold under `constraints`, and why (recursive over upstream)."""
    reasons: Dict[str, str] = {}

    def stale(aid: str) -> bool:
        if aid in reasons:
            return True
        a = store.artifact(aid) if store.visible(branch, aid) else None
        if a is None:
            reasons[aid] = "not in this branch's artifact store"
            return True
        for key, old in a["inputs"]["constraints"].items():
            if constraints.get(key) != old:
                reasons[aid] = f"constraint {key}: {old} -> {constraints.get(key)}"
                return True
        for name, digest in a["inputs"]["evidence"].items():
            if EVIDENCE[name]["sha256"] != digest:
                reasons[aid] = f"evidence {name} changed"
                return True
        for up in a["inputs"]["upstream"]:
            if stale(up):
                reasons[aid] = f"consumed {up}, which is stale"
                return True
        return False

    for aid in ids:
        stale(aid)
    return reasons


def invalidation_patch(state: Dict, reasons: Dict[str, str]) -> Dict:
    gone = set(reasons) & set(state["artifacts"])
    tasks = {t: {"status": "pending", "artifacts": [],
                 "note": "invalidated: " + "; ".join(reasons[a] for a in v.get("artifacts", []) if a in gone)}
             for t, v in state["tasks"].items() if gone & set(v.get("artifacts", []))}
    return {"phase": "research", "tasks": tasks, "artifacts": {a: None for a in gone},
            "findings": {f: None for f in state["findings"]}, "pending_action": None}

## 1. What notebook 1 left open

Notebook 1 pinned each execution to a digest of its skill, its always-loaded
artifacts, its schema, its model and its exact instructions. Before running, the
kernel verified that they still matched. That answers one question: *is this still
the configuration the execution started with?* A long-running system needs four
more answers:

- **Which version does a new execution get?** "The newest file" is not an answer
  once a version can be drafted, in review, approved, deprecated or revoked.
- **Can an older version keep running?** An investigation pinned to 1.0.0 must still
  be able to resolve 1.0.0's exact text after 1.1.0 is published. That needs a store
  that keeps every version, not a folder that holds the current one.
- **Who decided?** A version's approval, an agent's permissions and a branch's
  migration are decisions by named people, recorded where the harness can check them.
- **What may run at all?** Registration makes an agent or a skill discoverable.
  It grants nothing. Approval, policy and resolution decide what runs. Discovery and
  authority are kept apart throughout this notebook.

## 2. The registries

One SQLite file, `registry.db`, holds five logically distinct registries and an
append-only log of every change made to any of them:

| Registry | Records | Answers |
|---|---|---|
| **Agents** | identity, role, owner, capabilities (tools its runtime *can* call), operations, output kind, endpoint, lifecycle state | which agent fills a role, and what it is able to do |
| **Skills** | every version's exact source text, its digest, requested tools, the always-loaded artifacts it was written against, compatible agents, owner, lifecycle state and its history | which version is approved, and exactly what it says |
| **Identities and policies** | workload and person identities; per-agent policies: the tools an agent *may* call and its constraints, versioned | who acts, and what each agent is allowed to do |
| **Model bindings** | approved model configurations, the roles they serve, their requirements | which model a role runs on |
| **Operational metadata** | derived per-version counts: executions, endings, interrupts, a health note | what has been observed; it decides nothing (section 9) |

A skill version moves through `draft → in_review → approved`, then possibly
`deprecated` and `revoked`. A deprecated version keeps running where it is pinned,
but new resolutions skip it. A revoked version runs nowhere. The owner of a skill
cannot approve it. A version is immutable: the same version number with different
text is refused. A version is identified by the full SHA-256 of its source. The
first 12 characters are shown, and are what notebook 1's pins and provenance record.

**Resolution** turns a role into a configuration a new execution can run. It finds
the active agent for the role and the newest approved version of its skill compatible
with that agent. It then checks the agent's active policy, which must allow every
requested tool the agent can run, and finds an approved model binding. A requested
tool the agent cannot run at all is never granted, whatever any policy says. The
coordinator's skill requests `create_branch`, which no agent runs, because the buyer
branches through the harness. That request is recorded as ungrantable, as notebook 1
left it, and the version runs without it. The result,
`ResolvedRole`, names each record by version and digest. A `TeamConfig` is one
resolved role per role in the team, plus the always-loaded artifacts, identified by a
digest of its content.

Every record is a Pydantic model with validators. The registry also checks the rules
that span records before anything reaches SQLite, and validates every record again
when it reads it back. Section 3.1 shows what each layer rejects, and why the checks
belong in the data layer.

**Pinning** attaches a `TeamConfig` to a branch. Everything that runs in the branch
runs under that configuration, until a migration (section 7) pins another one.

In [ ]:
from typing import Literal

from pydantic import BaseModel, ConfigDict, ValidationError, field_validator, model_validator


class SkillStatus(str, Enum):
    DRAFT = "draft"
    IN_REVIEW = "in_review"
    APPROVED = "approved"
    DEPRECATED = "deprecated"      # still runs where it is pinned; new resolutions skip it
    REVOKED = "revoked"            # runs nowhere, pinned or not


SKILL_TRANSITIONS = {
    SkillStatus.DRAFT: {SkillStatus.IN_REVIEW, SkillStatus.REVOKED},
    SkillStatus.IN_REVIEW: {SkillStatus.APPROVED, SkillStatus.DRAFT, SkillStatus.REVOKED},
    SkillStatus.APPROVED: {SkillStatus.DEPRECATED, SkillStatus.REVOKED},
    SkillStatus.DEPRECATED: {SkillStatus.APPROVED, SkillStatus.REVOKED},
    SkillStatus.REVOKED: set(),
}


class AgentStatus(str, Enum):
    REGISTERED = "registered"
    ACTIVE = "active"
    SUSPENDED = "suspended"
    RETIRED = "retired"


# --- the shapes the data layer accepts -------------------------------------------------------
ROLE = re.compile(r"^[a-z][a-z_]*$")
SLUG = re.compile(r"^[a-z][a-z0-9-]*$")
SEMVER = re.compile(r"^\d+\.\d+\.\d+$")
SKILL_REF = re.compile(r"^[a-z][a-z0-9-]*@\d+\.\d+\.\d+$")
POLICY_REF = re.compile(r"^policy/[a-z][a-z_]*@[1-9]\d*$")
WORKLOAD = re.compile(r"^spiffe://[a-z0-9.-]+/agent/([a-z][a-z_]*)$")
SHA256 = re.compile(r"^[0-9a-f]{64}$")


def _unique(values: List[str], what: str) -> List[str]:
    if len(set(values)) != len(values):
        raise ValueError(f"{what} contains duplicates: {sorted(v for v in set(values) if values.count(v) > 1)}")
    return values


class Record(BaseModel):
    """Base for every registry record: unknown fields are rejected, and assignments are
    validated, so a record cannot be widened or bent after it is built."""
    model_config = ConfigDict(extra="forbid", validate_assignment=True)


# --- the five registry models ---------------------------------------------------------
class Identity(Record):
    """A workload or a person the harness can name as an actor or an owner."""
    id: str
    kind: Literal["workload", "person"]
    owner: str
    description: str = ""

    @model_validator(mode="after")
    def _shape(self):
        if self.kind == "workload" and not WORKLOAD.match(self.id):
            raise ValueError(f"a workload identity must look like spiffe://<domain>/agent/<role>, not {self.id}")
        if self.kind == "person" and not SLUG.match(self.id):
            raise ValueError(f"a person's id must be a lowercase slug, not {self.id}")
        return self


class AgentRecord(Record):
    id: str                                   # agent/<role>
    role: str                                 # the role the coordinator dispatches
    identity: str                             # the workload identity it runs as
    owner: str
    capabilities: List[str]                   # what its runtime CAN do: tools available to it
    operations: List[str]                     # what it can be asked for
    output_kind: Optional[str] = None
    endpoint: str
    status: AgentStatus = AgentStatus.REGISTERED

    @field_validator("capabilities", "operations")
    @classmethod
    def _nonempty_unique(cls, v, info):
        if not v:
            raise ValueError(f"{info.field_name} must not be empty")
        return _unique(v, info.field_name)

    @model_validator(mode="after")
    def _one_role(self):
        # An agent is its role: the id, the role and the workload identity must name the same one.
        if not ROLE.match(self.role) or self.id != f"agent/{self.role}":
            raise ValueError(f"agent id {self.id} does not match role {self.role}")
        m = WORKLOAD.match(self.identity)
        if not m or m.group(1) != self.role:
            raise ValueError(f"agent/{self.role} cannot run as {self.identity}: the workload identity names another role")
        if not self.endpoint.startswith(("inproc://", "https://")):
            raise ValueError(f"endpoint {self.endpoint} must be inproc:// or https://")
        return self


class SkillVersion(Record):
    name: str
    version: str
    kind: Literal["identity", "project", "skill"]
    digest: str                               # full SHA-256 of the exact source text
    source: str
    requires_tools: List[str] = []
    depends_on: Dict[str, str] = {}           # always-loaded artifacts it was written against
    compatible_agents: List[str] = []
    output_kind: Optional[str] = None
    owner: str
    status: SkillStatus = SkillStatus.DRAFT
    changes: str = ""
    history: List[Dict[str, Any]] = []

    @property
    def short(self) -> str:                   # for display, and to match notebook 1's pins
        return self.digest[:12]

    @property
    def ref(self) -> str:
        return f"{self.name}@{self.version}"

    @field_validator("requires_tools")
    @classmethod
    def _tools(cls, v):
        return _unique(v, "requires_tools")

    @model_validator(mode="after")
    def _consistent(self):
        if not SLUG.match(self.name) or not SEMVER.match(self.version):
            raise ValueError(f"{self.name}@{self.version} is not a name@major.minor.patch reference")
        # The digest is of this exact text: a record whose source was changed cannot be built.
        if not SHA256.match(self.digest) or hashlib.sha256(self.source.encode()).hexdigest() != self.digest:
            raise ValueError(f"{self.ref}: the digest is not the SHA-256 of the source")
        for name, ref in self.depends_on.items():
            if not SKILL_REF.match(ref) or ref.split("@")[0] != name:
                raise ValueError(f"{self.ref}: dependency {name} -> {ref} is not a reference to {name}")
        # The status must be where its own history leads: draft, then only allowed transitions.
        state = SkillStatus.DRAFT
        for step in self.history:
            frm, to = SkillStatus(step["from"]), SkillStatus(step["to"])
            if frm is not state or to not in SKILL_TRANSITIONS[frm]:
                raise ValueError(f"{self.ref}: history step {frm.value} -> {to.value} does not follow from {state.value}")
            state = to
        if state is not self.status:
            raise ValueError(f"{self.ref}: status {self.status.value} is not where its history leads ({state.value})")
        return self


class Policy(Record):
    """What an agent MAY do. Separate from what it can do, and from what is registered."""
    id: str                                   # policy/<role>
    version: int
    subject: str                              # agent/<role>
    allow_tools: List[str]
    constraints: Dict[str, Any] = {}
    owner: str
    approved_by: str
    status: Literal["active", "superseded", "revoked"] = "active"

    @property
    def ref(self) -> str:
        return f"{self.id}@{self.version}"

    @field_validator("allow_tools")
    @classmethod
    def _tools(cls, v):
        return _unique(v, "allow_tools")

    @model_validator(mode="after")
    def _subject(self):
        if self.version < 1 or not POLICY_REF.match(self.ref):
            raise ValueError(f"{self.ref} is not policy/<role>@<version from 1>")
        if self.subject != "agent/" + self.id.split("/", 1)[1]:
            raise ValueError(f"{self.id} cannot govern {self.subject}: a policy governs the agent of its own role")
        return self


class ModelBinding(Record):
    id: str
    provider: str
    model: str
    roles: List[str]
    requirements: Dict[str, Any] = {}
    approved_by: str
    status: Literal["approved", "revoked"] = "approved"

    @field_validator("roles")
    @classmethod
    def _roles(cls, v):
        if not v or not all(ROLE.match(r) for r in v):
            raise ValueError(f"roles must be a non-empty list of role names, not {v}")
        return _unique(v, "roles")


class OperationalMetadata(Record):
    """Derived from execution records; informs decisions, makes none."""
    subject: str                              # a skill ref
    executions: int
    by_status: Dict[str, int]
    interrupts: Dict[str, Dict[str, int]]    # observed / implicating this output / also implicating
    health: str
    derived_from: Dict[str, Any]

    @field_validator("subject")
    @classmethod
    def _subject(cls, v):
        if not SKILL_REF.match(v):
            raise ValueError(f"{v} is not a skill reference")
        return v


# --- what resolution produces and what a branch pins ------------------------------------
class ResolvedRole(Record):
    role: str
    agent: str
    skill: str                                # name@version
    digest: str
    policy: str                               # id@version
    granted: List[str]
    model_binding: str

    @model_validator(mode="after")
    def _consistent(self):
        if self.agent != f"agent/{self.role}" or not self.policy.startswith(f"policy/{self.role}@"):
            raise ValueError(f"role {self.role} must resolve to its own agent and policy, not {self.agent} / {self.policy}")
        if not SKILL_REF.match(self.skill) or not SHA256.match(self.digest) or not POLICY_REF.match(self.policy):
            raise ValueError(f"{self.role}: malformed skill, digest or policy reference")
        _unique(self.granted, "granted")
        return self


class TeamConfig(Record):
    roles: Dict[str, ResolvedRole]
    always: Dict[str, str]                    # name -> name@version
    resolved_at: int                          # registry log position at resolution

    @property
    def id(self) -> str:
        return "cfg_" + hashlib.sha256(self.model_dump_json(exclude={"resolved_at"}).encode()).hexdigest()[:10]

    @model_validator(mode="after")
    def _consistent(self):
        if "coordinator" not in self.roles:
            raise ValueError("a team configuration needs a coordinator")
        for key, r in self.roles.items():
            if r.role != key:
                raise ValueError(f"role key {key} holds the configuration of role {r.role}")
        for name, ref in self.always.items():
            if not SKILL_REF.match(ref) or ref.split("@")[0] != name:
                raise ValueError(f"always-loaded {name} -> {ref} is not a reference to {name}")
        return self


def revalidated(record: Record) -> Record:
    """Build the record again from its fields, so every validator runs on its current values.
    (model_copy(update=...) does not validate; a record changed that way is checked here.)"""
    return type(record).model_validate(record.model_dump())


class RegistryError(Exception):
    pass


def _semver(v: str) -> Tuple[int, ...]:
    return tuple(int(x) for x in re.findall(r"\d+", v)[:3])


class Registry:
    """Five logically distinct registries on one SQLite file, and an append-only log of every
    change to them. Registration makes something discoverable. Approval, policy and
    resolution decide whether it runs."""

    DDL = """
    CREATE TABLE IF NOT EXISTS identities (id TEXT PRIMARY KEY, record TEXT);
    CREATE TABLE IF NOT EXISTS agents (id TEXT PRIMARY KEY, role TEXT, status TEXT, record TEXT);
    CREATE TABLE IF NOT EXISTS skills (ref TEXT PRIMARY KEY, name TEXT, version TEXT, status TEXT, digest TEXT,
        record TEXT);
    CREATE TABLE IF NOT EXISTS policies (id TEXT, version INTEGER, status TEXT, record TEXT, PRIMARY KEY (id, version));
    CREATE TABLE IF NOT EXISTS model_bindings (id TEXT PRIMARY KEY, status TEXT, record TEXT);
    CREATE TABLE IF NOT EXISTS team_configs (id TEXT PRIMARY KEY, record TEXT);
    CREATE TABLE IF NOT EXISTS branch_configs (branch TEXT PRIMARY KEY, config_id TEXT, owner TEXT, pinned_at INTEGER,
        reason TEXT);
    CREATE TABLE IF NOT EXISTS migrations (id TEXT PRIMARY KEY, status TEXT, record TEXT);
    CREATE TABLE IF NOT EXISTS operational_metadata (subject TEXT PRIMARY KEY, record TEXT);
    CREATE TABLE IF NOT EXISTS log (seq INTEGER PRIMARY KEY AUTOINCREMENT, actor TEXT, action TEXT, subject TEXT,
        detail TEXT, at REAL);
    """

    def __init__(self, path: Path):
        self.path = Path(path)
        self.db = sqlite3.connect(self.path, check_same_thread=False)
        self.db.row_factory = sqlite3.Row
        self.db.executescript(self.DDL)

    # -- plumbing -------------------------------------------------------------------
    def _x(self, sql, *args):
        self.db.execute(sql, args)
        self.db.commit()

    def _rows(self, sql, *args) -> List[sqlite3.Row]:
        return self.db.execute(sql, args).fetchall()

    def log(self, actor: str, action: str, subject: str, **detail) -> int:
        cur = self.db.execute("INSERT INTO log (actor, action, subject, detail, at) VALUES (?,?,?,?,?)",
                              (actor, action, subject, json.dumps(detail, default=str), time.time()))
        self.db.commit()
        return cur.lastrowid

    def revision(self) -> int:
        return self._rows("SELECT COALESCE(MAX(seq), 0) AS n FROM log")[0]["n"]

    def df(self, table: str) -> pd.DataFrame:
        return pd.read_sql_query(f"SELECT * FROM {table}", self.db)

    @staticmethod
    def _load(cls, raw: str, where: str):
        """Every record read back is validated again: a row changed outside the registry fails
        here, loudly, instead of being resolved and run."""
        try:
            return cls.model_validate_json(raw)
        except ValidationError as e:
            msg = e.errors()[0]["msg"].removeprefix("Value error, ")
            raise RegistryError(f"{where}: stored record failed validation: {msg}") from None

    # -- identities and agents --------------------------------------------------------
    def register_identity(self, ident: Identity, actor: str):
        ident = revalidated(ident)
        known = self.identity(ident.id)
        if known and known != ident:
            raise RegistryError(f"identity {ident.id} is already registered as a {known.kind} owned by {known.owner}")
        if not known:
            self._x("INSERT INTO identities VALUES (?,?)", ident.id, ident.model_dump_json())
            self.log(actor, "register_identity", ident.id, kind=ident.kind, owner=ident.owner)

    def identity(self, iid: str) -> Optional[Identity]:
        r = self._rows("SELECT record FROM identities WHERE id=?", iid)
        return self._load(Identity, r[0]["record"], iid) if r else None

    def register_agent(self, agent: AgentRecord, actor: str):
        agent = revalidated(agent)
        ident = self.identity(agent.identity)
        if ident is None or ident.kind != "workload":
            raise RegistryError(f"{agent.id}: {agent.identity} is not a registered workload identity")
        if self._rows("SELECT id FROM agents WHERE id=? OR role=?", agent.id, agent.role):
            # Re-registering a role would let whoever registers it take over every dispatch to it.
            raise RegistryError(f"role {agent.role} already has a registered agent; it cannot be registered again")
        self._x("INSERT INTO agents VALUES (?,?,?,?)", agent.id, agent.role, agent.status.value, agent.model_dump_json())
        self.log(actor, "register_agent", agent.id, role=agent.role, capabilities=agent.capabilities)

    def set_agent_status(self, agent_id: str, status: AgentStatus, actor: str, reason: str = ""):
        a = self.agent(agent_id)
        a.status = status
        self._x("UPDATE agents SET status=?, record=? WHERE id=?", status.value, revalidated(a).model_dump_json(), agent_id)
        self.log(actor, "agent_status", agent_id, status=status.value, reason=reason)

    def agent(self, agent_id: str) -> Optional[AgentRecord]:
        r = self._rows("SELECT record FROM agents WHERE id=?", agent_id)
        return self._load(AgentRecord, r[0]["record"], agent_id) if r else None

    def agent_for_role(self, role: str) -> Optional[AgentRecord]:
        r = self._rows("SELECT id, record FROM agents WHERE role=?", role)
        return self._load(AgentRecord, r[0]["record"], r[0]["id"]) if r else None

    # -- skills -------------------------------------------------------------------------
    def register_skill(self, path: Path, owner: str, compatible_agents: List[str] = ()) -> SkillVersion:
        """A new version as a draft. The same version with different content is refused:
        a change needs a new version number."""
        a = parse(path)
        ref, source = f"{a.name}@{a.version}", path.read_text(encoding="utf-8")
        digest = hashlib.sha256(source.encode()).hexdigest()
        known = self.skill(ref)
        if known and known.digest != digest:
            raise RegistryError(f"{ref} is registered with digest {known.short}; this source has {digest[:12]}. "
                                "A changed skill needs a new version.")
        if known:
            return known
        always = {x.name: f"{x.name}@{x.version}" for x in self.approved_always()} if a.kind == "skill" else {}
        sv = SkillVersion(name=a.name, version=a.version, kind=a.kind, digest=digest,
                          source=source, requires_tools=a.requires.get("tools", []),
                          depends_on=always, compatible_agents=list(compatible_agents),
                          output_kind=next((self.agent(g).output_kind for g in compatible_agents if self.agent(g)), None),
                          owner=owner,
                          changes=str(a.meta.get("changes", "")).strip())
        self._save_skill(sv)
        self.log(owner, "register_skill", ref, digest=digest, requires_tools=sv.requires_tools)
        return sv

    IMMUTABLE = ("name", "version", "kind", "digest", "source", "requires_tools", "depends_on", "compatible_agents",
                 "output_kind", "owner", "changes")

    def _save_skill(self, sv: SkillVersion):
        """A version's content is fixed when it is registered. Only its status may change, and only
        by extending its history."""
        sv = revalidated(sv)
        known = self.skill(sv.ref)
        if known:
            changed = [f for f in self.IMMUTABLE if getattr(known, f) != getattr(sv, f)]
            if changed:
                raise RegistryError(f"{sv.ref}: {changed} cannot change after registration; register a new version")
            if sv.history[:len(known.history)] != known.history:
                raise RegistryError(f"{sv.ref}: its status history can only be extended, not rewritten")
        self._x("INSERT OR REPLACE INTO skills VALUES (?,?,?,?,?,?)", sv.ref, sv.name, sv.version, sv.status.value,
                sv.digest, sv.model_dump_json())

    def skill(self, ref: str) -> Optional[SkillVersion]:
        r = self._rows("SELECT record FROM skills WHERE ref=?", ref)
        return self._load(SkillVersion, r[0]["record"], ref) if r else None

    def versions(self, name: str) -> List[SkillVersion]:
        rows = [self._load(SkillVersion, r["record"], r["ref"]) for r in self._rows("SELECT ref, record FROM skills WHERE name=?", name)]
        return sorted(rows, key=lambda v: _semver(v.version))

    def all_skills(self) -> List[SkillVersion]:
        return [self._load(SkillVersion, r["record"], r["ref"]) for r in self._rows("SELECT ref, record FROM skills ORDER BY ref")]

    def approved_always(self) -> List[SkillVersion]:
        """The newest approved version of each always-loaded artifact (identity, project)."""
        rows = [v for v in self.all_skills() if v.kind in ("identity", "project") and v.status is SkillStatus.APPROVED]
        return [max((v for v in rows if v.name == n), key=lambda v: _semver(v.version)) for n in sorted({v.name for v in rows})]

    def compatibility(self, ref: str) -> List[str]:
        """Can this version run on the agents it names? Capabilities, not permissions."""
        sv, problems = self.skill(ref), []
        if hashlib.sha256(sv.source.encode()).hexdigest() != sv.digest:
            problems.append("stored source does not match its digest")
        for agent_id in sv.compatible_agents:
            a = self.agent(agent_id)
            if a is None or a.status is not AgentStatus.ACTIVE:
                problems.append(f"{agent_id} is not an active agent")
                continue
            if sv.output_kind and a.output_kind and sv.output_kind != a.output_kind:
                problems.append(f"{agent_id} produces {a.output_kind}, the skill promises {sv.output_kind}")
        for name, dep in sv.depends_on.items():
            d = self.skill(dep)
            if d is None or d.status not in (SkillStatus.APPROVED, SkillStatus.DEPRECATED):
                problems.append(f"depends on {dep}, which is not approved")
        return problems

    def ungrantable(self, ref: str) -> Dict[str, List[str]]:
        """Tools a version requests that its agents cannot run at all. Such a request is never
        granted, whatever a policy says; the version runs without them, as in notebook 1."""
        sv = self.skill(ref)
        return {g: sorted(set(sv.requires_tools) - set(self.agent(g).capabilities))
                for g in sv.compatible_agents if self.agent(g) and set(sv.requires_tools) - set(self.agent(g).capabilities)}

    def set_skill_status(self, ref: str, status: SkillStatus, actor: str, reason: str = ""):
        sv = self.skill(ref)
        if status not in SKILL_TRANSITIONS[sv.status]:
            raise RegistryError(f"{ref}: {sv.status.value} -> {status.value} is not allowed")
        if status is SkillStatus.APPROVED:
            problems = self.compatibility(ref)
            if problems:
                raise RegistryError(f"{ref} cannot be approved: {problems}")
            if actor == sv.owner:
                raise RegistryError(f"{ref}: the owner ({actor}) cannot approve their own skill")
        sv = sv.model_copy(update={"history": sv.history + [{"from": sv.status.value, "to": status.value, "by": actor,
                                                             "reason": reason, "revision": self.revision() + 1}],
                                   "status": status})
        self._save_skill(sv)                  # revalidated there: the new status must follow from the history
        self.log(actor, f"skill_{status.value}", ref, reason=reason)

    # -- policies and model bindings ----------------------------------------------------
    def publish_policy(self, policy: Policy, actor: str):
        policy = revalidated(policy)
        latest = self._rows("SELECT COALESCE(MAX(version), 0) AS v FROM policies WHERE id=?", policy.id)[0]["v"]
        if policy.version != latest + 1:
            # No second version 2, no rewriting version 1: a policy only moves forward.
            raise RegistryError(f"{policy.id}: the next version is {latest + 1}, not {policy.version}")
        if self.agent(policy.subject) is None:
            raise RegistryError(f"{policy.id}: {policy.subject} is not a registered agent")
        for r in self._rows("SELECT record FROM policies WHERE id=? AND status='active'", policy.id):
            old = self._load(Policy, r["record"], policy.id)
            old.status = "superseded"
            self._x("UPDATE policies SET status=?, record=? WHERE id=? AND version=?", "superseded",
                    old.model_dump_json(), old.id, old.version)
        self._x("INSERT INTO policies VALUES (?,?,?,?)", policy.id, policy.version, policy.status, policy.model_dump_json())
        self.log(actor, "publish_policy", policy.ref, allow_tools=policy.allow_tools, approved_by=policy.approved_by)

    def policy(self, ref: str) -> Optional[Policy]:
        pid, _, v = ref.rpartition("@")
        r = self._rows("SELECT record FROM policies WHERE id=? AND version=?", pid, int(v))
        return self._load(Policy, r[0]["record"], ref) if r else None

    def active_policy(self, subject: str) -> Optional[Policy]:
        rows = [self._load(Policy, r["record"], r["id"]) for r in self._rows("SELECT id, record FROM policies WHERE status='active'")]
        return next((p for p in rows if p.subject == subject), None)

    def bind_model(self, binding: ModelBinding, actor: str):
        binding = revalidated(binding)
        if self.binding(binding.id):
            raise RegistryError(f"model binding {binding.id} exists; a changed binding needs a new id")
        self._x("INSERT INTO model_bindings VALUES (?,?,?)", binding.id, binding.status, binding.model_dump_json())
        self.log(actor, "bind_model", binding.id, model=binding.model, roles=binding.roles)

    def binding(self, bid: str) -> Optional[ModelBinding]:
        r = self._rows("SELECT record FROM model_bindings WHERE id=?", bid)
        return self._load(ModelBinding, r[0]["record"], bid) if r else None

    def binding_for_role(self, role: str) -> Optional[ModelBinding]:
        rows = [self._load(ModelBinding, r["record"], r["id"]) for r in self._rows("SELECT id, record FROM model_bindings WHERE status='approved'")]
        return next((b for b in rows if role in b.roles), None)

    # -- discovery: what exists, not what may run ----------------------------------------
    def discover(self, capability: Optional[str] = None, output_kind: Optional[str] = None) -> pd.DataFrame:
        rows = []
        for r in self._rows("SELECT record FROM agents ORDER BY id"):
            a = self._load(AgentRecord, r["record"], "agents")
            if (capability is None or capability in a.capabilities) and (output_kind is None or a.output_kind == output_kind):
                rows += [{"agent": a.id, "agent_status": a.status.value, "skill": sv.ref, "skill_status": sv.status.value,
                          "requires": ", ".join(sv.requires_tools)}
                         for sv in self.all_skills() if a.id in sv.compatible_agents]
        return pd.DataFrame(rows)

    # -- resolution: what runs ---------------------------------------------------------
    def resolve_role(self, role: str, skill: Optional[str] = None) -> ResolvedRole:
        """The configuration a NEW execution of a role gets: an active agent, the newest approved
        (not deprecated) skill version compatible with it, the agent's active policy, which must
        allow every requested tool the agent can run, and an approved model binding."""
        a = self.agent_for_role(role)
        if a is None or a.status is not AgentStatus.ACTIVE:
            raise RegistryError(f"no active agent for role {role}")
        name = skill or (COORD_NAME if role == "coordinator" else AGENT_SKILL[role])
        candidates = [v for v in self.versions(name) if v.status is SkillStatus.APPROVED and a.id in v.compatible_agents]
        if not candidates:
            raise RegistryError(f"no approved version of {name} for {a.id}")
        sv = candidates[-1]
        pol = self.active_policy(a.id)
        if pol is None:
            raise RegistryError(f"{a.id} has no active policy")
        denied = sorted((set(sv.requires_tools) & set(a.capabilities)) - set(pol.allow_tools))
        if denied:                      # the agent could run them: the policy owner must decide
            raise RegistryError(f"{sv.ref} requests {denied}, which {pol.ref} does not allow for {a.id}")
        mb = self.binding_for_role(role)
        if mb is None:
            raise RegistryError(f"no approved model binding for role {role}")
        granted = sorted(set(sv.requires_tools) & set(pol.allow_tools) & set(a.capabilities))
        return ResolvedRole(role=role, agent=a.id, skill=sv.ref, digest=sv.digest, policy=pol.ref, granted=granted,
                            model_binding=mb.id)

    def resolve_team(self, roles: List[str]) -> TeamConfig:
        team = TeamConfig(roles={r: self.resolve_role(r) for r in roles},       # validated as it is built
                          always={v.name: v.ref for v in self.approved_always()}, resolved_at=self.revision())
        self._x("INSERT OR IGNORE INTO team_configs VALUES (?,?)", team.id, team.model_dump_json())
        return team

    def team(self, config_id: str) -> TeamConfig:
        team = self._load(TeamConfig, self._rows("SELECT record FROM team_configs WHERE id=?", config_id)[0]["record"], config_id)
        if team.id != config_id:
            raise RegistryError(f"{config_id}: stored configuration does not match its id")
        return team

    def pin_branch(self, branch: str, team: TeamConfig, actor: str, reason: str, owner: str):
        """Pin a configuration to a branch, and record who owns the investigation in it: the
        person whose approval any change to the branch's configuration needs."""
        who = self.identity(owner)
        if who is None or who.kind != "person":
            raise RegistryError(f"a branch's owner must be a registered person, not {owner}")
        team = revalidated(team)
        self._x("INSERT OR IGNORE INTO team_configs VALUES (?,?)", team.id, team.model_dump_json())
        self._x("INSERT OR REPLACE INTO branch_configs VALUES (?,?,?,?,?)", branch, team.id, owner, self.revision(), reason)
        self.log(actor, "pin_branch", branch, config=team.id, owner=owner, reason=reason,
                 skills={r: x.skill for r, x in team.roles.items()})

    def branch_owner(self, branch: str) -> str:
        r = self._rows("SELECT owner FROM branch_configs WHERE branch=?", branch)
        if not r:
            raise RegistryError(f"branch {branch} has no pinned configuration")
        return r[0]["owner"]

    def branch_team(self, branch: str) -> TeamConfig:
        r = self._rows("SELECT config_id FROM branch_configs WHERE branch=?", branch)
        if not r:
            raise RegistryError(f"branch {branch} has no pinned configuration")
        return self.team(r[0]["config_id"])

    def pinned_problems(self, role: ResolvedRole) -> Tuple[List[str], List[str]]:
        """Can a PINNED configuration still run? Problems stop it; warnings do not."""
        problems, warnings = [], []
        sv, pol, mb = self.skill(role.skill), self.policy(role.policy), self.binding(role.model_binding)
        if sv is None or sv.digest != role.digest:
            problems.append(f"{role.skill} is missing or its digest changed")
        elif sv.status is SkillStatus.REVOKED:
            problems.append(f"{role.skill} was revoked")
        elif sv.status is SkillStatus.DEPRECATED:
            warnings.append(f"{role.skill} is deprecated")
        if pol is None or pol.status == "revoked":
            problems.append(f"{role.policy} was revoked")
        if mb is None or mb.status != "approved":
            problems.append(f"model binding {role.model_binding} was revoked")
        a = self.agent(role.agent)
        if a is None or a.status in (AgentStatus.SUSPENDED, AgentStatus.RETIRED):
            problems.append(f"{role.agent} is {a.status.value if a else 'missing'}")
        return problems, warnings

### 2.1 Connecting the registry to the harness

Three replacements, and one extension of `fork`:

- **`resolve_agent`**, notebook 1's hook, now returns the skill version and the granted
  tools pinned for the agent's role in its branch. If the registry says anything in
  that pinned role was revoked, the dispatch raises `PIN_MISMATCH`, and the
  coordinator's recovery policy hands it to a person.
- **`ARTIFACTS`** becomes a lookup by `name@version` into the source text the registry
  stores, checked against its digest. An unversioned name is refused, so nothing can
  run "whatever the file currently says". The lookup caches parsed text per version,
  because a version's text never changes. It is a content cache, never an
  authorization cache: a revoked version's text stays readable for audits and
  replays. Whether a version may run is asked of the registry at every execution
  boundary: by the pin check before a kernel runs anything, and by `resolve_agent` at
  every dispatch.
- **The pin check** keeps notebook 1's comparison, and adds the registry. The branch
  must have a pinned configuration, and nothing in it may be revoked. A coordinator
  checks every role of its team, because it will dispatch every role.
- **`fork_pinned`** pins the new branch's configuration before forking. By default a
  branch inherits its source's configuration, so a newer approved version never
  reaches a running investigation by accident.

In [ ]:
REGISTRY = Registry(WORK / "registry.db")
COORD_NAME = COORD_SKILL                  # "home-purchase-research": from here on a name; versions come from the registry
FILE_ARTIFACTS = ARTIFACTS                # the files from Drive; used only to register them
SOURCES = WORK / "registry_sources"
SOURCES.mkdir(exist_ok=True)


class RegistryArtifacts:
    """`name@version` -> the artifact parsed from the source text the registry stores.
    An unversioned name is refused: what runs is always a resolved version.

    The cache holds content, which is immutable per version, and nothing else. It never
    answers whether a version may run: a revoked version's text stays readable here, for
    audits and replays. Whether it may run is checked against the registry at each execution
    boundary: by the pin check before a kernel runs anything, and by resolve_agent at every
    dispatch."""

    def __init__(self, registry: Registry):
        self.registry, self._cache = registry, {}

    def __getitem__(self, ref: str) -> Artifact:
        if "@" not in ref:
            raise KeyError(f"'{ref}' has no version; resolve it through the registry")
        if ref not in self._cache:
            sv = self.registry.skill(ref)
            if sv is None:
                raise KeyError(f"{ref} is not registered")
            path = SOURCES / f"{ref}.md"
            path.write_text(sv.source, encoding="utf-8")
            a = parse(path)
            if hashlib.sha256(sv.source.encode()).hexdigest() != sv.digest or a.digest != sv.short:
                raise RegistryError(f"{ref}: stored source does not match its digest")
            self._cache[ref] = a
        return self._cache[ref]


def resolve_agent(coord: HarnessKernel, agent: str) -> Tuple[str, List[str]]:
    """Replaces notebook 1's hook. An agent runs under the skill version pinned for its branch,
    with the tools the pinned policy grants, never under whatever is newest."""
    role = REGISTRY.branch_team(coord.branch).roles[agent]
    problems, warnings = REGISTRY.pinned_problems(role)
    if warnings:
        coord.event("pinned_warning", role=agent, warnings=warnings)
    if problems:
        raise SemanticInterrupt(Kind.PIN_MISMATCH, f"{agent}: " + "; ".join(problems), "action", role=agent)
    return role.skill, role.granted


_notebook1_pin_mismatches = HarnessKernel.pin_mismatches


def _pin_mismatches_with_registry(self) -> List[Dict[str, Any]]:
    """Notebook 1's check (model, skill digest, always-loaded, schema, instructions), plus the
    registry: the branch must have a pinned configuration, and nothing in it may be revoked.
    A coordinator checks its whole team, because it will dispatch every role."""
    out = _notebook1_pin_mismatches(self)
    try:
        team = REGISTRY.branch_team(self.branch)
    except RegistryError as e:
        return out + [{"pin": "branch configuration", "pinned": None, "current": str(e)}]
    roles = team.roles.values() if self.role == "coordinator" else [team.roles[self.role]]
    for r in roles:
        problems, _ = REGISTRY.pinned_problems(r)
        out += [{"pin": f"registry:{r.role}", "pinned": r.skill, "current": p} for p in problems]
    return out


HarnessKernel.pin_mismatches = _pin_mismatches_with_registry


def fork_pinned(from_config: Dict, branch: str, patch: Dict, team: Optional[TeamConfig] = None,
                actor: str = "harness", reason: str = "") -> str:
    """Notebook 1's fork, plus the configuration and the owner: a branch inherits its source's
    pinned configuration unless it is explicitly given another one, and always its owner."""
    src_branch = STORE.execution(READER.get_state(from_config).config["configurable"]["thread_id"])["branch"]
    team = team or REGISTRY.branch_team(src_branch)
    REGISTRY.pin_branch(branch, team, actor, reason or f"inherited from {src_branch}",
                        owner=REGISTRY.branch_owner(src_branch))
    return fork(STORE, from_config, branch, patch)


def team_table(team: TeamConfig) -> pd.DataFrame:
    return pd.DataFrame([{"role": r, "skill": x.skill, "policy": x.policy, "granted": ", ".join(x.granted),
                          "model": x.model_binding} for r, x in team.roles.items()])


ROLES = ["coordinator", *AGENT_SKILL]
print("registry:", REGISTRY.path)
print("roles:", ROLES)

registry: /content/ch04_work/registry.db
roles: ['coordinator', 'market_research', 'quantitative_analysis', 'environmental_risk', 'insurance_finance', 'synthesis']


## 3. Registration and discovery

The first cell registers the team, in the order a deployment would. First the people
and the workload identities the agents run as. Then the agents, with what their
runtime can call, each activated by its owner. Then one policy per agent, by the
security owner: version 1 allows exactly the tools the 1.0.0 skills request. Then
one model binding. Last, the skill files from Drive, each registered by its owner,
put in review, and approved by the review board.

The environmental-risk agent's runtime *can* call `insurance_scenarios`. Its policy
does not allow it. The second cell shows the difference between what discovery finds
and what policy allows.

The third cell switches the harness to the registry: from here on, skill text is read
by version from the registry, never from the files.

In [ ]:
# --- identities: the workloads the agents run as, and the people who own and approve things ---
PEOPLE = {"research-team": "owns the research skills", "risk-team": "owns the environmental-risk skill",
          "skill-review-board": "reviews and approves skill versions", "security-owner": "owns agent policies",
          "platform-owner": "owns agents and model bindings", "buyer": "the person the research is for"}
for pid, what in PEOPLE.items():
    REGISTRY.register_identity(Identity(id=pid, kind="person", owner=pid, description=what), actor="platform-owner")
for role in ROLES:
    REGISTRY.register_identity(Identity(id=f"spiffe://home-research.example/agent/{role}", kind="workload",
                                        owner="platform-owner", description=f"the {role} agent's runtime"),
                               actor="platform-owner")

# --- agents: what each can do (capabilities) -------------------------------------------
CAPABILITIES = {
    "coordinator": ["dispatch_agent", "request_approval", "submit_inquiry", "wait", "done"],
    "market_research": ["query_sales", "metro_value_series", "market_heat", "value_forecast"],
    "quantitative_analysis": ["query_sales", "metro_value_series", "run_regression"],
    "environmental_risk": ["hazard_profile", "insurance_scenarios"],      # its runtime can reach the insurance data
    "insurance_finance": ["insurance_scenarios", "affordability"],
    "synthesis": ["read_artifacts"],
}
for role in ROLES:
    REGISTRY.register_agent(AgentRecord(
        id=f"agent/{role}", role=role, identity=f"spiffe://home-research.example/agent/{role}", owner="platform-owner",
        capabilities=CAPABILITIES[role],
        operations=["coordinate:home-purchase-research"] if role == "coordinator" else [f"produce:{OUTPUT_KIND[role]}"],
        output_kind=None if role == "coordinator" else OUTPUT_KIND[role],
        endpoint=f"inproc://home-research/{role}"), actor="platform-owner")
    REGISTRY.set_agent_status(f"agent/{role}", AgentStatus.ACTIVE, actor="platform-owner", reason="initial rollout")

# --- policies: what each MAY do. Version 1 allows what the 1.0.0 skills request ---------------
for role in ROLES:
    skill = FILE_ARTIFACTS[COORD_NAME if role == "coordinator" else AGENT_SKILL[role]]
    allow = sorted(set(skill.requires.get("tools", [])) & set(CAPABILITIES[role]))
    REGISTRY.publish_policy(Policy(
        id=f"policy/{role}", version=1, subject=f"agent/{role}", allow_tools=allow,
        constraints={"max_steps": 40 if role == "coordinator" else 14, "network": [],
                     "external_actions": ["submit_inquiry: only with a recorded approval"] if role == "coordinator" else []},
        owner="security-owner", approved_by="security-owner"), actor="security-owner")

# --- the model binding --------------------------------------------------------------------
REGISTRY.bind_model(ModelBinding(id="model/primary", provider="openrouter", model=MODEL, roles=ROLES,
                                 requirements={"streaming": True}, approved_by="platform-owner"), actor="platform-owner")

# --- skills: every file on Drive, registered, reviewed and approved ------------------------------
ROLE_OF_SKILL = {COORD_NAME: "coordinator", **{s: r for r, s in AGENT_SKILL.items()}}
for path in sorted(SKILLS.glob("*.md"), key=lambda p: parse(p).kind != "identity" and parse(p).kind != "project"):
    a = parse(path)
    role = ROLE_OF_SKILL.get(a.name)
    owner = "risk-team" if a.name == "environmental-risk" else "research-team"
    sv = REGISTRY.register_skill(path, owner=owner, compatible_agents=[f"agent/{role}"] if role else [])
    REGISTRY.set_skill_status(sv.ref, SkillStatus.IN_REVIEW, actor=owner)
    REGISTRY.set_skill_status(sv.ref, SkillStatus.APPROVED, actor="skill-review-board", reason="initial release")

print(pd.DataFrame([{"ref": v.ref, "status": v.status.value, "sha256": v.short + "…"} for v in REGISTRY.all_skills()])
      .to_string(index=False))
print("\nrequests no agent can run (never granted):",
      {sv.ref: REGISTRY.ungrantable(sv.ref) for sv in REGISTRY.all_skills() if REGISTRY.ungrantable(sv.ref)})
print()
print(pd.DataFrame([{"agent": a, "capabilities": ", ".join(CAPABILITIES[a.split('/')[1]]),
                     "policy allows": ", ".join(REGISTRY.active_policy(a).allow_tools)}
                    for a in (f"agent/{r}" for r in ROLES)]).to_string(index=False))

                             ref   status        sha256
        environmental-risk@1.0.0 approved f5f4fe5d3bca…
    home-purchase-research@1.0.0 approved 7b6ed39e6af3…
       home-research-agent@1.0.0 approved 4029c8ea0750…
         insurance-finance@1.0.0 approved 965c6df4219f…
           market-research@1.0.0 approved 98b66cff194a…
     quantitative-analysis@1.0.0 approved b813e9b72b4b…
seattle-home-investigation@1.0.0 approved c50e285baedc…
                 synthesis@1.0.0 approved d7fe67bd3927…

requests no agent can run (never granted): {'home-purchase-research@1.0.0': {'agent/coordinator': ['create_branch']}}

                      agent                                                 capabilities                                                policy allows
          agent/coordinator dispatch_agent, request_approval, submit_inquiry, wait, done             dispatch_agent, request_approval, submit_inquiry
      agent/market_research query_sales, metro_value_series, market_heat, va

In [ ]:
# Discovery answers "what exists". It says nothing about what may run.
print("agents whose runtime can call insurance_scenarios, and the skills registered for them:")
print(REGISTRY.discover(capability="insurance_scenarios").to_string(index=False))
print("\nwho may call insurance_scenarios, by policy:",
      [a for a in (f"agent/{r}" for r in ROLES) if "insurance_scenarios" in REGISTRY.active_policy(a).allow_tools])

agents whose runtime can call insurance_scenarios, and the skills registered for them:
                   agent agent_status                    skill skill_status                           requires
agent/environmental_risk       active environmental-risk@1.0.0     approved                     hazard_profile
 agent/insurance_finance       active  insurance-finance@1.0.0     approved insurance_scenarios, affordability

who may call insurance_scenarios, by policy: ['agent/insurance_finance']


In [ ]:
# From here on, the harness reads skills from the registry, by version, never from the files.
ARTIFACTS = RegistryArtifacts(REGISTRY)
ALWAYS = [ARTIFACTS[v.ref] for v in REGISTRY.approved_always()]
print("always loaded:", [f"{a.name}@{a.version}" for a in ALWAYS])
try:
    ARTIFACTS["environmental-risk"]
except KeyError as e:
    print("unversioned lookup:", e)

always loaded: ['home-research-agent@1.0.0', 'seattle-home-investigation@1.0.0']
unversioned lookup: "'environmental-risk' has no version; resolve it through the registry"


### 3.1 Rejected at the data layer

Every registry record is a Pydantic model with validators, and the registry checks
the rules that span several records before it writes anything to SQLite.

**Why there, and not only in the code that writes records:**

- **The registry is the control plane.** What it holds is resolved, pinned and run, by
  every execution from then on. A malformed record is not a local bug. It becomes
  configuration, and it gets copied into every branch that pins it.
- **Records have many writers.** Here, the registry's methods and notebook cells.
  Later, an API, a migration script, an import from another system. A check in one
  caller is skipped by the next. A validator on the model holds on every path.
- **Fail when the change is made, not when it runs.** A rejected write is an error for
  the person making the change, at the moment they make it. An accepted bad record
  surfaces days later, as a strange failure in an unrelated execution, or not at all.
- **SQLite can be edited around the registry.** So every record is validated again when
  it is read back. A row changed outside the registry fails loudly instead of being
  resolved and run.

**What is checked where:**

| Layer | Checks | Examples |
|---|---|---|
| Model validators | that a record is well-formed and consistent with itself | an agent's id, role and workload identity name the same role; a version's digest is the SHA-256 of its text; its status is where its own history of allowed transitions leads; a policy governs the agent of its own role; a team configuration's role keys hold their own roles |
| Registry methods, before SQLite | rules across records | one agent per role, never re-registered; a version's content never changes; status history is only extended; policy versions only move forward; a model binding is never redefined |
| Approval and migration logic | authority | who may approve a version, a policy, a migration |

Validation is not authorization: a perfectly valid record can still be one that
nobody with authority approved. Pydantic's `model_copy(update=...)` skips validation,
so the registry rebuilds every record it writes (`revalidated`).

The cell attempts eight bad changes and prints where each is stopped. The last one
edits a row directly in SQLite, then restores it.

In [ ]:
def rejected(label: str, attempt):
    try:
        attempt()
        print(f"ACCEPTED  {label}")
    except ValidationError as e:
        print(f"model     {label}\n          {e.errors()[0]['msg'].removeprefix('Value error, ')}")
    except RegistryError as e:
        print(f"registry  {label}\n          {e}")


risk10 = REGISTRY.skill("environmental-risk@1.0.0")
market = REGISTRY.agent("agent/market_research")

rejected("a version whose text was edited but whose digest was kept",
         lambda: SkillVersion(**{**risk10.model_dump(), "source": risk10.source.replace("60 or more", "50 or more")}))
rejected("an agent record for market_research running as environmental_risk's workload",
         lambda: AgentRecord(**{**market.model_dump(), "identity": "spiffe://home-research.example/agent/environmental_risk"}))
REGISTRY.register_identity(Identity(id="spiffe://other.example/agent/environmental_risk", kind="workload",
                                    owner="platform-owner"), actor="platform-owner")
rejected("a second agent for the environmental_risk role, under another workload identity",
         lambda: REGISTRY.register_agent(AgentRecord(**{**REGISTRY.agent("agent/environmental_risk").model_dump(),
                                                        "identity": "spiffe://other.example/agent/environmental_risk"}),
                                         actor="platform-owner"))
rejected("an approved status without the history that leads to it",
         lambda: SkillVersion(**{**risk10.model_dump(), "history": []}))
rejected("environmental-risk@1.0.0 saved again with another tool",
         lambda: REGISTRY._save_skill(risk10.model_copy(update={"requires_tools": ["hazard_profile", "affordability"]})))
rejected("policy/market_research version 1 published again",
         lambda: REGISTRY.publish_policy(Policy(**{**REGISTRY.active_policy("agent/market_research").model_dump(),
                                                  "allow_tools": ["query_sales", "submit_inquiry"]}), actor="security-owner"))
t = REGISTRY.resolve_team(ROLES)
rejected("a team configuration whose environmental_risk key holds market_research's configuration",
         lambda: TeamConfig(roles={**t.roles, "environmental_risk": t.roles["market_research"]}, always=t.always,
                            resolved_at=t.resolved_at))

row = REGISTRY._rows("SELECT record FROM skills WHERE ref=?", risk10.ref)[0]["record"]
REGISTRY._x("UPDATE skills SET record=? WHERE ref=?", row.replace("60 or more", "50 or more"), risk10.ref)
rejected("environmental-risk@1.0.0 edited directly in SQLite, then read", lambda: REGISTRY.skill(risk10.ref))
REGISTRY._x("UPDATE skills SET record=? WHERE ref=?", row, risk10.ref)
print(f"\nrow restored; {risk10.ref} reads back: {REGISTRY.skill(risk10.ref).status.value}")

model     a version whose text was edited but whose digest was kept
          environmental-risk@1.0.0: the digest is not the SHA-256 of the source
model     an agent record for market_research running as environmental_risk's workload
          agent/market_research cannot run as spiffe://home-research.example/agent/environmental_risk: the workload identity names another role
registry  a second agent for the environmental_risk role, under another workload identity
          role environmental_risk already has a registered agent; it cannot be registered again
model     an approved status without the history that leads to it
          environmental-risk@1.0.0: status approved is not where its history leads (draft)
registry  environmental-risk@1.0.0 saved again with another tool
          environmental-risk@1.0.0: ['requires_tools'] cannot change after registration; register a new version
registry  policy/market_research version 1 published again
          policy/market_research: the next

## 4. An investigation starts, and pins what it resolved

The buyer starts the investigation. The harness resolves a configuration for the
whole team and pins it to the `shared` branch. The coordinator records the
constraints and then waits, because the buyer has not chosen a city. Its latest
checkpoint is `C0`, as in notebook 1. Branches forked from it later inherit the
configuration pinned here, not whatever is newest when they fork.

In [ ]:
CONSTRAINTS = {"budget_usd": 850000, "down_payment_pct": 20, "horizon_years": 10,
               "risk_tolerance": "medium", "financing": "30-year fixed"}

T1 = REGISTRY.resolve_team(ROLES)
COORD_SKILL = T1.roles["coordinator"].skill
print(f"resolved configuration {T1.id} at registry revision {T1.resolved_at}:")
print(team_table(T1).to_string(index=False))

STORE.add_branch("shared")
REGISTRY.pin_branch("shared", T1, actor="buyer", reason="investigation started", owner="buyer")
root_eid = start_execution(STORE, "shared", "coordinator", COORD_SKILL, initial_coord_state(), COORD_SCHEMA,
                           coordinator_instructions().text)
root = coordinator_kernel(STORE, root_eid, ENDPOINT)
root.run(f"Buyer constraints: {json.dumps(CONSTRAINTS)}. The buyer has not chosen which cities to compare yet. "
         "Record the constraints, then `wait` for the buyer.", max_steps=6)
C0 = root.snapshot().config
print(f"\nroot: {ending(root)}; constraints recorded as given: {root.state['constraints'] == CONSTRAINTS}")

resolved configuration cfg_0e0f439aec at registry revision 56:
                 role                        skill                         policy                                                      granted         model
          coordinator home-purchase-research@1.0.0           policy/coordinator@1             dispatch_agent, request_approval, submit_inquiry model/primary
      market_research        market-research@1.0.0       policy/market_research@1 market_heat, metro_value_series, query_sales, value_forecast model/primary
quantitative_analysis  quantitative-analysis@1.0.0 policy/quantitative_analysis@1              metro_value_series, query_sales, run_regression model/primary
   environmental_risk     environmental-risk@1.0.0    policy/environmental_risk@1                                               hazard_profile model/primary
    insurance_finance      insurance-finance@1.0.0     policy/insurance_finance@1                           affordability, insurance_scenarios model/pri

## 5. A newer environmental-risk skill

While the investigation waits, the risk team releases `environmental-risk` 1.1.0
into `ch04_skill_releases`. It adds a watch band for scores 40 to 59. For each material
hazard, it checks whether the cover the hazard implies is offered in the city, and
records unknown availability as an evidence gap. To check that, it requests a second
tool, `insurance_scenarios`.

The first cell registers it as a draft and shows what changed. It then tries to
register an edited copy under the same version number, which the registry refuses.

The second cell takes it through review. The registry checks compatibility: the
agent is active and can call both tools, the output kind matches, and its
dependencies are approved and its stored source matches its digest. Then approval: first by its owner, which the registry
refuses, then by the review board. Approval is still not enough to run it. The
agent's runtime can call the new tool, but its policy does not allow it, so
resolution fails until the security owner publishes policy version 2.

The third cell compares the configuration pinned to the waiting investigation with
the one a new investigation would get now.

In [ ]:
RELEASE = RELEASES / "skill-environmental-risk-1.1.0.md"
new = REGISTRY.register_skill(RELEASE, owner="risk-team", compatible_agents=["agent/environmental_risk"])
old = REGISTRY.skill("environmental-risk@1.0.0")
print(f"registered {new.ref}: {new.status.value}, sha256 {new.short}…")
print(f"  requests tools: {old.requires_tools} -> {new.requires_tools}")
print(f"  changes: {new.changes}")
print("\nprocedure, 1.0.0 -> 1.1.0:")
for line in difflib.unified_diff(ARTIFACTS[old.ref].sections["risk.procedure"].splitlines(),
                                 parse(RELEASE).sections["risk.procedure"].splitlines(), lineterm="", n=0):
    if not line.startswith(("---", "+++", "@@")):
        print("   ", line)

# The same version with different content is not a new version.
tampered = WORK / "tampered-environmental-risk-1.1.0.md"
tampered.write_text(RELEASE.read_text().replace("60 or more", "50 or more"))
try:
    REGISTRY.register_skill(tampered, owner="risk-team", compatible_agents=["agent/environmental_risk"])
except RegistryError as e:
    print("\nre-registering 1.1.0 with an edit:", e)

registered environmental-risk@1.1.0: draft, sha256 9fea3b781b3a…
  requests tools: ['hazard_profile'] -> ['hazard_profile', 'insurance_scenarios']
  changes: Adds a watch band for scores 40 to 59, and checks whether the cover each material hazard implies is offered in the city. Unknown availability is an evidence gap.

procedure, 1.0.0 -> 1.1.0:
    -2. Treat a score of 60 or more as material. List material hazards first.
    +2. Treat a score of 60 or more as material and a score from 40 to 59 as watch. List
    +   material hazards first, then watch hazards.
    +4. Get the insurance scenarios for the city and record, for each implied coverage,
    +   whether it is offered. If the city's scenarios cannot be retrieved, or a coverage is
    +   not listed, record its availability as unknown and add an evidence gap. Never
    +   record unknown availability as available.

re-registering 1.1.0 with an edit: environmental-risk@1.1.0 is registered with digest 9fea3b781b3a; this source has

In [ ]:
REGISTRY.set_skill_status(new.ref, SkillStatus.IN_REVIEW, actor="risk-team", reason="ready for review")
print("compatibility:", REGISTRY.compatibility(new.ref) or "no problems")
for actor in ("risk-team", "skill-review-board"):
    try:
        REGISTRY.set_skill_status(new.ref, SkillStatus.APPROVED, actor=actor, reason="review passed")
        print(f"approval by {actor}: {REGISTRY.skill(new.ref).status.value}")
    except RegistryError as e:
        print(f"approval by {actor}: refused, {e}")

# Approved is not runnable: the agent's policy does not yet allow the new tool.
try:
    REGISTRY.resolve_role("environmental_risk")
except RegistryError as e:
    print("\nresolution for a new execution:", e)

p1 = REGISTRY.active_policy("agent/environmental_risk")
REGISTRY.publish_policy(p1.model_copy(update={"version": 2, "allow_tools": sorted(p1.allow_tools + ["insurance_scenarios"]),
                                               "approved_by": "security-owner", "status": "active"}),
                        actor="security-owner")
r = REGISTRY.resolve_role("environmental_risk")
print(f"after {r.policy}: resolves to {r.skill}, granted {r.granted}")

compatibility: no problems
approval by risk-team: refused, environmental-risk@1.1.0: the owner (risk-team) cannot approve their own skill
approval by skill-review-board: approved

resolution for a new execution: environmental-risk@1.1.0 requests ['insurance_scenarios'], which policy/environmental_risk@1 does not allow for agent/environmental_risk
after policy/environmental_risk@2: resolves to environmental-risk@1.1.0, granted ['hazard_profile', 'insurance_scenarios']


In [ ]:
T2 = REGISTRY.resolve_team(ROLES)
pinned = REGISTRY.branch_team("shared")
diff = pd.DataFrame([{"role": role, "waiting investigation (pinned)": pinned.roles[role].skill + " / " + pinned.roles[role].policy,
                      "a new investigation (resolved now)": T2.roles[role].skill + " / " + T2.roles[role].policy}
                     for role in ROLES if pinned.roles[role] != T2.roles[role]])
print(f"pinned {pinned.id} vs newly resolved {T2.id}; roles that differ:")
print(diff.to_string(index=False))

pinned cfg_0e0f439aec vs newly resolved cfg_5a72a8e467; roles that differ:
              role                         waiting investigation (pinned)                     a new investigation (resolved now)
environmental_risk environmental-risk@1.0.0 / policy/environmental_risk@1 environmental-risk@1.1.0 / policy/environmental_risk@2


## 6. The pinned branch keeps its version

The buyer chooses Kent. The branch forks from `C0` and inherits the configuration the
investigation started with. It runs as in notebook 1. The cell then lists every agent
execution in the branch with the skill version it ran under, next to the version a
new execution would resolve today, and audits the branch's history.

In [ ]:
kent_eid = fork_pinned(C0, "kent", {"branch": {"city": "Kent", "scenario": "baseline"}, "phase": "research"},
                       actor="buyer", reason="the buyer chose Kent")
k = coordinator_kernel(STORE, kent_eid, ENDPOINT)
k.run(brief("Kent"), max_steps=40)
print(f"kent: {ending(k)}, at step {k.step_no}")
print(f"kent's configuration: {REGISTRY.branch_team('kent').id} (the one the investigation started with: "
      f"{REGISTRY.branch_team('kent').id == T1.id})\n")


def runs_of(coord_eid: str) -> pd.DataFrame:
    return pd.DataFrame([{"agent": e["role"], "task": e["task_id"], "status": e["status"],
                          "skill": f"{e['config']['skill']['name']}@{e['config']['skill']['version']}"}
                         for e in STORE.executions(parent_execution=coord_eid)])


print(runs_of(kent_eid).to_string(index=False))
print(f"\nnewest approved environmental-risk now: {REGISTRY.resolve_role('environmental_risk').skill}")
print()
show_audit(STORE, kent_eid)

kent: COMPLETED, at step 6
kent's configuration: cfg_0e0f439aec (the one the investigation started with: True)

                agent                       task    status                       skill
      market_research       kent_market_research COMPLETED       market-research@1.0.0
quantitative_analysis kent_quantitative_analysis COMPLETED quantitative-analysis@1.0.0
   environmental_risk    kent_environmental_risk COMPLETED    environmental-risk@1.0.0
    insurance_finance     kent_insurance_finance COMPLETED     insurance-finance@1.0.0
            synthesis             kent_synthesis SUSPENDED             synthesis@1.0.0

newest approved environmental-risk now: environmental-risk@1.1.0

audit of coordinator on kent (ex_00002):
  ok    checkpoints form one tree, every parent in this thread          16 checkpoints, 1 branch tip(s)
  ok    every logged action belongs to a commit of this thread          6 actions, 6 commits
  ok    only a branch tip's pending action may lack a complet

True

## 7. An explicit migration

Moving Kent to 1.1.0 is a decision, not a side effect of publishing 1.1.0. It is
planned, validated, authorized and executed as four separate steps, and every step is
recorded in the registry.

- **Plan.** Resolve the new version for the role, and build the branch's new
  configuration with only that role changed. Then find what the change invalidates,
  using notebook 1's staleness rules plus one: an artifact produced under a version
  other than the one the new configuration pins is stale, and so is everything that
  consumed it. Planning changes nothing.
- **Validate.** The new version must resolve, which needs approval and a policy that
  allows its tools. The output kind and the model binding must be unchanged, and the
  branch must be at rest.
- **Authorize.** Being a registered person is not authority. The plan names every
  authority it needs. The investigation's owner, recorded when the investigation was
  pinned and inherited by its branches, always has to approve. If the migration grants
  a role more than it had, for example a tool its old policy did not allow, the owner of
  the new policy has to approve too. Each approval records who approved, as which
  authority, the plan's digest and the target configuration. At execution the plan is
  computed again, and every required approval must match its digest and
  configuration. If the branch or the registry changed in between, the approvals do
  not apply.
- **Execute.** The migration creates a new branch, `kent/environmental_risk-1.1.0`,
  from Kent's latest checkpoint. It is pinned to the new configuration, the stale
  artifacts are invalidated and their tasks reset, and the coordinator reruns them.
  Kent itself stays as it was, still pinned to 1.0.0, so its history remains
  reproducible.

The first cell plans the migration and prints the approvals it requires and what
would be recomputed and why. The second goes through the approvals in an order that
shows each rule: an agent's workload identity, a registered person with no authority
here, the buyer, an attempt to execute with an approval still missing, and the
security owner. Then it executes the migration and runs the new branch. The third
compares the two risk assessments. It also checks that Kent's checkpoints and the
artifacts its state refers to are exactly what they were before the migration.

In [ ]:
class Approval(Record):
    by: str                                     # a registered person
    authority: str                              # what they approve as
    plan_digest: str
    to_config: str


class MigrationRequest(Record):
    id: str
    branch: str
    new_branch: str
    from_config: str
    to_config: str
    changes: Dict[str, Dict[str, Any]]          # role -> skill, policy and granted tools, from and to
    recompute: Dict[str, str]                   # artifact id -> why it is stale under the new configuration
    reuse: List[str]
    problems: List[str]
    required: Dict[str, str]                    # authority -> the person who holds it
    requested_by: str
    status: Literal["requested", "approved", "executed"] = "requested"
    approvals: List[Approval] = []
    digest: str = ""

    @model_validator(mode="after")
    def _consistent(self):
        if self.changes and self.from_config == self.to_config:
            raise ValueError("a migration with changes must move to a different configuration")
        if "investigation owner" not in self.required:
            raise ValueError("every migration needs the investigation owner's approval")
        for a in self.approvals:
            if a.authority not in self.required or a.by != self.required[a.authority] or a.to_config != self.to_config:
                raise ValueError(f"approval by {a.by} as {a.authority} does not match this request")
        if self.status != "requested" and not all(any(a.authority == r for a in self.approvals) for r in self.required):
            raise ValueError(f"status {self.status} without every required approval")
        return self


def stale_under(store: Store, branch: str, ids: List[str], team: TeamConfig, constraints: Dict) -> Dict[str, str]:
    """Notebook 1's staleness rules (constraints, evidence digests, upstream), plus one: an artifact
    produced under a skill version other than the one the configuration pins for its role."""
    reasons = dict(stale_artifacts(store, branch, ids, constraints))

    def stale(aid: str) -> bool:
        if aid in reasons:
            return True
        a = store.artifact(aid) if store.visible(branch, aid) else None
        if a is None:
            reasons[aid] = "not in this branch's artifact store"
            return True
        made, pinned = a["produced_by"]["skill"], team.roles.get(a["produced_by"]["agent"])
        if pinned and not pinned.digest.startswith(made["digest"]):     # provenance keeps the 12-character prefix
            reasons[aid] = f"produced by {made['name']}@{made['version']}; the configuration pins {pinned.skill}"
            return True
        for up in a["inputs"]["upstream"]:
            if stale(up):
                reasons[aid] = f"consumed {up}, which is stale"
                return True
        return False

    for aid in ids:
        stale(aid)
    return reasons


def plan_migration(branch: str, roles: List[str], requested_by: str, record: bool = True) -> MigrationRequest:
    """What moving a branch to newly resolved versions of some roles would change, and what it
    would have to recompute. Nothing is changed by planning."""
    eid = STORE.executions(branch=branch, role="coordinator")[-1]["id"]
    old, problems, roles_now = REGISTRY.branch_team(branch), [], {}
    for role in roles:
        try:
            roles_now[role] = REGISTRY.resolve_role(role)
        except RegistryError as e:
            problems.append(str(e))
    new = TeamConfig(roles={**old.roles, **roles_now}, always=old.always, resolved_at=REGISTRY.revision())
    for role, r in roles_now.items():
        if REGISTRY.skill(r.skill).output_kind != REGISTRY.skill(old.roles[role].skill).output_kind:
            problems.append(f"{role}: the output kind changes")
        if r.model_binding != old.roles[role].model_binding:
            problems.append(f"{role}: the model binding changes")
    if STORE.execution(eid)["status"] == Status.RUNNING.value:
        problems.append(f"{branch} is running; migrate a branch at rest")
    s = sigma_of(eid)
    recompute = stale_under(STORE, branch, list(s["artifacts"]), new, s["constraints"])
    changes = {r: {"skill": [old.roles[r].skill, x.skill], "policy": [old.roles[r].policy, x.policy],
                   "granted": [old.roles[r].granted, x.granted]} for r, x in roles_now.items()}
    # Who must approve: always the investigation's owner; and, for every role whose execution
    # would be granted more than before, the owner of the policy that grants it.
    required = {"investigation owner": REGISTRY.branch_owner(branch)}
    for r, x in roles_now.items():
        widened = sorted(set(x.granted) - set(old.roles[r].granted))
        if widened:
            required[f"policy owner ({x.policy}: grants {widened})"] = REGISTRY.policy(x.policy).owner
    req = MigrationRequest(
        id=f"mig_{branch.replace('/', '_')}_{len(STORE.df('branches'))}", branch=branch,
        new_branch=f"{branch}/" + "+".join(f"{r}-{roles_now[r].skill.split('@')[1]}" for r in roles_now),
        from_config=old.id, to_config=new.id, changes=changes, recompute=recompute,
        reuse=[a for a in s["artifacts"] if a not in recompute], problems=problems, required=required,
        requested_by=requested_by)
    req.digest = hashlib.sha256(req.model_dump_json(exclude={"status", "approvals", "digest"}).encode()).hexdigest()[:16]
    if record:
        REGISTRY._x("INSERT OR IGNORE INTO team_configs VALUES (?,?)", new.id, new.model_dump_json())
        REGISTRY._x("INSERT OR REPLACE INTO migrations VALUES (?,?,?)", req.id, req.status, req.model_dump_json())
        REGISTRY.log(requested_by, "migration_requested", req.id, branch=branch, changes=req.changes, digest=req.digest)
    return req


def history_fingerprint(eid: str) -> Dict[str, Any]:
    """Everything that must stay as it was: every checkpoint of the thread, and the content of
    every artifact its latest Σ refers to."""
    hist = list(READER.get_state_history(thread(eid)))
    refs = sigma_of(eid)["artifacts"]
    return {"checkpoints": [(h.config["configurable"]["checkpoint_id"], json.dumps(h.values, sort_keys=True, default=str))
                            for h in hist],
            "artifacts": {a: json.dumps(STORE.artifact(a), sort_keys=True, default=str) for a in refs}}


kent_before = history_fingerprint(kent_eid)
req = plan_migration("kent", ["environmental_risk"], requested_by="buyer")
print(f"{req.id}: {req.branch} -> {req.new_branch}   ({req.from_config} -> {req.to_config}), plan digest {req.digest}")
for role, c in req.changes.items():
    print(f"  {role}: {c['skill'][0]} -> {c['skill'][1]}, {c['policy'][0]} -> {c['policy'][1]}, "
          f"granted {c['granted'][0]} -> {c['granted'][1]}")
print("problems:", req.problems or "none")
print("approvals required:")
for authority, who in req.required.items():
    print(f"   {who:16} as {authority}")
print("to recompute:")
for aid, why in req.recompute.items():
    print(f"   {aid}  {STORE.artifact(aid)['kind'] if STORE.artifact(aid) else '?':16}{why}")
print("to reuse:", [(a, STORE.artifact(a)["kind"]) for a in req.reuse])

mig_kent_2: kent -> kent/environmental_risk-1.1.0   (cfg_0e0f439aec -> cfg_5a72a8e467), plan digest 12a41fbe6d0b05b3
  environmental_risk: environmental-risk@1.0.0 -> environmental-risk@1.1.0, policy/environmental_risk@1 -> policy/environmental_risk@2, granted ['hazard_profile'] -> ['hazard_profile', 'insurance_scenarios']
problems: none
approvals required:
   buyer            as investigation owner
   security-owner   as policy owner (policy/environmental_risk@2: grants ['insurance_scenarios'])
to recompute:
   a_7f09e535be  risk_assessment produced by environmental-risk@1.0.0; the configuration pins environmental-risk@1.1.0
   a_e9350a9494  affordability   consumed a_7f09e535be, which is stale
to reuse: [('a_f442af0d74', 'market_summary'), ('a_736f08ae50', 'model_report')]


In [ ]:
def approve_migration(req: MigrationRequest, approver: str) -> MigrationRequest:
    """Record one approval. The approver must be a registered person who holds one of the
    authorities the plan requires; the approval is bound to the plan's digest and target config."""
    who = REGISTRY.identity(approver)
    if who is None or who.kind != "person":
        raise RegistryError(f"{approver} is not a registered person")
    held = [a for a, holder in req.required.items() if holder == approver
            and not any(x.authority == a for x in req.approvals)]
    if not held:
        raise RegistryError(f"{approver} holds none of the authorities this migration still needs: "
                            f"{[a for a in req.required if not any(x.authority == a for x in req.approvals)]}")
    if req.problems:
        raise RegistryError(f"{req.id} has unresolved problems: {req.problems}")
    for authority in held:
        req.approvals = req.approvals + [Approval(by=approver, authority=authority, plan_digest=req.digest,
                                                  to_config=req.to_config)]           # validated on assignment
        REGISTRY.log(approver, "migration_approved", req.id, authority=authority, plan_digest=req.digest,
                     to_config=req.to_config)
    if all(any(x.authority == a for x in req.approvals) for a in req.required):
        req.status = "approved"
    REGISTRY._x("INSERT OR REPLACE INTO migrations VALUES (?,?,?)", req.id, req.status, req.model_dump_json())
    return req


def execute_migration(req: MigrationRequest) -> str:
    """Only when every required authority approved this exact plan and target configuration, and
    planning again gives the same plan: the branch and the registry must not have changed."""
    missing = [a for a in req.required if not any(x.authority == a and x.plan_digest == req.digest
                                                  and x.to_config == req.to_config for x in req.approvals)]
    if missing:
        raise RegistryError(f"{req.id} is waiting for approval by: {[req.required[a] + ' as ' + a for a in missing]}")
    again = plan_migration(req.branch, list(req.changes), req.requested_by, record=False)
    if again.digest != req.digest:
        raise RegistryError(f"{req.id}: the plan changed since it was approved ({req.digest} -> {again.digest})")
    eid = STORE.executions(branch=req.branch, role="coordinator")[-1]["id"]
    s = sigma_of(eid)
    patch = {"branch": {"scenario": f"migrated: {', '.join(c['skill'][1] for c in req.changes.values())}"},
             **invalidation_patch(s, req.recompute)}
    new_eid = fork_pinned(head(eid).config, req.new_branch, patch, team=REGISTRY.team(req.to_config),
                          actor=req.requested_by, reason=f"migration {req.id}")
    req.status = "executed"
    REGISTRY._x("INSERT OR REPLACE INTO migrations VALUES (?,?,?)", req.id, req.status, req.model_dump_json())
    REGISTRY.log(req.requested_by, "migration_executed", req.id, new_branch=req.new_branch, execution=new_eid,
                 approved_by=[(x.by, x.authority) for x in req.approvals])
    return new_eid


def attempt(label: str, fn):
    try:
        fn()
        print(f"{label}: done")
    except RegistryError as e:
        print(f"{label}: refused, {e}")


attempt("approval by the coordinator's workload identity",
        lambda: approve_migration(req, "spiffe://home-research.example/agent/coordinator"))
attempt("approval by research-team (a registered person, no authority here)", lambda: approve_migration(req, "research-team"))
attempt("approval by buyer", lambda: approve_migration(req, "buyer"))
attempt("execution", lambda: execute_migration(req))
attempt("approval by security-owner", lambda: approve_migration(req, "security-owner"))
print("approvals:", [(x.by, x.authority) for x in req.approvals], "->", req.status)

mig_eid = execute_migration(req)
reset = [t for t, v in sigma_of(mig_eid)["tasks"].items() if v["status"] == "pending"]
k = coordinator_kernel(STORE, mig_eid, ENDPOINT)
k.run(f"This branch was migrated: {req.changes['environmental_risk']['skill'][0]} -> "
      f"{req.changes['environmental_risk']['skill'][1]}. Tasks {reset} were reset because their artifacts were "
      "produced under the old version or consumed one that was. The other artifacts are still valid; reuse them. "
      "Rerun the pending tasks, run synthesis last, then `done`.", max_steps=40)
print(f"\n{req.new_branch}: {ending(k)}")
print(runs_of(mig_eid).to_string(index=False))
print()
show_audit(STORE, mig_eid)

approval by the coordinator's workload identity: refused, spiffe://home-research.example/agent/coordinator is not a registered person
approval by research-team (a registered person, no authority here): refused, research-team holds none of the authorities this migration still needs: ['investigation owner', "policy owner (policy/environmental_risk@2: grants ['insurance_scenarios'])"]
approval by buyer: done
execution: refused, mig_kent_2 is waiting for approval by: ["security-owner as policy owner (policy/environmental_risk@2: grants ['insurance_scenarios'])"]
approval by security-owner: done
approvals: [('buyer', 'investigation owner'), ('security-owner', "policy owner (policy/environmental_risk@2: grants ['insurance_scenarios'])")] -> approved

kent/environmental_risk-1.1.0: COMPLETED
             agent                    task    status                    skill
environmental_risk kent_environmental_risk COMPLETED environmental-risk@1.1.0
 insurance_finance  kent_insurance_finance COMPL

True

In [ ]:
def risk_of(eid: str) -> Optional[Dict]:
    s = sigma_of(eid)
    ids = [i for i, a in s["artifacts"].items() if a["kind"] == "risk_assessment"]
    return STORE.artifact(ids[-1]) if ids else None


for name, eid in (("kent", kent_eid), (req.new_branch, mig_eid)):
    a = risk_of(eid)
    if a is None:
        print(f"{name}: no risk assessment")
        continue
    made = a["produced_by"]["skill"]
    print(f"{name}: {made['name']}@{made['version']}  evidence read: {sorted(a['inputs']['evidence'])}")
    print(f"   {textwrap.shorten(a['summary'], 160)}")
    print(f"   {textwrap.shorten(compact(a['content'], 600), 400)}\n")

shared = set(sigma_of(kent_eid)["artifacts"]) & set(sigma_of(mig_eid)["artifacts"])
print(f"artifacts the migrated branch reuses from kent: {len(shared)}; recomputed: "
      f"{len(set(sigma_of(mig_eid)['artifacts']) - shared)}")
kent_after = history_fingerprint(kent_eid)
print(f"\nkent's checkpoints unchanged by the migration: {kent_after['checkpoints'] == kent_before['checkpoints']} "
      f"({len(kent_before['checkpoints'])} checkpoints)")
print(f"kent's artifacts unchanged: {kent_after['artifacts'] == kent_before['artifacts']} "
      f"({len(kent_before['artifacts'])} artifacts)")
print("kent is still pinned to", REGISTRY.branch_team("kent").roles["environmental_risk"].skill)
assert kent_after == kent_before, "the migration changed the source branch's history"

kent: environmental-risk@1.0.0  evidence read: ['hazard_profiles']
   Kent's material hazards are earthquake and river flood; earthquake and flood coverage should be evaluated.
   {"city":"Kent","source":"Illustrative table","materiality_threshold_score_0_100":60,"hazards":[{"hazard":"earthquake","score_0_100":70,"notes":"Valley soils amplify shaking; liquefaction on the valley floor","material":true,"coverage_implied":"earthquake cover"},{"hazard":"landslide","score_0_100":26,"notes":"Valley [...]

kent/environmental_risk-1.1.0: environmental-risk@1.1.0  evidence read: ['hazard_profiles', 'insurance_scenarios']
   Kent has material earthquake and river-flood hazards; flood coverage is offered, while earthquake coverage availability is unknown.
   {"city":"Kent","hazard_scores":{"earthquake":70,"landslide":26,"river_flood":68,"coastal_flood":5,"wildfire_smoke":55,"volcanic":24},"material_hazards":[{"hazard":"earthquake","score":70,"coverage":"earthquake","coverage_offered":"unknown","e

## 8. Deprecation and revocation

Two different ways of retiring a version.

**Deprecated** means new resolutions skip it, but executions already pinned to it may
continue. The registry reports a warning for them, not a problem.

**Revoked** means it runs nowhere. A branch pinned to a revoked version cannot
continue until it is migrated. The kernel's pin check refuses to run before any model
call, and the recovery policy hands the execution to a person.

The cell deprecates `environmental-risk` 1.0.0, then revokes it, and after each step
shows what a new execution resolves and what the registry says about Kent. It then
tries to reopen Kent, and checks the migrated branch, which is pinned to 1.1.0.

In [ ]:
OLD = "environmental-risk@1.0.0"
kent_role = REGISTRY.branch_team("kent").roles["environmental_risk"]

REGISTRY.set_skill_status(OLD, SkillStatus.DEPRECATED, actor="skill-review-board", reason="superseded by 1.1.0")
print(f"{OLD} deprecated")
print("  a new execution resolves:", REGISTRY.resolve_role("environmental_risk").skill)
print("  kent, pinned to it:", dict(zip(("problems", "warnings"), REGISTRY.pinned_problems(kent_role))))

REGISTRY.set_skill_status(OLD, SkillStatus.REVOKED, actor="skill-review-board",
                          reason="risk assessments without coverage availability are withdrawn")
print(f"\n{OLD} revoked")
print("  kent, pinned to it:", dict(zip(("problems", "warnings"), REGISTRY.pinned_problems(kent_role))))

calls_before = len(USAGE)
k = coordinator_kernel(STORE, kent_eid, ENDPOINT)
k.run("The buyer asks for an update on Kent.", max_steps=8)
print(f"  reopening kent: {ending(k)}; model calls made: {len(USAGE) - calls_before}")
for d in k.last_interrupt.detail.get("mismatches", []):
    print(f"     {d['pin']:28}{d['pinned']}: {d['current']}")

mig_role = REGISTRY.branch_team(req.new_branch).roles["environmental_risk"]
print(f"\n{req.new_branch}, pinned to {mig_role.skill}:",
      dict(zip(("problems", "warnings"), REGISTRY.pinned_problems(mig_role))))

environmental-risk@1.0.0 deprecated
  a new execution resolves: environmental-risk@1.1.0
  kent, pinned to it: {'problems': [], 'warnings': ['environmental-risk@1.0.0 is deprecated']}

environmental-risk@1.0.0 revoked
  kent, pinned to it: {'problems': ['environmental-risk@1.0.0 was revoked'], 'warnings': []}
  reopening kent: SUSPENDED, PIN_MISMATCH (a person decides): configuration differs from the pinned one: registry:environmental_risk; model calls made: 0
     registry:environmental_risk environmental-risk@1.0.0: environmental-risk@1.0.0 was revoked

kent/environmental_risk-1.1.0, pinned to environmental-risk@1.1.0: {'problems': [], 'warnings': []}


## 9. Operational metadata and the registry log

The registry keeps one more kind of record: what has been observed about each skill
version. The first cell derives it from the execution records of this run: per
version, how many executions and how they ended, and the interrupts, counted two
ways.

Where an interrupt was *observed* is not necessarily where it was *caused*. A
coordinator's dispatch fails because an agent's data source is down. A loop repeats
a call to a tool that keeps failing. A branch stops because a version it pins was
revoked. So every interrupt event records both: `observed_at` (the execution, its role
and its skill version) and `implicated`. The second is a list of candidate components,
set by the kernel from the kind of interrupt: the skill's output, the model, a tool or
data source, a delegated agent, the configuration, an external system, or a human
decision. The table shows, per version, the interrupts observed in its executions,
those that implicate its own output, and what else the observed ones implicate. Even
"implicates this skill's output" is a candidate, not a finding.

The metadata is derived, replaceable, and decides nothing. A version is not
approved, deprecated or revoked because of it. Turning such observations into
evidence, and evidence into a governed revision, is notebook 3.

The second cell prints the registry log: every registration, status change, policy,
pin and migration, with who made it, in order. Then it prints each branch with the
configuration it is pinned to.

In [ ]:
def derive_operational_metadata(store: Store, registry: Registry) -> pd.DataFrame:
    """Per skill version, from the execution records: executions, how they ended, the interrupts
    observed in them, and what those interrupts implicate. An interrupt observed in an execution
    of a skill is not evidence against that skill: it may implicate a tool, a data source, a
    delegated agent, the configuration, or nothing defective at all (an approval). Derived and
    replaceable; it informs decisions and makes none."""
    ex, ev = store.df("executions"), store.df("events")
    ex["skill"] = ex.config.map(lambda c: "{name}@{version}".format(**json.loads(c)["skill"]))
    its = [json.loads(d) for d in ev[ev.kind == "interrupt"].detail]
    rows = []
    for ref, group in ex.groupby("skill"):
        observed = [d for d in its if d["observed_at"]["skill"] == ref]
        implicating_self = [d for d in its if any(i["kind"] == "skill_output" and i["id"] == ref for i in d["implicated"])]
        elsewhere = Counter(i["kind"] for d in observed for i in d["implicated"] if i["kind"] != "skill_output")
        n = len(group)
        md = OperationalMetadata(
            subject=ref, executions=n, by_status=dict(Counter(group.status)),
            interrupts={"observed": dict(Counter(d["kind"] for d in observed)),
                        "implicating this skill's output": dict(Counter(d["kind"] for d in implicating_self)),
                        "also implicating": dict(elsewhere)},
            health="insufficient evidence" if n < 5 else "see notebook 3",
            derived_from={"executions": n, "events_up_to": int(ev.seq.max())})
        registry._x("INSERT OR REPLACE INTO operational_metadata VALUES (?,?)", ref, md.model_dump_json())
        rows.append({"subject": ref, "executions": n, "endings": md.by_status, **md.interrupts, "health": md.health})
    registry.log("harness", "derive_operational_metadata", "operational_metadata", subjects=len(rows))
    return pd.DataFrame(rows)


with pd.option_context("display.max_colwidth", 60, "display.width", 200):
    print(derive_operational_metadata(STORE, REGISTRY).to_string(index=False))

                     subject  executions                          endings             observed implicating this skill's output     also implicating                health
    environmental-risk@1.0.0           1                 {'COMPLETED': 1}                   {}                              {}                   {} insufficient evidence
    environmental-risk@1.1.0           1                 {'COMPLETED': 1}                   {}                              {}                   {} insufficient evidence
home-purchase-research@1.0.0           3 {'SUSPENDED': 2, 'COMPLETED': 1}  {'PIN_MISMATCH': 1}                              {} {'configuration': 1} insufficient evidence
     insurance-finance@1.0.0           2                 {'COMPLETED': 2}                   {}                              {}                   {} insufficient evidence
       market-research@1.0.0           1                 {'COMPLETED': 1} {'LOOP_DETECTED': 2}            {'LOOP_DETECTED': 2}                   {} in

In [ ]:
log = REGISTRY.df("log")
print(f"registry log: {len(log)} changes\n")
for r in log.itertuples():
    print(f"  {r.seq:>3} {r.actor:20}{r.action:28}{r.subject}")

print("\nskill versions:")
print(REGISTRY.df("skills")[["ref", "status"]].to_string(index=False))
usage = pd.DataFrame(USAGE)
print(f"\nmodel calls in this run: {len(usage)}, served from the recording: {int(usage.replayed.sum())}, "
      f"prompt tokens: {usage.prompt_tokens.sum():,}")
print("\nbranches and the configuration each is pinned to:")
for r in REGISTRY.df("branch_configs").itertuples():
    t = REGISTRY.team(r.config_id)
    print(f"  {r.branch:40}{r.config_id}  environmental_risk: {t.roles['environmental_risk'].skill:28}({r.reason})")

registry log: 70 changes

    1 platform-owner      register_identity           research-team
    2 platform-owner      register_identity           risk-team
    3 platform-owner      register_identity           skill-review-board
    4 platform-owner      register_identity           security-owner
    5 platform-owner      register_identity           platform-owner
    6 platform-owner      register_identity           buyer
    7 platform-owner      register_identity           spiffe://home-research.example/agent/coordinator
    8 platform-owner      register_identity           spiffe://home-research.example/agent/market_research
    9 platform-owner      register_identity           spiffe://home-research.example/agent/quantitative_analysis
   10 platform-owner      register_identity           spiffe://home-research.example/agent/environmental_risk
   11 platform-owner      register_identity           spiffe://home-research.example/agent/insurance_finance
   12 platform-owner      reg

### What comes next

Every execution now runs under a configuration that names its versions, its policy
and its model binding. Each one was resolved from approved records and pinned to its
branch, and none of them changes without a recorded decision. Notebook 3 starts from
the operational metadata: interrupts and failures recorded against specific skill
versions. It separates execution failures from skill defects, traces a defect to the
skill sections behind it, and takes a targeted revision through evaluation,
approval, promotion and rollback.